# DIMER Notebook: Speech Recognition and Transcription with Whisper

**When can we trust an automatically generated transcript?**

**Profile:** `TASK-INFERENCE` · **Mode:** `WORKSHOP` · **Notebook specification:** `2.2` · **Status:** Candidate, awaiting fresh Colab qualification.

This modular, self-paced Level 2 Applied Tasks unit needs basic Python and Colab familiarity, but no prior ML or audio experience. It can follow OCR and Structured Document Extraction. Here we turn **Audio → Whisper → Transcript**, compare supplied reference text (itself produced by another speech recognizer) with model output, and investigate one controlled change: background noise.

By the end you can explain word errors, distinguish agreement from confidence, compare paired conditions fairly, and state what your evidence does and does not establish. No training occurs in this notebook. Translation, diarization, speaker identity, streaming, timestamps and long recordings are outside this unit. Turbo is used for **English transcription only**.

**Run all:** choose Runtime → Change runtime type → T4 GPU, then Run all. The default path needs no token, upload, restart, repository clone or DIMER service. It carries its Python implementation and integrity manifests. It installs a separate Python 3.12 environment, then evaluates 64 recordings and runs a clean/noisy comparison on 16 different activity recordings. Playback and written answers are optional.

**Resource planning:** allow several GiB for pinned dependencies and the roughly 1.6 GB model. A fresh GPU run is still needed to measure total time, memory and storage; no measured performance promise is made. Installation and downloads may take several minutes.

**Roadmap:** prepare → listen and validate → learn metrics → evaluate → change noise → compare → export → conclude → optional own audio.

Personal learning notes are optional and do not need to be submitted.

## 1. Prepare the runtime

**Input:** a selected GPU runtime and the source carried below. **System:** an isolated Python interpreter with pinned dependencies. **Output:** a local environment; Colab's already-loaded NumPy and torch stay untouched.

The following setup cell is infrastructure. You may collapse it after running. It downloads a hash-verified uv wheel, uses uv's managed interpreter verification for Python 3.12.12, and installs the complete hash-locked dependency set. No downloaded DIMER Python code is executed.

**What to notice:** a GPU name, interpreter version and successful installation. Missing CUDA or integrity errors stop clearly; do not bypass them. A Hugging Face token is optional for the wider course, but is unnecessary here.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time
import urllib.request
import uuid
import zipfile

SESSION_START = time.perf_counter()
ROOT = Path.cwd() / 'outputs' / 'whisper_speech_recognition' / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then start Run all.')
print(gpu.stdout.strip())

In [ ]:
CARRIED_FILES = {
    'whisper_reference.py': (
        '"""Multilingual speech recognition with the pinned ``openai/whisper-large-v3-turbo`` checkpoint.\n'
        '\n'
        'The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)\n'
        'or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with\n'
        '``trust_remote_code=False``: the architecture comes from the pinned ``transformers`` release, the\n'
        'weights are SafeTensors, and no model-repository code is executed.\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import hashlib\n'
        'import json\n'
        'import re\n'
        'from collections.abc import Callable, Mapping, Sequence\n'
        'from dataclasses import dataclass\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'MODEL_ID = "openai/whisper-large-v3-turbo"\n'
        'MODEL_REVISION = "41f01f3fe87f28c78e2fbf8b568835947dd65ed9"\n'
        'MODEL_LICENSE = "MIT"\n'
        'MODEL_KEY = "whisper-large-v3-turbo"\n'
        'DEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n'
        'MANIFEST_NAME = "dimer-base-manifest.json"\n'
        'WEIGHTS_FILE = "model.safetensors"\n'
        'CONFIG_FILE = "config.json"\n'
        '\n'
        'TASKS = ("transcribe", "translate")\n'
        'MIN_CHUNK_LENGTH_S = 1\n'
        "MAX_CHUNK_LENGTH_S = 30  # Whisper's receptive field; longer audio is chunked by the transformers pipeline\n"
        '# A LoRA bundle is accepted only in the safetensors format; peft would otherwise fall back to a\n'
        "# pickle-based adapter_model.bin, which this repository's trust boundary refuses.\n"
        'ADAPTER_WEIGHTS = "adapter_model.safetensors"\n'
        '# Basic WER normalization: case-fold and drop punctuation so that "classes," and "gospel."\n'
        '# match an unpunctuated reference. Curly apostrophes are folded to the straight form first;\n'
        '# word-internal apostrophes and hyphens are kept (a hyphenated compound stays one token).\n'
        '# Numbers, abbreviations and spelled-out forms are NOT normalized ("Mr." vs "Mister" is an\n'
        '# error).\n'
        '_APOSTROPHES = str.maketrans({"\\u2019": "\'", "\\u2018": "\'", "\\u02bc": "\'"})\n'
        '_PUNCTUATION = re.compile(r"[^\\w\\s\'-]|(?<!\\w)[\'-]|[\'-](?!\\w)", re.UNICODE)\n'
        '\n'
        '\n'
        'def _sha256(path: Path) -> str:\n'
        '    digest = hashlib.sha256()\n'
        '    with open(path, "rb") as fh:\n'
        '        for chunk in iter(lambda: fh.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n'
        '    """Check a local snapshot against its manifest; raise naming the first mismatch."""\n'
        '    root = Path(path or DEFAULT_WEIGHTS_DIR)\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID:\n'
        '        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n'
        '    if manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n'
        '    for entry in manifest.get("files", []):\n'
        '        file_path = root / entry["path"]\n'
        '        if not file_path.is_file():\n'
        '            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n'
        '        size = file_path.stat().st_size\n'
        '        if size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n'
        '        digest = _sha256(file_path)\n'
        '        if digest != entry["sha256"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n'
        '    return {"path": str(root), **manifest}\n'
        '\n'
        '\n'
        'def _hub_download(relative_path: str, root: Path) -> None:\n'
        '    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n'
        '    from huggingface_hub import hf_hub_download\n'
        '\n'
        '    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n'
        '\n'
        '\n'
        'def stage_missing_files(\n'
        '    path: str | Path | None = None,\n'
        '    *,\n'
        '    allow_download: bool = False,\n'
        '    downloader: Callable[[str, Path], None] | None = None,\n'
        ') -> list[str]:\n'
        '    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and the\n'
        '    tokenizer/config files but git-ignores the weights). Returns the relative paths fetched;\n'
        '    `verify_snapshot` still runs after."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(\n'
        '            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n'
        '            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n'
        '        )\n'
        '    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n'
        '    if not missing:\n'
        '        return []\n'
        '    if not allow_download:\n'
        '        raise FileNotFoundError(\n'
        '            f"snapshot at {root} is missing {missing}; "\n'
        '            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n'
        '        )\n'
        '    fetch = downloader or _hub_download\n'
        '    for relative_path in missing:\n'
        '        fetch(relative_path, root)\n'
        '    return missing\n'
        '\n'
        '\n'
        'def _tokens(text: str) -> list[str]:\n'
        '    return _PUNCTUATION.sub(" ", text.casefold().translate(_APOSTROPHES)).split()\n'
        '\n'
        '\n'
        'def word_error_count(reference: str, hypothesis: str) -> tuple[int, int]:\n'
        '    """Word-level edit distance and reference length after basic normalization.\n'
        '\n'
        '    Summing the pairs over a corpus and dividing gives the corpus WER; ``word_error_rate``\n'
        '    is the single-utterance ratio.\n'
        '    """\n'
        '    reference_tokens = _tokens(reference)\n'
        '    hypothesis_tokens = _tokens(hypothesis)\n'
        '    if not reference_tokens:\n'
        '        return len(hypothesis_tokens), 0\n'
        '\n'
        '    previous = list(range(len(hypothesis_tokens) + 1))\n'
        '    for row_index, reference_token in enumerate(reference_tokens, 1):\n'
        '        current = [row_index]\n'
        '        for column_index, hypothesis_token in enumerate(hypothesis_tokens, 1):\n'
        '            current.append(\n'
        '                min(\n'
        '                    current[-1] + 1,\n'
        '                    previous[column_index] + 1,\n'
        '                    previous[column_index - 1] + (reference_token != hypothesis_token),\n'
        '                )\n'
        '            )\n'
        '        previous = current\n'
        '    return previous[-1], len(reference_tokens)\n'
        '\n'
        '\n'
        'def word_error_rate(reference: str, hypothesis: str) -> float:\n'
        '    """Word error rate after basic normalization (lowercase, punctuation removed)."""\n'
        '    errors, reference_length = word_error_count(reference, hypothesis)\n'
        '    if reference_length == 0:\n'
        '        return 0.0 if errors == 0 else 1.0\n'
        '    return errors / reference_length\n'
        '\n'
        '\n'
        'def adapter_digest(adapter_dir: str | Path) -> str:\n'
        '    """SHA-256 of the bundle\'s ``adapter_model.safetensors``: the adapter\'s identity."""\n'
        '    digest = hashlib.sha256()\n'
        '    with open(Path(adapter_dir) / ADAPTER_WEIGHTS, "rb") as handle:\n'
        '        for chunk in iter(lambda: handle.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def _load_base(\n'
        '    device: str | None,\n'
        '    weights_dir: str | Path | None,\n'
        '    allow_download: bool,\n'
        '    adapter_dir: str | Path | None,\n'
        ') -> tuple[Any, Any, str, str, Any]:\n'
        '    """Shared loader: ``(model, processor, source, device, dtype)`` from a digest-verified local\n'
        '    snapshot or, only when ``allow_download`` is set and no snapshot exists, from the Hub at the\n'
        '    pinned revision. There is no silent fallback: a missing or unverified snapshot raises unless\n'
        '    downloading was explicitly allowed (MOD8). ``trust_remote_code`` is always False."""\n'
        '    # The bundle-shape check runs before the heavyweight imports so a malformed adapter directory is\n'
        '    # refused (and testable) without torch installed.\n'
        '    adapter_path = None if adapter_dir is None else Path(adapter_dir)\n'
        '    if adapter_path is not None:\n'
        '        for required in ("adapter_config.json", ADAPTER_WEIGHTS):\n'
        '            if not (adapter_path / required).is_file():\n'
        '                raise FileNotFoundError(f"{required} not found in {adapter_path}")\n'
        '\n'
        '    import torch\n'
        '    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor\n'
        '\n'
        '    resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n'
        '    dtype = torch.float16 if resolved_device.startswith("cuda") else torch.float32\n'
        '    root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)\n'
        '    if (root / MANIFEST_NAME).is_file():\n'
        '        stage_missing_files(root, allow_download=allow_download)\n'
        '        verify_snapshot(root)\n'
        '        # A directory argument makes transformers read config/tokenizer/weights from it directly\n'
        '        # (no Hub resolution, no cache lookup).\n'
        '        location: dict[str, Any] = {"pretrained_model_name_or_path": str(root)}\n'
        '        source = "local-snapshot"\n'
        '    elif allow_download:\n'
        '        location = {"pretrained_model_name_or_path": MODEL_ID, "revision": MODEL_REVISION}\n'
        '        source = "hf-hub"\n'
        '    else:\n'
        '        raise FileNotFoundError(\n'
        '            f"no verified snapshot at {root} and allow_download=False; "\n'
        '            f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"\n'
        '        )\n'
        '    processor = AutoProcessor.from_pretrained(**location, trust_remote_code=False)\n'
        '    model = AutoModelForSpeechSeq2Seq.from_pretrained(\n'
        '        **location,\n'
        '        torch_dtype=dtype,\n'
        '        low_cpu_mem_usage=True,\n'
        '        trust_remote_code=False,\n'
        '    )\n'
        '    if adapter_path is not None:\n'
        '        try:\n'
        '            from peft import PeftModel\n'
        '        except ImportError as exc:  # pragma: no cover - depends on the optional extra\n'
        '            raise ImportError("loading an adapter requires the \'finetune\' extra (peft)") from exc\n'
        '        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)\n'
        '        model = model.merge_and_unload()\n'
        '    if resolved_device.startswith("cuda"):\n'
        '        model = model.to(resolved_device)\n'
        '    return model, processor, source, resolved_device, dtype\n'
        '\n'
        '\n'
        'def load_model(\n'
        '    device: str | None = None,\n'
        '    adapter_dir: str | Path | None = None,\n'
        '    *,\n'
        '    weights_dir: str | Path | None = None,\n'
        '    allow_download: bool = True,\n'
        ') -> tuple[Any, Any]:\n'
        '    """Return ``(model, processor)`` for the pinned upstream revision, on ``device``.\n'
        '\n'
        '    Weights are loaded in float16 on CUDA and float32 on CPU, from the digest-verified snapshot\n'
        "    under ``weights_dir`` when one exists (the fine-tuning tutorial passes the notebook's staged\n"
        '    directory) and otherwise from the Hub at the pinned revision. With ``adapter_dir``, a PEFT\n'
        '    LoRA adapter saved by ``PeftModel.save_pretrained`` is attached and merged into the weights,\n'
        '    so the result is a plain Whisper model; ``peft`` is only imported on that path, and only a\n'
        '    safetensors bundle is accepted. Remote model code is always refused.\n'
        '    """\n'
        '    model, processor, _source, _device, _dtype = _load_base(device, weights_dir, allow_download, adapter_dir)\n'
        '    return model, processor\n'
        '\n'
        '\n'
        'def corpus_word_error_rate(references: Sequence[str], hypotheses: Sequence[str]) -> float:\n'
        '    """Corpus WER: total word edits over total reference words (``word_error_count`` summed)."""\n'
        '    if len(references) != len(hypotheses):\n'
        '        raise ValueError("references and hypotheses must have the same length")\n'
        '    counts = [\n'
        '        word_error_count(reference, hypothesis)\n'
        '        for reference, hypothesis in zip(references, hypotheses, strict=True)\n'
        '    ]\n'
        '    total_words = sum(length for _, length in counts)\n'
        '    if total_words == 0:\n'
        '        raise ValueError("corpus WER is undefined for empty references")\n'
        '    return sum(errors for errors, _ in counts) / total_words\n'
        '\n'
        '\n'
        'ADAPTER_BUNDLE_FORMAT = "peft_adapter"\n'
        'ADAPTER_BUNDLE_FORMAT_VERSION = 1\n'
        'ARTIFACT_MANIFEST_NAME = "artifact-manifest.json"\n'
        '\n'
        '\n'
        'def export_adapter_bundle(\n'
        '    model: Any, adapter_dir: str | Path, *, metrics: Mapping[str, Any], provenance: Mapping[str, Any]\n'
        ') -> list[dict[str, Any]]:\n'
        '    """Write the deployable adapter bundle and return its file manifest.\n'
        '\n'
        '    The bundle is what ``from_pretrained(adapter_dir=...)`` consumes: ``adapter_config.json`` and\n'
        '    ``adapter_model.safetensors`` from ``PeftModel.save_pretrained`` (safetensors only), plus\n'
        '    ``metrics.json``, ``provenance.json`` and ``artifact-manifest.json`` (every file with its size\n'
        '    and SHA-256). Saved LoRA ``B`` matrices that are all zero are refused: such an adapter is a\n'
        '    no-op and exporting it would hide a training run that never updated anything.\n'
        '    """\n'
        '    from safetensors.torch import load_file\n'
        '\n'
        '    root = Path(adapter_dir)\n'
        '    if root.exists():\n'
        '        for stale in sorted(root.rglob("*"), reverse=True):\n'
        '            stale.unlink() if stale.is_file() else stale.rmdir()\n'
        '    root.mkdir(parents=True)\n'
        '    model.save_pretrained(str(root), safe_serialization=True)\n'
        '    if not (root / ADAPTER_WEIGHTS).is_file():\n'
        '        raise RuntimeError(f"{ADAPTER_WEIGHTS} was not written; only safetensors adapters are accepted")\n'
        '    saved_b = [tensor for name, tensor in load_file(str(root / ADAPTER_WEIGHTS)).items() if "lora_B" in name]\n'
        '    if not saved_b or max(float(tensor.abs().max()) for tensor in saved_b) == 0:\n'
        '        raise RuntimeError("saved adapter weights are zero or missing")\n'
        '    (root / "metrics.json").write_text(json.dumps(dict(metrics), indent=2), encoding="utf-8")\n'
        '    (root / "provenance.json").write_text(json.dumps(dict(provenance), indent=2), encoding="utf-8")\n'
        '    manifest = [\n'
        '        {"path": path.relative_to(root).as_posix(), "bytes": path.stat().st_size, "sha256": _sha256(path)}\n'
        '        for path in sorted(root.rglob("*"))\n'
        '        if path.is_file()\n'
        '    ]\n'
        '    (root / ARTIFACT_MANIFEST_NAME).write_text(\n'
        '        json.dumps(\n'
        '            {\n'
        '                "format": ADAPTER_BUNDLE_FORMAT,\n'
        '                "formatVersion": ADAPTER_BUNDLE_FORMAT_VERSION,\n'
        '                "files": manifest,\n'
        '            },\n'
        '            indent=2,\n'
        '        ),\n'
        '        encoding="utf-8",\n'
        '    )\n'
        '    return manifest\n'
        '\n'
        '\n'
        'def verify_adapter_merge(\n'
        '    adapter_dir: str | Path,\n'
        '    module_name: str,\n'
        '    base_weight: Any,\n'
        '    reloaded_weight: Any,\n'
        '    *,\n'
        '    rank: int,\n'
        '    alpha: int,\n'
        '    tolerance: float,\n'
        ') -> dict[str, Any]:\n'
        '    """Weight-level proof that a fresh load applied the saved adapter to the base weights.\n'
        '\n'
        '    For one probe module, ``W_reloaded`` must equal ``W_base + (alpha / rank) * B @ A`` read back\n'
        '    from the bundle, within ``tolerance``; an adapter whose delta is below the verification\n'
        '    resolution is rejected too, because the check would then pass on an unmodified base.\n'
        '    """\n'
        '    import torch\n'
        '    from safetensors.torch import load_file\n'
        '\n'
        '    saved = load_file(str(Path(adapter_dir) / ADAPTER_WEIGHTS))\n'
        '    lora_a = saved[f"base_model.model.{module_name}.lora_A.weight"].to(torch.float32)\n'
        '    lora_b = saved[f"base_model.model.{module_name}.lora_B.weight"].to(torch.float32)\n'
        '    base = base_weight.detach().to("cpu", torch.float32)\n'
        '    expected = base + (alpha / rank) * (lora_b @ lora_a)\n'
        '    reloaded = reloaded_weight.detach().to("cpu", torch.float32)\n'
        '    adapter_delta = float((expected - base).abs().max())\n'
        '    merge_error = float((reloaded - expected).abs().max())\n'
        '    if adapter_delta <= 4 * tolerance:\n'
        '        raise RuntimeError(\n'
        '            f"adapter delta {adapter_delta:.2e} on {module_name} is below the {tolerance:.0e} verification "\n'
        '            "resolution; train longer or with a higher learning rate"\n'
        '        )\n'
        '    if merge_error > tolerance:\n'
        '        raise RuntimeError(\n'
        '            f"reloaded weights differ from base + scaled B@A by {merge_error:.2e} "\n'
        '            f"(> {tolerance:.0e}) on {module_name}"\n'
        '        )\n'
        '    return {\n'
        '        "module": module_name,\n'
        '        "adapter_delta_max": adapter_delta,\n'
        '        "merge_error_max": merge_error,\n'
        '        "tolerance": tolerance,\n'
        '    }\n'
        '\n'
        '\n'
        'def adaptation_report(\n'
        '    *,\n'
        '    baseline_wer: float,\n'
        '    adapted_wer: float,\n'
        '    reloaded_wer: float | None,\n'
        '    n_eval: int,\n'
        '    history: Sequence[Mapping[str, Any]],\n'
        '    dataset: Mapping[str, Any],\n'
        '    sample_kind: str = "public-sample",\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage for the fine-tuning tutorial: corpus WER before/after adaptation.\n'
        '\n'
        '    The verdict is always ``sample-sanity``: one held-out split of one corpus says whether the\n'
        '    adapter helped *here*, not how it generalises, and ``history`` (teacher-forced losses) is\n'
        '    optimisation evidence only (FT7).\n'
        '    """\n'
        '    metrics = [\n'
        '        {\n'
        '            "id": "baseline_wer",\n'
        '            "value": baseline_wer,\n'
        '            "estimation": f"corpus WER over {n_eval} held-out utterances, zero-shot",\n'
        '        },\n'
        '        {\n'
        '            "id": "adapted_wer",\n'
        '            "value": adapted_wer,\n'
        '            "estimation": f"corpus WER over the same {n_eval} utterances, in-memory adapter",\n'
        '        },\n'
        '    ]\n'
        '    if reloaded_wer is not None:\n'
        '        metrics.append(\n'
        '            {\n'
        '                "id": "reloaded_wer",\n'
        '                "value": reloaded_wer,\n'
        '                "estimation": "same split, adapter reloaded from the bundle",\n'
        '            }\n'
        '        )\n'
        '    return {\n'
        '        "task": "automatic speech recognition (LoRA adaptation)",\n'
        '        "score_semantics": "generated transcript; the pipeline exposes no confidence score or threshold",\n'
        '        "sample_kind": sample_kind,\n'
        '        "n_utterances": n_eval,\n'
        '        "dataset": dict(dataset),\n'
        '        "baselines": [\n'
        '            {"id": "zero_shot_base_model", "word_error_rate": baseline_wer},\n'
        '        ],\n'
        '        "metrics": metrics,\n'
        '        "optimisation_history": [dict(row) for row in history],\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": (\n'
        '            "one seeded held-out split of one corpus; a few points of WER can change sign with another seed"\n'
        '        ),\n'
        '        "needs": (\n'
        '            "a referenced evaluation set from the deployment domain (speakers, microphones, noise) of "\n'
        '            "several "\n'
        '            "hundred utterances, and a check outside the adaptation distribution for forgetting, before any "\n'
        '            "generalisable claim"\n'
        '        ),\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '\n'
        '\n'
        'INPUT_SCHEMA: dict[str, Any] = {\n'
        '    "input": (\n'
        '        "one audio input: a local file path readable by the ASR stack, an http(s) URL, or a dict "\n'
        '        "{\'array\': float waveform, \'sampling_rate\': int}"\n'
        '    ),\n'
        '    "task": list(TASKS),\n'
        '    "language": "optional ISO language name/code forwarded to Whisper; None lets the model detect it",\n'
        '    "chunk_length_s": [MIN_CHUNK_LENGTH_S, MAX_CHUNK_LENGTH_S],\n'
        '    "preprocessing": (\n'
        '        "the transformers ASR pipeline resamples to 16 kHz, computes 128-bin log-Mel features in 30 s "\n'
        '        "windows and chunks longer audio; nothing is altered by this module"\n'
        '    ),\n'
        '}\n'
        '\n'
        '\n'
        'def _check_inputs(audio: Any, task: str, chunk_length_s: int) -> None:\n'
        '    """Raise ValueError/FileNotFoundError naming the first violated rule (shared with transcribe)."""\n'
        '    if task not in TASKS:\n'
        '        raise ValueError("task must be \'transcribe\' or \'translate\'")\n'
        '    if (\n'
        '        isinstance(audio, str | Path)\n'
        '        and not str(audio).startswith(("http://", "https://"))\n'
        '        and not Path(audio).is_file()\n'
        '    ):\n'
        '        raise FileNotFoundError(f"audio file not found: {audio}")\n'
        '    if not MIN_CHUNK_LENGTH_S <= chunk_length_s <= MAX_CHUNK_LENGTH_S:\n'
        '        raise ValueError(\n'
        '            f"chunk_length_s must be between {MIN_CHUNK_LENGTH_S} and {MAX_CHUNK_LENGTH_S} seconds"\n'
        '        )\n'
        '\n'
        '\n'
        'def _observe(audio: Any) -> dict[str, Any]:\n'
        '    if isinstance(audio, Mapping):\n'
        '        array = audio.get("array")\n'
        '        rate = audio.get("sampling_rate")\n'
        '        samples = len(array) if array is not None and hasattr(array, "__len__") else None\n'
        '        has_rate = isinstance(rate, int | float) and bool(rate)\n'
        '        seconds = round(samples / rate, 3) if samples is not None and has_rate else None\n'
        '        return {"kind": "waveform", "samples": samples, "sampling_rate": rate, "seconds": seconds}\n'
        '    if isinstance(audio, str | Path) and str(audio).startswith(("http://", "https://")):\n'
        '        return {"kind": "url", "value": str(audio)}\n'
        '    path = Path(audio)\n'
        '    return {"kind": "file", "name": path.name, "bytes": path.stat().st_size}\n'
        '\n'
        '\n'
        'def validate_inputs(\n'
        '    audio: str | Path | Mapping[str, Any],\n'
        '    *,\n'
        '    language: str | None = None,\n'
        '    task: str = "transcribe",\n'
        '    chunk_length_s: int = 30,\n'
        '    names: Sequence[str] | None = None,\n'
        ') -> dict[str, Any]:\n'
        '    """Validation stage: return the input manifest (schema, observed input, request, verdict).\n'
        '\n'
        '    Rejection is reported by raising exactly as ``transcribe`` would; a caller that wants the\n'
        '    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.\n'
        '    """\n'
        '    _check_inputs(audio, task, chunk_length_s)\n'
        '    if names is not None and len(names) != 1:\n'
        '        raise ValueError("names must have exactly one entry (one audio input per call)")\n'
        '    return {\n'
        '        "schema": dict(INPUT_SCHEMA),\n'
        '        "inputs": [{"id": names[0] if names else "audio-0", **_observe(audio)}],\n'
        '        "task": task,\n'
        '        "language": language,\n'
        '        "chunk_length_s": chunk_length_s,\n'
        '        "verdict": "accepted",\n'
        '        "findings": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '\n'
        '\n'
        'def evaluation_report(\n'
        '    result: Mapping[str, Any], reference: str | None = None, *, sample_kind: str = "public-sample"\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage: a machine-readable report even when nothing is measurable.\n'
        '\n'
        "    With a ``reference`` transcript the report carries ``word_error_rate`` (the repository's own\n"
        '    normalised WER) as sample-sanity evidence for that one utterance; without one the verdict is\n'
        '    ``not-measurable`` and the report says what would make the task measurable.\n'
        '    """\n'
        '    base = {\n'
        '        "task": f"automatic speech recognition ({result.get(\'task\', \'transcribe\')})",\n'
        '        "score_semantics": "generated transcript; the pipeline exposes no confidence score or threshold",\n'
        '        "sample_kind": sample_kind,\n'
        '        "n_utterances": 1,\n'
        '        "language": result.get("language"),\n'
        '        "baselines": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '    if reference is None:\n'
        '        return {\n'
        '            **base,\n'
        '            "metrics": [],\n'
        '            "verdict": "not-measurable",\n'
        '            "reason": "no reference transcript was supplied for the evaluated audio",\n'
        '            "needs": (\n'
        '                "reference transcripts for audio from the deployment domain (speakers, microphones, noise), "\n'
        '                "scored with word_error_rate after the same normalisation; several hundred utterances before "\n'
        '                "any rate is quoted"\n'
        '            ),\n'
        '        }\n'
        '    return {\n'
        '        **base,\n'
        '        "metrics": [\n'
        '            {\n'
        '                "id": "word_error_rate",\n'
        '                "value": word_error_rate(reference, str(result["text"])),\n'
        '                "normalisation": "casefold, punctuation removed, curly apostrophes folded",\n'
        '                "estimation": "single utterance, no dispersion estimate",\n'
        '            }\n'
        '        ],\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": "one referenced utterance from the tutorial sample; not a benchmark",\n'
        '        "needs": "a referenced evaluation set from the deployment domain for any generalisable WER claim",\n'
        '    }\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class WhisperASRPipeline:\n'
        '    _runner: Callable[..., dict[str, Any]]\n'
        '    device: str\n'
        '    adapter: str | None = None\n'
        '    adapter_sha256: str | None = None\n'
        '    source: str = "injected"\n'
        '\n'
        '    @property\n'
        '    def model(self) -> Any:\n'
        '        """The underlying Transformers model, for inspection (weights, config, dtype)."""\n'
        '        return self._runner.model\n'
        '\n'
        '    @classmethod\n'
        '    def from_pretrained(\n'
        '        cls,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '        adapter_dir: str | Path | None = None,\n'
        '    ) -> WhisperASRPipeline:\n'
        '        """Build the pipeline from a digest-verified local snapshot (or, only when\n'
        '        ``allow_download`` is set, from the Hub at the pinned revision), optionally merging a\n'
        '        PEFT LoRA adapter (safetensors only). There is no silent fallback: a missing or\n'
        '        unverified snapshot raises unless downloading was explicitly allowed."""\n'
        '        model, processor, source, resolved_device, dtype = _load_base(\n'
        '            device, weights_dir, allow_download, adapter_dir\n'
        '        )\n'
        '        adapter_path = None if adapter_dir is None else Path(adapter_dir)\n'
        '        return cls.from_model(\n'
        '            model,\n'
        '            processor,\n'
        '            adapter=None if adapter_path is None else str(adapter_path),\n'
        '            adapter_sha256=None if adapter_path is None else adapter_digest(adapter_path),\n'
        '            source=source,\n'
        '            device=resolved_device,\n'
        '            dtype=dtype,\n'
        '        )\n'
        '\n'
        '    @classmethod\n'
        '    def from_model(\n'
        '        cls,\n'
        '        model: Any,\n'
        '        processor: Any,\n'
        '        adapter: str | None = None,\n'
        '        adapter_sha256: str | None = None,\n'
        '        source: str = "injected",\n'
        '        device: str | None = None,\n'
        '        dtype: Any | None = None,\n'
        '    ) -> WhisperASRPipeline:\n'
        '        """Wrap an already-loaded Whisper model (plain or PEFT-wrapped) in the same decoding path.\n'
        '\n'
        '        Lets a caller that holds a live model — the fine-tuning tutorial, between training and\n'
        '        export — transcribe through exactly the pipeline that ``from_pretrained`` builds, so\n'
        '        in-memory and reloaded results are comparable.\n'
        '        """\n'
        '        from transformers import pipeline\n'
        '\n'
        '        resolved_device = device or str(model.device)\n'
        '        runner = pipeline(\n'
        '            "automatic-speech-recognition",\n'
        '            model=model,\n'
        '            tokenizer=processor.tokenizer,\n'
        '            feature_extractor=processor.feature_extractor,\n'
        '            torch_dtype=model.dtype if dtype is None else dtype,\n'
        '            device=resolved_device,\n'
        '        )\n'
        '        return cls(runner, resolved_device, adapter, adapter_sha256, source)\n'
        '\n'
        '    def transcribe(\n'
        '        self,\n'
        '        audio: str | Path | dict[str, Any],\n'
        '        *,\n'
        '        language: str | None = None,\n'
        '        task: str = "transcribe",\n'
        '        return_timestamps: bool = False,\n'
        '        chunk_length_s: int = 30,\n'
        '    ) -> dict[str, Any]:\n'
        '        _check_inputs(audio, task, chunk_length_s)\n'
        '\n'
        '        generate_kwargs: dict[str, Any] = {"task": task}\n'
        '        if language:\n'
        '            generate_kwargs["language"] = language\n'
        '        raw = self._runner(\n'
        '            str(audio) if isinstance(audio, Path) else audio,\n'
        '            return_timestamps=return_timestamps,\n'
        '            chunk_length_s=chunk_length_s,\n'
        '            generate_kwargs=generate_kwargs,\n'
        '        )\n'
        '        if not isinstance(raw, dict) or "text" not in raw:\n'
        '            raise RuntimeError("ASR backend returned an invalid result")\n'
        '        return {\n'
        '            "text": str(raw["text"]).strip(),\n'
        '            "chunks": raw.get("chunks"),\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '            "task": task,\n'
        '            "language": language,\n'
        '            "device": self.device,\n'
        '            "adapter": self.adapter,\n'
        '            "adapter_sha256": self.adapter_sha256,\n'
        '            "source": self.source,\n'
        '        }\n'
    ),
    'workshop.py': (
        '"""Standalone instructional ASR stages, carried verbatim in the generated notebook.\n'
        '\n'
        'No heavyweight imports at module load; CPU validation and metric tests need no model.\n'
        'The carried pipeline module supplies snapshot verification and model loading.\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import argparse\n'
        'import csv\n'
        'import hashlib\n'
        'import io\n'
        'import json\n'
        'import math\n'
        'import re\n'
        'import stat\n'
        'import time\n'
        'import unicodedata\n'
        'import urllib.request\n'
        'import uuid\n'
        'import zipfile\n'
        'from pathlib import Path, PurePosixPath\n'
        'from typing import Any\n'
        '\n'
        'NORMALIZATION = "nfc-casefold-apostrophe-word-internal-hyphen-v1"\n'
        'DATASET = {\n'
        '    "repo": "PolyAI/minds14",\n'
        '    "revision": "40ce77cb32a384e4d50a568e1ec39ac804019d33",\n'
        '    "path": "en-US/train-00000-of-00001.parquet",\n'
        '    "bytes": 34196221,\n'
        '    "sha256": "37004471dc896ce20771b3fdda0ee8fb33ec7a030fb4e2fd047c561ec0a1ee30",\n'
        '    "license": "CC-BY-4.0",\n'
        '}\n'
        'DECODE = {\n'
        '    "language": "en",\n'
        '    "task": "transcribe",\n'
        '    "do_sample": False,\n'
        '    "num_beams": 1,\n'
        '    "max_new_tokens": 256,\n'
        '    "return_timestamps": False,\n'
        '    "condition_on_prev_tokens": False,\n'
        '    "return_dict_in_generate": True,\n'
        '    "temperature": 0.0,\n'
        '}\n'
        '\n'
        '\n'
        'def digest(data: bytes) -> str:\n'
        '    return hashlib.sha256(data).hexdigest()\n'
        '\n'
        '\n'
        'def file_digest(path: Path) -> str:\n'
        '    with path.open("rb") as handle:\n'
        '        return hashlib.file_digest(handle, "sha256").hexdigest()\n'
        '\n'
        '\n'
        'def write_json(path: Path, value: Any) -> None:\n'
        '    path.parent.mkdir(parents=True, exist_ok=True)\n'
        '    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n'
        '\n'
        '\n'
        'def verify_file(path: Path, entry: dict) -> None:\n'
        '    if path.stat().st_size != entry["bytes"] or file_digest(path) != entry["sha256"]:\n'
        '        raise ValueError(f"Integrity failure: {path.name}; expected pinned size and SHA-256")\n'
        '\n'
        '\n'
        'def normalize(text: str) -> str:\n'
        '    text = unicodedata.normalize("NFC", text).casefold().translate(str.maketrans("’‘ʼ", "\'\'\'"))\n'
        '    return " ".join(re.sub(r"[^\\w\\s\'-]|(?<!\\w)[\'-]|[\'-](?!\\w)", " ", text).split())\n'
        '\n'
        '\n'
        'def alignment(reference: Any, hypothesis: Any) -> dict:\n'
        '    """Levenshtein alignment; ties prefer match/substitution, deletion, insertion."""\n'
        '    r, h = list(reference), list(hypothesis)\n'
        '    costs = [list(range(len(h) + 1))]\n'
        '    for i, token in enumerate(r, 1):\n'
        '        row = [i]\n'
        '        for j, other in enumerate(h, 1):\n'
        '            row.append(min(costs[i - 1][j - 1] + (token != other), costs[i - 1][j] + 1, row[-1] + 1))\n'
        '        costs.append(row)\n'
        '    i, j = len(r), len(h)\n'
        '    counts = {"S": 0, "D": 0, "I": 0, "N": len(r)}\n'
        '    steps = []\n'
        '    while i or j:\n'
        '        if i and j and costs[i][j] == costs[i - 1][j - 1] + (r[i - 1] != h[j - 1]):\n'
        '            op = "=" if r[i - 1] == h[j - 1] else "S"\n'
        '            a, b = r[i - 1], h[j - 1]\n'
        '            i, j = i - 1, j - 1\n'
        '        elif i and costs[i][j] == costs[i - 1][j] + 1:\n'
        '            op, a, b = "D", r[i - 1], None\n'
        '            i -= 1\n'
        '        else:\n'
        '            op, a, b = "I", None, h[j - 1]\n'
        '            j -= 1\n'
        '        if op != "=":\n'
        '            counts[op] += 1\n'
        '        steps.append({"op": op, "reference": a, "hypothesis": b})\n'
        '    return {**counts, "alignment": list(reversed(steps))}\n'
        '\n'
        '\n'
        'def score(reference: str | None, hypothesis: str) -> dict:\n'
        '    norm_h = normalize(hypothesis)\n'
        '    norm_r = None if reference is None else normalize(reference)\n'
        '    words = alignment((norm_r or "").split(), norm_h.split())\n'
        '    chars = alignment((norm_r or "").replace(" ", ""), norm_h.replace(" ", ""))\n'
        '    error = sum(chars[k] for k in ("S", "D", "I"))\n'
        '    reason = "missing-reference" if reference is None else ("empty-reference" if not norm_r else None)\n'
        '    return {\n'
        '        "reference": reference,\n'
        '        "hypothesis": hypothesis,\n'
        '        "normalized_reference": norm_r,\n'
        '        "normalized_hypothesis": norm_h,\n'
        '        **words,\n'
        '        "character_edits": error,\n'
        '        "character_N": chars["N"],\n'
        '        "wer": sum(words[k] for k in ("S", "D", "I")) / words["N"] if words["N"] else None,\n'
        '        "cer": error / chars["N"] if chars["N"] else None,\n'
        '        "reason": reason,\n'
        '    }\n'
        '\n'
        '\n'
        'def aggregate(rows: list[dict]) -> dict:\n'
        '    labelled = [r for r in rows if r["reference"] is not None]\n'
        '    totals = {k: sum(r[k] for r in labelled) for k in ("S", "D", "I", "N", "character_edits", "character_N")}\n'
        '    seconds = sum(r.get("seconds", 0) for r in rows)\n'
        '    elapsed = sum(r.get("inference_seconds", 0) for r in rows)\n'
        '    return {\n'
        '        **totals,\n'
        '        "wer": sum(totals[k] for k in ("S", "D", "I")) / totals["N"] if totals["N"] else None,\n'
        '        "cer": totals["character_edits"] / totals["character_N"] if totals["character_N"] else None,\n'
        '        "evaluated": len(rows),\n'
        '        "labelled": len(labelled),\n'
        '        "missing_reference": len(rows) - len(labelled),\n'
        '        "empty_reference": sum(r["N"] == 0 for r in labelled),\n'
        '        "exact_match_rate": sum(r["normalized_reference"] == r["normalized_hypothesis"] for r in labelled)\n'
        '        / len(labelled)\n'
        '        if labelled\n'
        '        else None,\n'
        '        "failed": sum(bool(r.get("failure")) for r in rows),\n'
        '        "truncated": sum(bool(r.get("truncated")) for r in rows),\n'
        '        "audio_seconds": seconds,\n'
        '        "inference_seconds": elapsed,\n'
        '        "real_time_factor": elapsed / seconds if seconds else None,\n'
        '    }\n'
        '\n'
        '\n'
        'def decode_audio(source: bytes | Path, max_seconds: float = 30) -> tuple[Any, dict]:\n'
        '    import numpy as np\n'
        '    import soundfile as sf\n'
        '    from scipy.signal import resample_poly\n'
        '\n'
        '    if isinstance(source, Path) and source.stat().st_size > 20 * 1024**2:\n'
        '        raise ValueError("Audio file exceeds 20 MiB")\n'
        '    if isinstance(source, bytes) and len(source) > 20 * 1024**2:\n'
        '        raise ValueError("Audio bytes exceed 20 MiB")\n'
        '    try:\n'
        '        with sf.SoundFile(io.BytesIO(source) if isinstance(source, bytes) else source) as handle:\n'
        '            rate, channels, frames = handle.samplerate, handle.channels, len(handle)\n'
        '            if handle.format not in {"WAV", "WAVEX", "FLAC"}:\n'
        '                raise ValueError("Only WAV/FLAC audio is supported")\n'
        '            if channels not in (1, 2) or not 8000 <= rate <= 48000:\n'
        '                raise ValueError("Expected mono/stereo at 8–48 kHz")\n'
        '            if not 1 <= frames / rate <= max_seconds:\n'
        '                raise ValueError(f"Expected 1–{max_seconds:g} seconds; never silently cropped")\n'
        '            samples = handle.read(frames=frames, dtype="float32", always_2d=True)\n'
        '    except RuntimeError as exc:\n'
        '        raise ValueError(f"Cannot decode WAV/FLAC: {exc}") from exc\n'
        '    if not np.isfinite(samples).all():\n'
        '        raise ValueError("Audio contains nonfinite samples")\n'
        '    if np.max(np.abs(samples)) > 1.01:\n'
        '        raise ValueError("Audio amplitude must be normalized PCM-like samples within [-1.01, 1.01]")\n'
        '    original_hash = digest(samples.astype("<f4").tobytes())\n'
        '    mono = samples.mean(axis=1, dtype=np.float64).astype(np.float32)\n'
        '    divisor = math.gcd(rate, 16000)\n'
        '    audio = np.asarray(\n'
        '        resample_poly(mono, 16000 // divisor, rate // divisor) if rate != 16000 else mono, dtype=np.float32\n'
        '    )\n'
        '    if not np.isfinite(audio).all():\n'
        '        raise ValueError("Audio became nonfinite during preprocessing")\n'
        '    return audio, {\n'
        '        "input_rate": rate,\n'
        '        "input_channels": channels,\n'
        '        "input_frames": frames,\n'
        '        "seconds": len(audio) / 16000,\n'
        '        "sampling_rate": 16000,\n'
        '        "decoded_audio_sha256": original_hash,\n'
        '        "waveform_sha256": digest(audio.astype("<f4").tobytes()),\n'
        '        "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '    }\n'
        '\n'
        '\n'
        'def noise_pair(waveform: Any, identity: str, snr: float = 10) -> tuple[Any, Any, dict]:\n'
        '    import numpy as np\n'
        '\n'
        '    if snr not in (5, 10, 20):\n'
        '        raise ValueError("SNR must be 5, 10, or 20 dB")\n'
        '    clean = np.asarray(waveform, dtype=np.float64)\n'
        '    if clean.ndim != 1 or not len(clean) or not np.isfinite(clean).all():\n'
        '        raise ValueError("Expected a finite mono waveform")\n'
        '    rms = float(np.sqrt(np.mean(clean**2)))\n'
        '    if rms < 1e-7:\n'
        '        raise ValueError("Near-silent input is not eligible for the speech noise experiment")\n'
        '    seed = int(digest(("noise-v1:" + identity).encode())[:16], 16)\n'
        '    noise = np.random.default_rng(seed).normal(size=len(clean))\n'
        '    noise -= noise.mean()\n'
        '    noise *= rms / (10 ** (snr / 20) * np.sqrt(np.mean(noise**2)))\n'
        '    # A shared worst-case gain over every offered SNR keeps clean playback invariant in exploration.\n'
        '    worst_noise = noise * 10 ** ((snr - 5) / 20)\n'
        '    gain = min(1.0, 0.99 / max(np.max(np.abs(clean)), np.max(np.abs(clean + worst_noise)), 1e-12))\n'
        '    a, b = (clean * gain).astype(np.float32), ((clean + noise) * gain).astype(np.float32)\n'
        '    actual = float(\n'
        '        20 * np.log10(np.sqrt(np.mean(a.astype(float) ** 2)) / np.sqrt(np.mean((b.astype(float) - a) ** 2)))\n'
        '    )\n'
        '    if abs(actual - snr) > 1e-3 or max(np.max(np.abs(a)), np.max(np.abs(b))) > 1:\n'
        '        raise ValueError("Noise construction failed SNR/headroom checks")\n'
        '    return (\n'
        '        a,\n'
        '        b,\n'
        '        {\n'
        '            "seed": seed,\n'
        '            "target_snr": snr,\n'
        '            "actual_snr": actual,\n'
        '            "gain": float(gain),\n'
        '            "clean_sha256": digest(a.astype("<f4").tobytes()),\n'
        '            "noisy_sha256": digest(b.astype("<f4").tobytes()),\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def fetch_dataset(cache: Path) -> Path:\n'
        '    cache.mkdir(parents=True, exist_ok=True)\n'
        '    path = cache / "minds14-en-US.parquet"\n'
        '    if not path.exists():\n'
        '        url = f"https://huggingface.co/datasets/{DATASET[\'repo\']}/resolve/{DATASET[\'revision\']}/{DATASET[\'path\']}"\n'
        '        temporary = path.with_suffix(".partial")\n'
        '        for attempt in range(3):\n'
        '            try:\n'
        '                with urllib.request.urlopen(url, timeout=90) as response, temporary.open("wb") as handle:\n'
        '                    count = 0\n'
        '                    while block := response.read(1024 * 1024):\n'
        '                        count += len(block)\n'
        '                        if count > DATASET["bytes"]:\n'
        '                            raise ValueError("Dataset download exceeds pinned size")\n'
        '                        handle.write(block)\n'
        '                verify_file(temporary, DATASET)\n'
        '                temporary.replace(path)\n'
        '                break\n'
        '            except (OSError, TimeoutError):\n'
        '                if attempt == 2:\n'
        '                    raise\n'
        '                time.sleep(attempt + 1)\n'
        '    verify_file(path, DATASET)\n'
        '    return path\n'
        '\n'
        '\n'
        'def sample_records(cache: Path, manifest: dict) -> list[dict]:\n'
        '    import pyarrow.parquet as pq\n'
        '\n'
        '    table = pq.read_table(fetch_dataset(cache)).to_pylist()\n'
        '    output = []\n'
        '    seen = set()\n'
        '    for entry in manifest["records"]:\n'
        '        row = table[entry["row"]]\n'
        '        if row["path"] != entry["id"] or row["transcription"] != entry["reference"]:\n'
        '            raise ValueError("Frozen sample identity mismatch")\n'
        '        raw = row["audio"]["bytes"]\n'
        '        if digest(raw) != entry["raw_sha256"]:\n'
        '            raise ValueError("Frozen audio bytes mismatch")\n'
        '        waveform, metadata = decode_audio(raw, max_seconds=20)\n'
        '        if metadata["decoded_audio_sha256"] != entry["decoded_audio_sha256"]:\n'
        '            raise ValueError("Decoded audio identity mismatch")\n'
        '        if metadata["decoded_audio_sha256"] in seen:\n'
        '            raise ValueError("Duplicate recording across sample roles")\n'
        '        seen.add(metadata["decoded_audio_sha256"])\n'
        '        output.append({**entry, **metadata, "waveform": waveform})\n'
        '    if [sum(r["role"] == role for r in output) for role in ("evaluation", "activity")] != [64, 16]:\n'
        '        raise ValueError("Expected frozen 64 evaluation / 16 activity recordings")\n'
        '    return output\n'
        '\n'
        '\n'
        'def safe_relative(value: str) -> str:\n'
        '    path = PurePosixPath(value)\n'
        '    if not value or "\\\\" in value or ":" in value or path.is_absolute() or ".." in path.parts:\n'
        '        raise ValueError(f"Unsafe relative path: {value!r}")\n'
        '    if str(path) != value or value.endswith("/"):\n'
        '        raise ValueError(f"Noncanonical file path: {value!r}")\n'
        '    return value\n'
        '\n'
        '\n'
        'def load_byod(path: Path, output: Path) -> list[dict]:\n'
        '    """Bounded WAV/FLAC or ZIP input. Never extracts to user-selected filesystem paths."""\n'
        '    if not path.is_file():\n'
        '        raise ValueError("BYOD_PATH must identify a WAV, FLAC, or ZIP file")\n'
        '    if path.stat().st_size > 100 * 1024**2:\n'
        '        raise ValueError("Input exceeds 100 MiB")\n'
        '    if path.suffix.lower() in {".wav", ".flac"}:\n'
        '        payload = {path.name: path.read_bytes()}\n'
        '    elif path.suffix.lower() == ".zip":\n'
        '        with zipfile.ZipFile(path) as archive:\n'
        '            infos = archive.infolist()\n'
        '            if len(infos) > 100 or sum(i.file_size for i in infos) > 200 * 1024**2:\n'
        '                raise ValueError("ZIP exceeds 100 members or 200 MiB expanded")\n'
        '            names = set()\n'
        '            payload = {}\n'
        '            for info in infos:\n'
        '                name = info.filename\n'
        '                if info.is_dir():\n'
        '                    safe_relative(name.rstrip("/"))\n'
        '                    continue\n'
        '                safe_relative(name)\n'
        '                if stat.S_ISLNK(info.external_attr >> 16) or name.casefold() in names:\n'
        '                    raise ValueError(f"ZIP contains a symlink or case-insensitive collision: {name}")\n'
        '                names.add(name.casefold())\n'
        '                if info.file_size > 20 * 1024**2:\n'
        '                    raise ValueError(f"ZIP member exceeds 20 MiB: {name}")\n'
        '                payload[name] = archive.read(info)\n'
        '    else:\n'
        '        raise ValueError("Only WAV, FLAC, or ZIP inputs are supported")\n'
        '    audios = {n for n in payload if Path(n).suffix.lower() in {".wav", ".flac"}}\n'
        '    undeclared = sorted(set(payload) - audios - {"transcripts.csv"})\n'
        '    if undeclared:\n'
        '        raise ValueError(\n'
        '            "ZIP has unsupported or undeclared files: "\n'
        '            + ", ".join(undeclared[:5])\n'
        '            + (" ..." if len(undeclared) > 5 else "")\n'
        '            + ". Keep only WAV/FLAC clips and an optional transcripts.csv, then run the BYOD cell again."\n'
        '        )\n'
        '    if not 1 <= len(audios) <= 20:\n'
        '        raise ValueError("Expected 1–20 audio clips")\n'
        '    if "transcripts.csv" in payload:\n'
        '        reader = csv.DictReader(io.StringIO(payload["transcripts.csv"].decode("utf-8-sig")))\n'
        '        if reader.fieldnames != ["id", "file", "reference_text"]:\n'
        '            raise ValueError("CSV schema must be id,file,reference_text")\n'
        '        rows = list(reader)\n'
        '        if any(None in r or any(v is None for v in r.values()) for r in rows):\n'
        '            raise ValueError("Malformed CSV row")\n'
        '        if len(rows) != len(audios) or {r["file"] for r in rows} != audios:\n'
        '            raise ValueError("Every audio file must be declared exactly once")\n'
        '        for row in rows:\n'
        '            safe_relative(row["file"])\n'
        '        labelled = [bool(normalize(r["reference_text"])) for r in rows]\n'
        '        if any(labelled) and not all(labelled):\n'
        '            raise ValueError("Mixed empty/nonempty references are ambiguous; supply all labels or omit CSV")\n'
        '    else:\n'
        '        rows = [{"id": str(i), "file": n, "reference_text": None} for i, n in enumerate(sorted(audios))]\n'
        '    if any(not r["id"].strip() for r in rows) or len({r["id"] for r in rows}) != len(rows):\n'
        '        raise ValueError("IDs must be unique and nonempty")\n'
        '    output.mkdir(parents=True, exist_ok=True)\n'
        '    records, hashes, total = [], {}, 0.0\n'
        '    for row in rows:\n'
        '        try:\n'
        '            waveform, meta = decode_audio(payload[row["file"]])\n'
        '        except ValueError as exc:\n'
        '            raise ValueError(f"{row[\'file\']}: {exc}. Fix or remove this clip, then run the BYOD cell again.") from exc\n'
        '        if meta["decoded_audio_sha256"] in hashes:\n'
        '            raise ValueError(f"Duplicate audio recordings: {hashes[meta[\'decoded_audio_sha256\']]} and {row[\'file\']}")\n'
        '        hashes[meta["decoded_audio_sha256"]] = row["file"]\n'
        '        total += meta["seconds"]\n'
        '        records.append(\n'
        '            {\n'
        '                "id": row["id"],\n'
        '                "file": row["file"],\n'
        '                "reference": row["reference_text"],\n'
        '                "role": "byod",\n'
        '                "waveform": waveform,\n'
        '                **meta,\n'
        '            }\n'
        '        )\n'
        '    if total > 300:\n'
        '        raise ValueError("BYOD exceeds 300 seconds total audio")\n'
        '    return records\n'
        '\n'
        '\n'
        'def export_report(output: Path, rows: list[dict], metrics: dict, provenance: dict) -> Path:\n'
        '    output.mkdir(parents=True, exist_ok=True)\n'
        '    write_json(output / "transcripts.json", rows)\n'
        '    write_json(output / "metrics.json", metrics)\n'
        '    write_json(output / "provenance.json", provenance)\n'
        '    fields = sorted({k for row in rows for k in row} - {"alignment"})\n'
        '    with (output / "transcripts.csv").open("w", encoding="utf-8", newline="") as handle:\n'
        '        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n'
        '        writer.writeheader()\n'
        '        writer.writerows(rows)\n'
        '    restored = json.loads((output / "transcripts.json").read_text(encoding="utf-8"))\n'
        '    for condition in {r.get("condition") for r in rows}:\n'
        '        if (\n'
        '            condition in metrics\n'
        '            and aggregate([r for r in restored if r.get("condition") == condition]) != metrics[condition]\n'
        '        ):\n'
        '            raise ValueError("Reloaded aggregate metrics mismatch")\n'
        '    with (output / "transcripts.csv").open(encoding="utf-8", newline="") as handle:\n'
        '        csv_rows = list(csv.DictReader(handle))\n'
        '    if (\n'
        '        restored != rows\n'
        '        or len(csv_rows) != len(rows)\n'
        '        or any(\n'
        '            a["hypothesis"] != b["hypothesis"] or (a["reference"] or "") != b["reference"]\n'
        '            for a, b in zip(rows, csv_rows, strict=True)\n'
        '        )\n'
        '    ):\n'
        '        raise ValueError("Report reload mismatch")\n'
        '    references = (\n'
        '        "References: your supplied reference_text; rates measure agreement with it.\\n"\n'
        '        if provenance.get("dataset") == "BYOD"\n'
        '        else "References: the dataset\'s supplied transcriptions. Per Gerz et al. (2021, section 4) they are the "\n'
        '        "1-best output of a commercial recognizer (Google ASR), not human transcripts, so rates measure "\n'
        '        "agreement between two automatic transcripts; an edit can be an error in either one.\\n"\n'
        '    )\n'
        '    summary = (\n'
        '        "# Whisper speech recognition report\\n\\n"\n'
        '        "Scope: sample agreement with supplied references; not a benchmark or calibrated reliability.\\n"\n'
        '        + references\n'
        '        + "\\n"\n'
        '        + json.dumps(metrics, indent=2, allow_nan=False)\n'
        '        + "\\n\\n"\n'
        '        "Noise is synthetic Gaussian noise. Missing references cannot establish transcription quality.\\n"\n'
        '        "Raw text is retained; WER may exceed 1. Personal learning notes are optional.\\n"\n'
        '    )\n'
        '    (output / "summary.md").write_text(summary, encoding="utf-8")\n'
        '    names = ["transcripts.json", "transcripts.csv", "metrics.json", "provenance.json", "summary.md"]\n'
        '    names += [\n'
        '        n\n'
        '        for n in ("input_manifest.json", "frozen_experiment.json", "noise_experiment.csv", "run_summary.json")\n'
        '        if (output / n).is_file()\n'
        '    ]\n'
        '    inventory = [\n'
        '        {"path": n, "bytes": (output / n).stat().st_size, "sha256": file_digest(output / n)} for n in names\n'
        '    ]\n'
        '    write_json(output / "inventory.json", inventory)\n'
        '    archive_path = output / "report.zip"\n'
        '    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:\n'
        '        for name in [*names, "inventory.json"]:\n'
        '            archive.write(output / name, name)\n'
        '    with zipfile.ZipFile(archive_path) as archive:\n'
        '        for entry in inventory:\n'
        '            data = archive.read(entry["path"])\n'
        '            if len(data) != entry["bytes"] or digest(data) != entry["sha256"]:\n'
        '                raise ValueError("Report ZIP integrity mismatch")\n'
        '    write_json(\n'
        '        output / "report_identity.json",\n'
        '        {"sha256": file_digest(archive_path), "bytes": archive_path.stat().st_size, "reload_verified": True},\n'
        '    )\n'
        '    return archive_path\n'
        '\n'
        '\n'
        'def transcriber(weights: Path) -> tuple[Any, dict]:\n'
        '    import torch\n'
        '    import whisper_reference as base\n'
        '\n'
        '    if not torch.cuda.is_available():\n'
        '        raise RuntimeError("Select a fresh Colab T4 GPU runtime; this canonical run requires CUDA")\n'
        '    model, processor = base.load_model(device="cuda:0", weights_dir=weights, allow_download=False)\n'
        '    model.eval()\n'
        '    model.generation_config.forced_decoder_ids = None\n'
        '\n'
        '    def transcribe(audio: Any) -> dict:\n'
        '        base.validate_inputs({"array": audio, "sampling_rate": 16000}, language="en", task="transcribe")\n'
        '        features = processor(audio, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)\n'
        '        features = {\n'
        '            k: v.to("cuda:0", dtype=torch.float16 if k == "input_features" else v.dtype)\n'
        '            for k, v in features.items()\n'
        '        }\n'
        '        torch.cuda.synchronize()\n'
        '        start = time.perf_counter()\n'
        '        with torch.inference_mode():\n'
        '            tokens = model.generate(**features, **DECODE).sequences\n'
        '        torch.cuda.synchronize()\n'
        '        ids = tokens[0].tolist()\n'
        '        return {\n'
        '            "text": processor.batch_decode(tokens, skip_special_tokens=True)[0].strip(),\n'
        '            "inference_seconds": time.perf_counter() - start,\n'
        '            "truncated": processor.tokenizer.eos_token_id not in ids[1:],\n'
        '            "sequence_tokens": len(ids),\n'
        '        }\n'
        '\n'
        '    return transcribe, {\n'
        '        "device": torch.cuda.get_device_name(),\n'
        '        "dtype": "float16",\n'
        '        "torch": torch.__version__,\n'
        '        "cuda": torch.version.cuda,\n'
        '        "generation": DECODE,\n'
        '        "effective_generation": model.generation_config.to_dict(),\n'
        '    }\n'
        '\n'
        '\n'
        'def run_stage(root: Path, stage: str, byod: Path | None = None, snr: int = 10) -> None:\n'
        '    import platform\n'
        '\n'
        '    import numpy as np\n'
        '    import soundfile as sf\n'
        '    import whisper_reference as base\n'
        '\n'
        '    started = time.perf_counter()\n'
        '    sample = json.loads((root / "sample.json").read_text(encoding="utf-8"))\n'
        '    if stage == "prepare":\n'
        '        records = sample_records(root / "cache", sample)\n'
        '        for row in records:\n'
        '            file = root / "audio" / (digest(row["id"].encode()) + ".wav")\n'
        '            file.parent.mkdir(exist_ok=True)\n'
        '            sf.write(file, row.pop("waveform"), 16000, subtype="FLOAT")\n'
        '            row["audio_path"] = str(file)\n'
        '        write_json(root / "prepared.json", records)\n'
        '        print(\n'
        '            json.dumps(\n'
        '                {\n'
        '                    "counts": {\n'
        '                        role: sum(r["role"] == role for r in records) for role in ("evaluation", "activity")\n'
        '                    },\n'
        '                    "seconds": sum(r["seconds"] for r in records),\n'
        '                    "data_seconds": time.perf_counter() - started,\n'
        '                }\n'
        '            )\n'
        '        )\n'
        '        return\n'
        '    if stage == "model":\n'
        '        import torch\n'
        '\n'
        '        if not torch.cuda.is_available():\n'
        '            raise RuntimeError("CUDA unavailable: select a fresh T4 runtime before downloading weights")\n'
        '        base.stage_missing_files(root / "weights", allow_download=True)\n'
        '        base.verify_snapshot(root / "weights")\n'
        '        print("Pinned model snapshot verified")\n'
        '        return\n'
        '    is_byod = stage == "byod"\n'
        '    if is_byod:\n'
        '        if byod is None:\n'
        '            raise ValueError("BYOD path required")\n'
        '        records = load_byod(byod, root / "byod_validation")\n'
        '    else:\n'
        '        records = json.loads((root / "prepared.json").read_text(encoding="utf-8"))\n'
        '        for row in records:\n'
        '            # The prepared file is already-validated 16 kHz model input, not new learner audio: resampling\n'
        '            # 8 kHz speech can overshoot 1.0 slightly, so it is reloaded as stored and identified by hash.\n'
        '            row["waveform"], rate = sf.read(row["audio_path"], dtype="float32")\n'
        '            if rate != 16000:\n'
        '                raise ValueError("Prepared waveform must be 16 kHz")\n'
        '            if digest(row["waveform"].astype("<f4").tobytes()) != row["waveform_sha256"]:\n'
        '                raise ValueError("Prepared waveform changed")\n'
        '        role = "evaluation" if stage == "evaluate" else "activity"\n'
        '        records = [r for r in records if r["role"] == role]\n'
        '    output = root / "outputs" / f"{stage}-{uuid.uuid4().hex[:12]}"\n'
        '    output.mkdir(parents=True)\n'
        '    write_json(root / f"{stage}_output.json", {"path": str(output)})\n'
        '    write_json(\n'
        '        output / "input_manifest.json", [{k: v for k, v in r.items() if k != "waveform"} for r in records]\n'
        '    )\n'
        '    frozen = {\n'
        '        "protocol": "whisper-workshop-v1",\n'
        '        "model": base.MODEL_ID,\n'
        '        "revision": base.MODEL_REVISION,\n'
        '        "normalization": NORMALIZATION,\n'
        '        "decoding": DECODE,\n'
        '        "snr": snr if stage in {"activity", "explore"} else None,\n'
        '        "role_digest": file_digest(output / "input_manifest.json"),\n'
        '        "sample_manifest": file_digest(root / "sample.json"),\n'
        '    }\n'
        '    write_json(output / "frozen_experiment.json", frozen)\n'
        '    infer, runtime = transcriber(root / "weights")\n'
        '    # One explicit, unscored warm-up; no reference or transcript prompt is used.\n'
        '    infer(np.zeros(16000, dtype=np.float32))\n'
        '    rows, pairs = [], []\n'
        '    for index, record in enumerate(records):\n'
        '        audio = record["waveform"]\n'
        '        conditions = [("clean", audio)]\n'
        '        if stage in {"activity", "explore"}:\n'
        '            clean, noisy, pair = noise_pair(audio, record["id"], snr)\n'
        '            conditions = [("clean", clean), ("noisy", noisy)]\n'
        '            pairs.append({"id": record["id"], **pair})\n'
        '        for condition, waveform in conditions:\n'
        '            try:\n'
        '                result = infer(waveform)\n'
        '                rows.append(\n'
        '                    {\n'
        '                        "id": record["id"],\n'
        '                        **({"file": record["file"]} if is_byod else {}),\n'
        '                        "role": record["role"],\n'
        '                        "condition": condition,\n'
        '                        "seconds": record["seconds"],\n'
        '                        **score(record["reference"], result.pop("text")),\n'
        '                        **result,\n'
        '                        "failure": None,\n'
        '                    }\n'
        '                )\n'
        '            except Exception as exc:\n'
        '                write_json(\n'
        '                    output / "failure.json",\n'
        '                    {\n'
        '                        "id": record["id"],\n'
        '                        "condition": condition,\n'
        '                        "type": type(exc).__name__,\n'
        '                        "message": str(exc),\n'
        '                        "completed": len(rows),\n'
        '                        "expected": len(records) * len(conditions),\n'
        '                    },\n'
        '                )\n'
        '                write_json(output / "partial_transcripts.json", rows)\n'
        '                raise\n'
        '            if index == 0 and not is_byod:\n'
        '                sf.write(output / f"listen_{condition}.wav", waveform, 16000, subtype="FLOAT")\n'
        '        print(f"{stage}: {index + 1}/{len(records)}", flush=True)\n'
        '    metrics = {\n'
        '        condition: aggregate([r for r in rows if r["condition"] == condition])\n'
        '        for condition in sorted({r["condition"] for r in rows})\n'
        '    }\n'
        '    metrics["empty_transcript_baseline"] = aggregate(\n'
        '        [\n'
        '            {\n'
        '                **r,\n'
        '                **score(r["reference"], ""),\n'
        '                "inference_seconds": 0,\n'
        '                "truncated": False,\n'
        '                "sequence_tokens": 0,\n'
        '            }\n'
        '            for r in rows\n'
        '            if r["condition"] == "clean"\n'
        '        ]\n'
        '    )\n'
        '    if pairs:\n'
        '        deltas = []\n'
        '        for pair in pairs:\n'
        '            matched = {r["condition"]: r for r in rows if r["id"] == pair["id"]}\n'
        '            pair.update(\n'
        '                {\n'
        '                    "clean_wer": matched["clean"]["wer"],\n'
        '                    "noisy_wer": matched["noisy"]["wer"],\n'
        '                    "delta_wer": matched["noisy"]["wer"] - matched["clean"]["wer"],\n'
        '                }\n'
        '            )\n'
        '            deltas.append(pair["delta_wer"])\n'
        '        metrics["paired"] = {\n'
        '            "improved": sum(d < 0 for d in deltas),\n'
        '            "unchanged": sum(d == 0 for d in deltas),\n'
        '            "worsened": sum(d > 0 for d in deltas),\n'
        '            "n": len(deltas),\n'
        '        }\n'
        '        with (output / "noise_experiment.csv").open("w", encoding="utf-8", newline="") as handle:\n'
        '            writer = csv.DictWriter(handle, fieldnames=list(pairs[0]))\n'
        '            writer.writeheader()\n'
        '            writer.writerows(pairs)\n'
        '    import importlib.metadata\n'
        '    import resource\n'
        '\n'
        '    import torch\n'
        '\n'
        '    provenance = {\n'
        '        "runtime": runtime,\n'
        '        "python": platform.python_version(),\n'
        '        "dataset": DATASET if not is_byod else "BYOD",\n'
        '        "normalization": NORMALIZATION,\n'
        '        "configuration": frozen,\n'
        '        "source": json.loads((root / "source.json").read_text()),\n'
        '        "lock_sha256": file_digest(root / "requirements.txt"),\n'
        '        "packages": {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()},\n'
        '    }\n'
        '    write_json(\n'
        '        output / "run_summary.json",\n'
        '        {\n'
        '            "status": "completed",\n'
        '            "profile": "TASK-INFERENCE",\n'
        '            "stage": stage,\n'
        '            "elapsed_seconds": time.perf_counter() - started,\n'
        '            "rows": len(rows),\n'
        '            "peak_gpu_bytes": torch.cuda.max_memory_allocated(),\n'
        '            "peak_host_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,\n'
        '            "release_status": "Candidate; hosted qualification recorded separately",\n'
        '        },\n'
        '    )\n'
        '    archive = export_report(output, rows, metrics, provenance)\n'
        '    print(json.dumps({"output": str(output), "report": str(archive), "metrics": metrics}, indent=2))\n'
        '\n'
        '\n'
        'def main() -> None:\n'
        '    parser = argparse.ArgumentParser()\n'
        '    parser.add_argument("--root", type=Path, required=True)\n'
        '    parser.add_argument(\n'
        '        "--stage", choices=["prepare", "model", "evaluate", "activity", "explore", "byod"], required=True\n'
        '    )\n'
        '    parser.add_argument("--byod", type=Path)\n'
        '    parser.add_argument("--snr", type=int, choices=[5, 10, 20], default=10)\n'
        '    args = parser.parse_args()\n'
        '    run_stage(args.root.resolve(), args.stage, args.byod, args.snr)\n'
        '\n'
        '\n'
        'if __name__ == "__main__":\n'
        '    main()\n'
    ),
    'sample.json': (
        '{\n'
        '  "dataset": {\n'
        '    "repo": "PolyAI/minds14",\n'
        '    "revision": "40ce77cb32a384e4d50a568e1ec39ac804019d33",\n'
        '    "path": "en-US/train-00000-of-00001.parquet",\n'
        '    "bytes": 34196221,\n'
        '    "sha256": "37004471dc896ce20771b3fdda0ee8fb33ec7a030fb4e2fd047c561ec0a1ee30",\n'
        '    "license": "CC-BY-4.0"\n'
        '  },\n'
        '  "seed": 42,\n'
        '  "source_count": 563,\n'
        '  "eligible_count": 533,\n'
        '  "exclusions": [\n'
        '    {\n'
        '      "row": 2,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602b9f97963e11ccd901cc32.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 103,\n'
        '      "id": "en-US~PAY_BILL/602baedc5f67b421554f64c5.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 110,\n'
        '      "id": "en-US~PAY_BILL/602b9fd95f67b421554f63c6.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 166,\n'
        '      "id": "en-US~FREEZE/602ba8f705f96973d6794493.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 187,\n'
        '      "id": "en-US~APP_ERROR/602ba131963e11ccd901cc45.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 189,\n'
        '      "id": "en-US~APP_ERROR/602bab0f963e11ccd901cd80.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 195,\n'
        '      "id": "en-US~APP_ERROR/602ba6dc963e11ccd901cce0.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 206,\n'
        '      "id": "en-US~APP_ERROR/602ba787963e11ccd901ccf4.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 210,\n'
        '      "id": "en-US~APP_ERROR/602ba6b9963e11ccd901ccdb.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 248,\n'
        '      "id": "en-US~ATM_LIMIT/602bad24963e11ccd901cdd0.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 254,\n'
        '      "id": "en-US~ATM_LIMIT/602ba7815f67b421554f6430.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 255,\n'
        '      "id": "en-US~ATM_LIMIT/602ba7f4963e11ccd901cd09.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 268,\n'
        '      "id": "en-US~ATM_LIMIT/602ba71405f96973d6794465.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 305,\n'
        '      "id": "en-US~ABROAD/602ba089bb1e6d0fbce91fd9.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 310,\n'
        '      "id": "en-US~ABROAD/602ba50b05f96973d6794438.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 313,\n'
        '      "id": "en-US~ABROAD/602ba5e2963e11ccd901ccbf.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 358,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602baafbbb1e6d0fbce9217d.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 368,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602ba44cbb1e6d0fbce92037.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 371,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602baf95963e11ccd901ce4a.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 372,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602ba078bb1e6d0fbce91fd7.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 404,\n'
        '      "id": "en-US~HIGH_VALUE_PAYMENT/602ba967963e11ccd901cd35.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 405,\n'
        '      "id": "en-US~HIGH_VALUE_PAYMENT/602ba9dd963e11ccd901cd4e.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 418,\n'
        '      "id": "en-US~HIGH_VALUE_PAYMENT/602b9f13963e11ccd901cc2e.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 456,\n'
        '      "id": "en-US~CASH_DEPOSIT/602ba8dd963e11ccd901cd25.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 462,\n'
        '      "id": "en-US~CASH_DEPOSIT/602ba8aebb1e6d0fbce92105.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 479,\n'
        '      "id": "en-US~CARD_ISSUES/602ba87bbb1e6d0fbce920f7.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 501,\n'
        '      "id": "en-US~CARD_ISSUES/602ba7efbb1e6d0fbce920dd.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 502,\n'
        '      "id": "en-US~CARD_ISSUES/602bae3005f96973d67944ee.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 505,\n'
        '      "id": "en-US~CARD_ISSUES/602ba333963e11ccd901cc6c.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    },\n'
        '    {\n'
        '      "row": 506,\n'
        '      "id": "en-US~CARD_ISSUES/602ba965bb1e6d0fbce92124.wav",\n'
        '      "reason": "Expected 1–20 seconds; never silently cropped"\n'
        '    }\n'
        '  ],\n'
        '  "grouping": "recording-disjoint; no reliable speaker/session field in source schema",\n'
        '  "records": [\n'
        '    {\n'
        '      "row": 95,\n'
        '      "id": "en-US~PAY_BILL/602baa7805f96973d67944ac.wav",\n'
        '      "reference": "I would like to make a payment",\n'
        '      "raw_sha256": "12c98fba0afd49b5f5e9d47287a5655cd32130b0d1a783a861cc92bc666c1948",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 17749,\n'
        '      "seconds": 2.218625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "b8d6579f78fb0eb8a4422edd3f7b10afe5b131694da205c4ec9711b8939e717e",\n'
        '      "waveform_sha256": "4411d39fc7814d45b00328ecf0d4b534eb745a20246f6ff7bdd2615b1e1f5cf0",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 4,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602b9d4cbb1e6d0fbce91fa4.wav",\n'
        '      "reference": "can you help me set up a joint bank account",\n'
        '      "raw_sha256": "3d0322962708e20e459ca54f4bfb6ee3d487a781a38a4cc65fa47a55bc2611a1",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 30720,\n'
        '      "seconds": 3.84,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "bd4a73aa5ab166211cfe1439b0203ff80e93bbadd196440c7c47e6a4678dbd8e",\n'
        '      "waveform_sha256": "fc408e7cafb4da574a8aa6b8a986b72ca4330e043646ba00da96d6076f7a16a3",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 120,\n'
        '      "id": "en-US~PAY_BILL/602ba97dbb1e6d0fbce92128.wav",\n'
        '      "reference": "I have to pay a bill",\n'
        '      "raw_sha256": "4ca5dfb85641c2309a34b315b9679b67eb17742e8ef26399f4c82591df0ba489",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 36181,\n'
        '      "seconds": 4.522625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "51fa2561a85c19a8a4c0b938c4cc4fa7c05f7ac0efcb7eebe75b478f93eae0fa",\n'
        '      "waveform_sha256": "e37c75601a394bd9d815d8ad8bfe8c6db8b5be098c1419559a674106930fb9e9",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 158,\n'
        '      "id": "en-US~FREEZE/602b997e963e11ccd901cbb5.wav",\n'
        '      "reference": "hello I just ate only lost my card at the airport will you be able to stop listen to increase the card",\n'
        '      "raw_sha256": "8176463a03e2c015bd359d47d0002de2f82f6eb33d8bf4208eb8a6377179efdc",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 75093,\n'
        '      "seconds": 9.386625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "5770a7b7128696134a07b32b66c0a26f1cda242bfa8f5ba0dc08e50a973f957f",\n'
        '      "waveform_sha256": "7d92c9d92a526eff6f19b67a01250a339edbbb97d9fc1e17fb390ea55e668668",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 286,\n'
        '      "id": "en-US~ABROAD/602b9fa105f96973d67943f0.wav",\n'
        '      "reference": "yes I\'m going to be traveling to the United Kingdom for a couple of weeks next month I need to know if I can use my card from from my bank account while I\'m in Europe",\n'
        '      "raw_sha256": "f46ac793fbb09f04e60d72c198ddb7391ecc7abdfedaeb456b69d10e05d63c7b",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 142677,\n'
        '      "seconds": 17.834625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "66aee5060281eea94a96c7f984b2621e40f987506c734feb05b6e9b116dd5149",\n'
        '      "waveform_sha256": "d279f368662e8804c43e46b54d43defdaa5a1ed07bd2f5ec2c4ca8755daabccb",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 469,\n'
        '      "id": "en-US~CASH_DEPOSIT/602baf14bb1e6d0fbce922a1.wav",\n'
        '      "reference": "deposit money into my account today how do I transfer my account",\n'
        '      "raw_sha256": "9aa91e50a5e7bfc5be462a045e7f9e36d2f23251b354ab732b39885571ef028d",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 76459,\n'
        '      "seconds": 9.557375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "8ed28a85c2c29b731d5317d00666cddf981ce1e85b17e158156998ef6ebc9ed9",\n'
        '      "waveform_sha256": "a81eedc9dde2bf7b7c6536d5e9ab6dca05e625bd2e33a76babf7d93ce7548abc",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 386,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602ba8395f67b421554f643d.wav",\n'
        '      "reference": "what are my most recent transactions",\n'
        '      "raw_sha256": "934d89e57612fbd61aa89becfa6aeaf7026cade9820b7def468355cb3ce48edc",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 40124,\n'
        '      "seconds": 5.0155,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "aed5ec5b9317d219b29e277ffbb76a9fa4d1dee9bf93ba20b12c5c3fc513615e",\n'
        '      "waveform_sha256": "4818453a2f6129cc2c3df8f5098d61e8b9057e516a1f4c8d7d32b3a44b6606d3",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 123,\n'
        '      "id": "en-US~PAY_BILL/602bb2c5963e11ccd901ce52.wav",\n'
        '      "reference": "I am calling because I want to pay a bill",\n'
        '      "raw_sha256": "4b34b29bff3ca880f7c52e57b1815e1860d73fdab9cada1de1f8d5a1546eb1c4",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 45325,\n'
        '      "seconds": 5.665625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "91b187050a65925f7017b18f5c602c1c23fb8e55ba57fc44b00e594a05b1afd8",\n'
        '      "waveform_sha256": "87f19d703a1b808d182439cae6bc8dc3751ae9df07a6e4dcf2f4b2d10df67c32",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 521,\n'
        '      "id": "en-US~CARD_ISSUES/602baa9d963e11ccd901cd6f.wav",\n'
        '      "reference": "my card isn\'t working in the office on speakerphone",\n'
        '      "raw_sha256": "753999fb5b618e233774a826b0084682053da311749a23433fbbfce9cce74818",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 55979,\n'
        '      "seconds": 6.997375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "d56838110e774ac1d7e0d8179cda38563d3f06f7f7d8a87a7349641b5fca82d8",\n'
        '      "waveform_sha256": "74bdbf0c4b6ca7e82c4aeb118ebcd0f7bd886b794b34ed53952e848d116eaad0",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 558,\n'
        '      "id": "en-US~BUSINESS_LOAN/602bac4d963e11ccd901cda8.wav",\n'
        '      "reference": "hi I\'m an account holder have been for 18 years and I\'d like to speak to somebody about taking out a business loan",\n'
        '      "raw_sha256": "e8e994093e78c1a7d1bda63867f453ea517575d380234614f0e540b4ba93df09",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 114005,\n'
        '      "seconds": 14.250625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "6c373f980b3fd2eafb14ada98259398893bad16349c485f5022208268947bec8",\n'
        '      "waveform_sha256": "dd867f6b321ad9109ed3ba309fad9b7af975a872d14299c9bd548d05bab002ac",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 539,\n'
        '      "id": "en-US~BUSINESS_LOAN/602bab57bb1e6d0fbce92194.wav",\n'
        '      "reference": "hello I am falling to talk about a new business loan that I want to take out and trying to start a business with my husband and I wanted to know how can you trust that process as far as applying for this business law",\n'
        '      "raw_sha256": "fbfea872f9f27a313302cf1e30b2517d6a36c0fcc7d2697e4c9da41267aea200",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 133803,\n'
        '      "seconds": 16.725375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "fa05b9ecdf22810a6c0c716f2ef6e74460f497dfcb530280d38c772208327ced",\n'
        '      "waveform_sha256": "014327ab64ff9db417e481523c8343fe34757d684be3bb518005b8e12e3694b3",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 225,\n'
        '      "id": "en-US~ADDRESS/602ba181bb1e6d0fbce91fe8.wav",\n'
        '      "reference": "yes hello I\'d like to change my address please",\n'
        '      "raw_sha256": "80ea94025e3b79969f4263d7c35c8f9baf41100fa6f1ad93ad63033a59d6ee43",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 31403,\n'
        '      "seconds": 3.925375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "e65d600a0519a3495ca5d9740b26af34d068f2336c433d1a1543ddbf2515de8b",\n'
        '      "waveform_sha256": "61d9b15896f54b22707e5092741e37440aa559e24acb579f6979c15e2b486a2f",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 36,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602bae9fbb1e6d0fbce92278.wav",\n'
        '      "reference": "how do I set up a joint account",\n'
        '      "raw_sha256": "78b92f513f5620813f2ec95da08862e381932ad3e7cf2356e1f1acf433fb5b46",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 30720,\n'
        '      "seconds": 3.84,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "e8d358b0aeeef17367ba7b4d5ab7f9476872ba25b3aa3ce22a968a9c2c8de013",\n'
        '      "waveform_sha256": "ce2c42811c81865815a27afae8266c295b4f73df64664a756ddb6e9e6337828f",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 447,\n'
        '      "id": "en-US~CASH_DEPOSIT/602b9a59963e11ccd901cbce.wav",\n'
        '      "reference": "how do I deposit money into my account",\n'
        '      "raw_sha256": "57a376818943f09a4e6efa99537717257409d2199cd52922902d15e94d699d2b",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 38229,\n'
        '      "seconds": 4.778625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "5dc96774b46e69e5c18960ec96da09ff5e868d6fb8fdfa4d646d8856fa0d0093",\n'
        '      "waveform_sha256": "3b6bea168d7498f1918a686afccf4fded2fc12d01e7f01f1638100926140d29c",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 91,\n'
        '      "id": "en-US~PAY_BILL/602ba4d505f96973d6794433.wav",\n'
        '      "reference": "yeah I know I want to pay bill I have a particular payment I need to make my credit card is due so I\'m to pay that bill now",\n'
        '      "raw_sha256": "3c66443a3d865996b5b222bb74b78419398cbb3f5a8f76436d2ab1e70f742593",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 78507,\n'
        '      "seconds": 9.813375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "83da7a73054b803d61aa734ca11767d0ca48725c7de0dad21ef2f377e2fe455c",\n'
        '      "waveform_sha256": "8e3f3e3e00b54f8b5725f1883a9872894f94cd732f4e1bda5964a087c60d25e5",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 379,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602bae74bb1e6d0fbce92262.wav",\n'
        '      "reference": "hi I would like to get my most recent bank transaction thank you very much",\n'
        '      "raw_sha256": "11658fa9390b3177444e1963716d8b8eb36acf580c64db473da58a65d90844de",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 35499,\n'
        '      "seconds": 4.437375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "33b4d02aaaca0d5efa0162295b4ededaf158ba591dd800d4af4bb68c7d76ff32",\n'
        '      "waveform_sha256": "8256e238157ad357bd70e60f11e1e1e2a0de9c294785b22fd60637d106def170",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 230,\n'
        '      "id": "en-US~ADDRESS/602ba42105f96973d6794424.wav",\n'
        '      "reference": "how do I change my address",\n'
        '      "raw_sha256": "b716bf7986ff2ad9bd59f3467c08e7afbfe8b9489e4ff0cfd8cac0a51f50bf05",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 37152,\n'
        '      "seconds": 4.644,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "97a68c71ffff28d34a4dc86d84f41c561ecf0d4aae24e0592280c4fb3aff8e49",\n'
        '      "waveform_sha256": "66322a68fa05ee1eee7fba28622b83713c573c0e0eeefbc9155d7d06c6ff3745",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 222,\n'
        '      "id": "en-US~ADDRESS/602ba0fe5f67b421554f63cc.wav",\n'
        '      "reference": "I want to change my address",\n'
        '      "raw_sha256": "e18c828dfb7a3bebac3e455c1aa25307888c3f341c620e30cb5313dd558be3e6",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 47104,\n'
        '      "seconds": 5.888,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "92c6b80355dcb59ec92f0bb74d39240174a3d544ce49d74c3699a360b887b6e6",\n'
        '      "waveform_sha256": "55cd79feada6be26c7c89aa05691c36480c0a6706f93658e5e58c971c07bdfba",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 380,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602baf29bb1e6d0fbce922a9.wav",\n'
        '      "reference": "please show me my latest transactions",\n'
        '      "raw_sha256": "2acdbb064ddc2b9b32f7fc6305bb70f0a5d7e77a8384c2544f0f75b89109d35e",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 24576,\n'
        '      "seconds": 3.072,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "12a6364976d05e68204159c55af68b8542bde64d3ebbe979817f0e7e7b650b82",\n'
        '      "waveform_sha256": "a5f8dfa5689cb1c6b58e9f8f1f8fd62206ff31a00d901ae512f9ab037b9cee6d",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 28,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602ba857bb1e6d0fbce920ec.wav",\n'
        '      "reference": "I\'m on your ass and my husband and I are trying to set up a joint account I can see where I set up an account by myself but I\'m not sure where to go to set up a joint account can you help me with this please thank you",\n'
        '      "raw_sha256": "b301d2799639929069463a82bf7ed4332c7930b61e829ebdc6c7de13f6356182",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 127659,\n'
        '      "seconds": 15.957375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "cd544ad4b4bd687d770fdb230a9ef87f215f955a7394b7cd2be96bab16c545c0",\n'
        '      "waveform_sha256": "adacc95f05d3f30a64cd0b1e2a903dcd33f35bd989279ec4ea45f0a3195d907f",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 438,\n'
        '      "id": "en-US~CASH_DEPOSIT/602baf245f67b421554f64cf.wav",\n'
        '      "reference": "can I deposit money",\n'
        '      "raw_sha256": "07b6b88acf68f51bf4abfd0dc7c419a68bf7953d1b19c907946206d54ad1e807",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 26624,\n'
        '      "seconds": 3.328,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "aea7523849664bad66362f3681eb49c307fed6fcaf092ebabaac2dc240581a46",\n'
        '      "waveform_sha256": "b77035256ed7aafda9a87a6b67146d027058eca4894b67695d07a1f70a21b901",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 304,\n'
        '      "id": "en-US~ABROAD/602ba090bb1e6d0fbce91fda.wav",\n'
        '      "reference": "hello I will be traveling in Europe next month and I wanted to know if my card was work abroad or if there\'s anything I need to do in order to set that up",\n'
        '      "raw_sha256": "8963aa89379fc3926121d47cb63c3f340713d094446e418b512a97078605edc9",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 120832,\n'
        '      "seconds": 15.104,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "8cf2af07df972e67b27bc99ad7b7738fec75a4da403ac87fb480462bc88bd7e3",\n'
        '      "waveform_sha256": "0882d7d5be004dd8917fa339a25fdb708308b3afd1f67ff8a665b6d2830ad7f6",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 70,\n'
        '      "id": "en-US~BALANCE/602ba418bb1e6d0fbce92030.wav",\n'
        '      "reference": "so you spent the money I\'d like to see my new account balance",\n'
        '      "raw_sha256": "da2157ba492948dc4c8d1b0e627feba5c166b512015ba896f5e32e467d16a024",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 40960,\n'
        '      "seconds": 5.12,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "c86c4db46cfab7d13017125eeaf7c5d6ad5b4a5c2df26b7d51c63e95e2445e72",\n'
        '      "waveform_sha256": "5e5f15eda69014c6214c49c5077f48169989043747c400ad0686935c1ce4c5e0",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 34,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602bafa95f67b421554f64dc.wav",\n'
        '      "reference": "yes I\'d like to set up a joint account I\'m allowed to anyone",\n'
        '      "raw_sha256": "6a4c6e1c1f1b8d89f36bf6abcb6a4778c93ab686244a002d1f46f5488a048124",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 116053,\n'
        '      "seconds": 14.506625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "502c57aca9b48b327aea9ca7b2fedde025d7405fc54d234a89feac6a9cc9f749",\n'
        '      "waveform_sha256": "36df31caa0481c41f972c58c33690f93fb7e1b27751fd803304b4cac4d4313c3",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 54,\n'
        '      "id": "en-US~BALANCE/602ba717bb1e6d0fbce920bc.wav",\n'
        '      "reference": "hello I\'m calling about my account balance",\n'
        '      "raw_sha256": "dcb09f260f42cd41aca487a3c37fee5a8e17189b88bed8f5978775b3e824f22e",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 72363,\n'
        '      "seconds": 9.045375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "efb3fb43d2d885d98f4eb6839c4ad731c5210a982c5b8fd052521b316df1e84d",\n'
        '      "waveform_sha256": "dac27a8aa024cd57b5529bc0391ce05d7a948c5fdb974a6d580ff3a355197208",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 177,\n'
        '      "id": "en-US~APP_ERROR/602ba642963e11ccd901cccc.wav",\n'
        '      "reference": "hi I\'m having issue with your app coming up on my phone and being able to walk again",\n'
        '      "raw_sha256": "d2445dd575553fe8b19f70805a574c188bd7ef21662c348b95bf628c674600a1",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 110592,\n'
        '      "seconds": 13.824,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "c9138a155f04e0fafde1cce38ef8750cc5054a942251755c4b9b08578c49861d",\n'
        '      "waveform_sha256": "a75d0b39eeb304c7510d3d1d33c59b0aa938430c876dbe8ca3b2c328b5c8bae3",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 53,\n'
        '      "id": "en-US~BALANCE/602baca4963e11ccd901cdb7.wav",\n'
        '      "reference": "show me my account balance",\n'
        '      "raw_sha256": "3671c05c4b09ca0da42f9092109cc1478419ef5d608cfb6d70ab3fc7510f0e67",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 45056,\n'
        '      "seconds": 5.632,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "6fc7134cff8ae1413e9ee566a3db43caac1c1f5f60162f6611ec6737a2548948",\n'
        '      "waveform_sha256": "4c919a46831c15ffa641989b7e05cfba81a4705477c4845be60fc30c5024b38b",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 473,\n'
        '      "id": "en-US~CASH_DEPOSIT/602bac19bb1e6d0fbce921bb.wav",\n'
        '      "reference": "where can I deposit money",\n'
        '      "raw_sha256": "286854ea2f19abfd9c701cc2091882a234471d4a749868884a577a975f82e76b",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 31403,\n'
        '      "seconds": 3.925375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "50ad2580febe93391f3fb400dfcf45ff6f3476dbbe48d5c2707eff65b0a41042",\n'
        '      "waveform_sha256": "4d2896d446b95df2f1bb43c9515d63aeae06f3e7fbffa27e197e878be1801fc2",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 522,\n'
        '      "id": "en-US~CARD_ISSUES/602ba087bb1e6d0fbce91fd8.wav",\n'
        '      "reference": "I need help activating my scars",\n'
        '      "raw_sha256": "2ae81ef4cb9e3570be5644a103dc5af2f35047aefdaadb7eebbeb028b77266b1",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 30720,\n'
        '      "seconds": 3.84,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "884dff536d7031763b6751464f70f4eb397078a94cb9a8ad8a2dd1a13bacbafc",\n'
        '      "waveform_sha256": "8c195fed46872005bbba417774c2e27b0165c31091236c444fa63d0ee62ae623",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 274,\n'
        '      "id": "en-US~ATM_LIMIT/602bac43963e11ccd901cda6.wav",\n'
        '      "reference": "how much money can I take out",\n'
        '      "raw_sha256": "70a4dea87cd05c27a775942750e93c3d4d01ffe5740077b04777d07ed47b28f5",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 43008,\n'
        '      "seconds": 5.376,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "37845f3767b6ea7c98518103f0f27b80a502acb99e75bb24fd5f87d27cd87b8a",\n'
        '      "waveform_sha256": "d7426f114f66f5971779c2637ca14a875b2d64841dc5f4ca725b81301771be4b",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 82,\n'
        '      "id": "en-US~BALANCE/602b97db5f67b421554f636a.wav",\n'
        '      "reference": "I would like to know what my current account balance shows",\n'
        '      "raw_sha256": "ef74bc44fadd08b6f6c287b3dc63f75cd808b6bba56737111f1a382c4baa7dc8",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 25941,\n'
        '      "seconds": 3.242625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "651b8e7de17e6d468d88a7e92735dc50eec89e8b2bf87b480f61f713eadc856c",\n'
        '      "waveform_sha256": "1b555e224ebed0fb700987254e3c4d05f7c8a96aff2e558795b36141a7baf6ea",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 406,\n'
        '      "id": "en-US~HIGH_VALUE_PAYMENT/602bafce05f96973d679451d.wav",\n'
        '      "reference": "hi I\'m trying to make a large payment online and isn\'t getting us I don\'t know what it is and I don\'t know what they\'re supposed to do can you help me please",\n'
        '      "raw_sha256": "2e9d0cb55a1758c61d4c3c125957e4128471f071d9191fa5d5e3d128dd52be81",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 147456,\n'
        '      "seconds": 18.432,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "8da63fb3c8493852eef37574df363e13c432d8b36b373f9868f79159f3fcbf4b",\n'
        '      "waveform_sha256": "999c4d8da11546d70fa450db0e01b48abfc7f9cc461ea7eb7c0a3d669fad5fe6",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 440,\n'
        '      "id": "en-US~CASH_DEPOSIT/602bab41bb1e6d0fbce9218f.wav",\n'
        '      "reference": "hi I want to deposit money into my savings account",\n'
        '      "raw_sha256": "502e72ef9eab8df394fd1175fee9f4dcda236b44d27be6268e98c4f5431886a2",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 30037,\n'
        '      "seconds": 3.754625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "191ee6275979cc0c4fd6189535947a67881eb7d392c3e5d9ae4fae23d0ed39f6",\n'
        '      "waveform_sha256": "793b52cef2263f7d96796a7cfdce619f73bc64e3f891074c9320203930fa040e",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 243,\n'
        '      "id": "en-US~ADDRESS/602bab7fbb1e6d0fbce921a1.wav",\n'
        '      "reference": "I want to change my address",\n'
        '      "raw_sha256": "f102775443dd62b91500e71801332d44cb0c13227a8eaa8c7b34d21341bf26af",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 55979,\n'
        '      "seconds": 6.997375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "a595a6a709a8717e6c994fed68ae8c98baaabe91696fda888a964fd63eaf915e",\n'
        '      "waveform_sha256": "c374bb3ead5fd1250f7c19683b2dca849806e16042d5e6bd377b1d48edff6b15",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 560,\n'
        '      "id": "en-US~BUSINESS_LOAN/602b98a5963e11ccd901cba2.wav",\n'
        '      "reference": "hi there I\'m working about a business loan I would like to take a new one out",\n'
        '      "raw_sha256": "c1b13af2dc07289e5a13b85211f90f01cdf4cb97f90fe2229a06b16e4c9ed7f5",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 77276,\n'
        '      "seconds": 9.6595,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "bd0b4543ef18da6ca011dbc0c5615edc6cc940ba003502d5ebd7f6eabf027cc3",\n'
        '      "waveform_sha256": "188330763482d1f1b6dd2b9d12e4a64616c53d93dd562808a478e5d013905eb2",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 381,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602baf425f67b421554f64d3.wav",\n'
        '      "reference": "did something in my account that I\'m not sure. Can you show me my last turned back because I\'m not familiar with that payment",\n'
        '      "raw_sha256": "f67dc164a61c48a4d73cc267399dd2a490e1a50f9e38ddd8032b0e40ba7b5edf",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 82603,\n'
        '      "seconds": 10.325375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "da02688882cb7f3b7f7abb1b400571c082bc0174bf68366f739680cd1f31797b",\n'
        '      "waveform_sha256": "1f8114c5115c6a4996b0ba50e41f05bcf9bdc538d77369c5b96dbcd4907923a2",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 483,\n'
        '      "id": "en-US~CARD_ISSUES/602ba23d963e11ccd901cc5c.wav",\n'
        '      "reference": "my card isn\'t working can you help me",\n'
        '      "raw_sha256": "a3134e34b8cf92759bc4dcac9676b188f857af5f06ede68dd6e7bd62c71bc4e7",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 64171,\n'
        '      "seconds": 8.021375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "df9bc69e2ec8a3451bc9c0634b8ec1aecb459668839d03de13b7e3c773031f4c",\n'
        '      "waveform_sha256": "5a1a398ff5cee69ce66d356d62a62b4993871878e7ee5c0d63991cc7f47965f5",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 143,\n'
        '      "id": "en-US~FREEZE/602ba3cc963e11ccd901cc7b.wav",\n'
        '      "reference": "freeze my card",\n'
        '      "raw_sha256": "b54a2ce15efe57f407a0f09e2b56fa77e2bc829545f21d4cbf42f09e94cdaad1",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 24576,\n'
        '      "seconds": 3.072,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "6efa6cd394aaa0dcf843bf12ba7aed0556941452c0aa41cf6108950d06233fa0",\n'
        '      "waveform_sha256": "11b93360aa8f58834e58df3b40ec36ccbdcda483da883d47ca2d06bc8161cf39",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 86,\n'
        '      "id": "en-US~PAY_BILL/602ba23a963e11ccd901cc5b.wav",\n'
        '      "reference": "my bills are due",\n'
        '      "raw_sha256": "0db220bdb9c73862875ac883df3fe05fbf70391ebf6f5402b92e8c8b672f99ad",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 13653,\n'
        '      "seconds": 1.706625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "1a82f287e8528bf589bcb69dab7e25d6d4ae77f054650264aac2e1e29559f7e1",\n'
        '      "waveform_sha256": "29b2efe286735746663cb743f404953ba6ca9af87d2621f5a59e069ab802b312",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 128,\n'
        '      "id": "en-US~FREEZE/602ba947bb1e6d0fbce92119.wav",\n'
        '      "reference": "good morning single lost my ATM card like a debit card I need to put a freezer",\n'
        '      "raw_sha256": "324eb4a04af63e7b6a5f4e939e16d67c93957a1099e0bc6b9cfd0c113fde4b60",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 90795,\n'
        '      "seconds": 11.349375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "cf6fbc2e66eda8f91aa11f8130d15637acf99cde71741ca0504c6b906bbfbf40",\n'
        '      "waveform_sha256": "f5644338e6dce57f01d0da1c38a20503bbaa04e1697f6a610e76eb1285abd850",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 196,\n'
        '      "id": "en-US~APP_ERROR/602babfdbb1e6d0fbce921b9.wav",\n'
        '      "reference": "it\'s not working now I cannot get into my account",\n'
        '      "raw_sha256": "aeacbaf56122bc98c109d43087b296a1da071fe19aada8f14f5c55c563b5ee70",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 53931,\n'
        '      "seconds": 6.741375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "318eaae46a2254fbffe7d7f2c6ec1b0b1a40e86a7432fdd83d4cfe02b0b2b551",\n'
        '      "waveform_sha256": "8daac8cf22ea5d70c3c6b6869987385a4a3e4ddd826e20cf125197dd1faf55a9",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 531,\n'
        '      "id": "en-US~BUSINESS_LOAN/602bae5dbb1e6d0fbce92255.wav",\n'
        '      "reference": "I\'d like some information on getting a new or existing loan",\n'
        '      "raw_sha256": "d357935bbfee0852842688d1055b91acc5665fb346923d3f099f2af44ae7dc67",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 59392,\n'
        '      "seconds": 7.424,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "fb0bceaf5edcef07e0875b6acd87565d3c668a07f35c2c53443709452fb83092",\n'
        '      "waveform_sha256": "b9c3c88a24c607e7b69edbdedaafc86cab9793c7506000353ace7e28fd4f6728",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 284,\n'
        '      "id": "en-US~ATM_LIMIT/602babf35f67b421554f6474.wav",\n'
        '      "reference": "what\'s the most amount of money I can withdraw at one time and during an entire day from an ATM machine",\n'
        '      "raw_sha256": "ef9e77cb1b1f72d24a52152dee25ba3d8cef8b0e8dd1826603a51965c6fc08fe",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 79872,\n'
        '      "seconds": 9.984,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "2ad53149d8b02ecf1631fb22ea71476c76a0787b3555a9a1933358679ff44180",\n'
        '      "waveform_sha256": "f7195e74d644c3ffa4e4c6c527c4622a5c7423a666c06082f9046e5b25d9edd0",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 562,\n'
        '      "id": "en-US~BUSINESS_LOAN/602ba6adbb1e6d0fbce920ae.wav",\n'
        '      "reference": "hi I\'m calling from Snyder hardware and I need to discuss a business loan it is an existing loan that I need to have a tree figured help please let me know thank you",\n'
        '      "raw_sha256": "bd04957a283e80c560ef99f2895d213f0e806e66011338d510174c3e3a6bbc8d",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 125611,\n'
        '      "seconds": 15.701375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "760b3218699a350e2b5708ab236e0d2d6260247b1c22d9f8e8c675071fefef18",\n'
        '      "waveform_sha256": "e40efe7fe57d52972e9b8ba59abe54a189c7a91e66f75396d60895deee716617",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 204,\n'
        '      "id": "en-US~APP_ERROR/602ba9e2963e11ccd901cd4f.wav",\n'
        '      "reference": "hi I\'m trying to use the banking app on my phone and currently my checking and savings account balance is not refreshing",\n'
        '      "raw_sha256": "4003aacbdb48400351c691134e78ec8733efbbfa41fd97b23a527b987a1505f1",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 62805,\n'
        '      "seconds": 7.850625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "9d4e5c759da320c2cd9719bd0d85258aa8f143699ae9ea6b9ce3aa18fc522d4d",\n'
        '      "waveform_sha256": "7a01ce6a2b5dc22acd20c83e832b755379ac49590d0a4e720f8dadf5b4992822",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 194,\n'
        '      "id": "en-US~APP_ERROR/602b98fc05f96973d67943a3.wav",\n'
        '      "reference": "the app isn\'t working correctly am I able to access my bank account",\n'
        '      "raw_sha256": "a2666137546b3e0719faadd77a8332cb0eb251f1b78234a0cba4af8d47110741",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 42325,\n'
        '      "seconds": 5.290625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "ad64c2cf8a321e63b24726faff9bbb6a108839befba7030e69128525e6154810",\n'
        '      "waveform_sha256": "9ef0c173d1b8e54ee9207d0498bdaf88be391036ccc8f19a49d2c4d3f46a0ecd",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 155,\n'
        '      "id": "en-US~FREEZE/602ba8da963e11ccd901cd24.wav",\n'
        '      "reference": "freeze my card immediately cease all purchases",\n'
        '      "raw_sha256": "1ceb559d6433315a3ac70cdf5ac08bc73f1ce72bca81f9a0d530f9c2f7f11cdc",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 57344,\n'
        '      "seconds": 7.168,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "9c27e62c117391714f0e4338fa505a23a076b94e31f028ef06d15f314ac648f4",\n'
        '      "waveform_sha256": "9809888e6527d7b2f1d2a781d93cc4b322b3dd6a3838e27f50e064435be8db42",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 150,\n'
        '      "id": "en-US~FREEZE/602bb461963e11ccd901ce54.wav",\n'
        '      "reference": "I would like you to freeze all transactions on my card please",\n'
        '      "raw_sha256": "72a646a586703a4c27568ba21c10b044654e45eac5c005b74694d49647dadfea",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 46421,\n'
        '      "seconds": 5.802625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "b146df941b403025771c36f9f0a6faa71aef0758114f846bf0516e0b9fc8fe4a",\n'
        '      "waveform_sha256": "83db5738bd9fbbb8d76d15b82e1b4a60cd88bde7f4f69e315fb32aa9c7db0bc1",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 510,\n'
        '      "id": "en-US~CARD_ISSUES/602babba05f96973d67944c1.wav",\n'
        '      "reference": "my payment is being declined",\n'
        '      "raw_sha256": "bc8d1c80f5baa82aa09c798e267b7194d184b25f73b1f8f68af3233cdac81d23",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 32085,\n'
        '      "seconds": 4.010625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "a1cd68f75de4186a51f6930229c3d06c69bfd44482ab8819e82849637e2dc656",\n'
        '      "waveform_sha256": "9354865563c4f43dbde4e289c6b183ffb0ec9cca5d1c3a228f9293b191db734f",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 217,\n'
        '      "id": "en-US~ADDRESS/602ba61b05f96973d6794448.wav",\n'
        '      "reference": "good morning yes I\'m moving and quick and I need to change my address of record",\n'
        '      "raw_sha256": "b882bf740750c06735c1e57cb4d70e9efa02bb44f9f71a60ed19f92ff93c8a94",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 73045,\n'
        '      "seconds": 9.130625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "74d735a40a56d0c43d64d1b5d12928a9a62d4f5d5e7ece30aa4f38282d6a8546",\n'
        '      "waveform_sha256": "d52fb96a481122d10c9d9ded04b991c666d48d21cf7551efc384cdefafd30e6a",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 65,\n'
        '      "id": "en-US~BALANCE/602bad6bbb1e6d0fbce921f7.wav",\n'
        '      "reference": "I\'m wondering if I could find out what my account balance is right now on my checking and savings thank you",\n'
        '      "raw_sha256": "8753ae8d3db6e118994d3916f8b79a26004cc67d32592795b80b484afc5acc49",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 93525,\n'
        '      "seconds": 11.690625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "2a538a334fb5b656935920745c38cbf7f2be4544ca72000c15a97affec0b1b5d",\n'
        '      "waveform_sha256": "05344229c50df4b0357cdb85cbe1882b713cebcec0a79c86ae6d557db224bdbc",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 62,\n'
        '      "id": "en-US~BALANCE/602ba38bbb1e6d0fbce9201d.wav",\n'
        '      "reference": "can you tell me what my current account balance is",\n'
        '      "raw_sha256": "c2aa6859fac4107c92d1bee1dffc0d1d44c973c3688833b2fda71909542404ad",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 37152,\n'
        '      "seconds": 4.644,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "d833fce6ed4b498482339b60417c65ba851d7400aff71d9f769c9b85f2e0bda8",\n'
        '      "waveform_sha256": "94e1758997b7d3ccfa8d7ba5cbe4d85970fc7e9b294f214602b3b79a7730bfa7",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 530,\n'
        '      "id": "en-US~BUSINESS_LOAN/602bae0c5f67b421554f64a2.wav",\n'
        '      "reference": "I\'d like to apply for a new business",\n'
        '      "raw_sha256": "778919bc655f6a603f3f1a11be3b0190915f691f506a10c4068dd58fd6352e95",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 40960,\n'
        '      "seconds": 5.12,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "d2e82cae83b5bd34275b46ce88a8ff27be326a901d4a235a02a9f59f4418d35b",\n'
        '      "waveform_sha256": "51379a4b6ecf3f4b16ce135450aed8c4d499f1ad4de24a5353820662a27aae22",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 355,\n'
        '      "id": "en-US~DIRECT_DEBIT/602bae94bb1e6d0fbce92271.wav",\n'
        '      "reference": "make a question about my direct debit that I have I don\'t know if it went into my account and I\'d like to know thank you",\n'
        '      "raw_sha256": "6149ee351749d4909dfe7d2ceb7e46c7b9dd3fc776919d97aa60e4992ad021a9",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 70315,\n'
        '      "seconds": 8.789375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "2253c5da324a88adc1779f85b49184b8725bc6c912cbb86e644b9cdb5e450d83",\n'
        '      "waveform_sha256": "893709185ea102927f11fbf8b462dcbb628e0d6ec1db3deb560dfe738cb25f00",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 544,\n'
        '      "id": "en-US~BUSINESS_LOAN/602baed35f67b421554f64c4.wav",\n'
        '      "reference": "I would like to get a business loan for a new business",\n'
        '      "raw_sha256": "974d9b2a8fd5db9da2540ff4b0734a690efedcff12e38d8571d2404cdc34c209",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 69846,\n'
        '      "seconds": 8.73075,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "0f3f3f17c2c7f33bcccf569047473ce406df4ad73572c3eb4d80d2fb5f29f97c",\n'
        '      "waveform_sha256": "007a8bbebb1b1694e281ac19990f7ec002b1d5e16e23e6bc77e70ca13755c5e9",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 144,\n'
        '      "id": "en-US~FREEZE/602b98ec05f96973d67943a0.wav",\n'
        '      "reference": "could you please freeze my card and make sure no transaction happens melim",\n'
        '      "raw_sha256": "627144a91b146c0a6093dfe96e91efe74bee7fa50cd9d4eaecee636e9d701e97",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 68360,\n'
        '      "seconds": 8.545,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "a83781031053025a8d9145be84cd9932fc5498d12d9caf01706a599ece67fe79",\n'
        '      "waveform_sha256": "b2832b2677cd71d4e6dd89f11e2aa32c87213413fbb7c8ce5aed4aa381d9cf22",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 250,\n'
        '      "id": "en-US~ATM_LIMIT/602b99bdbb1e6d0fbce91f42.wav",\n'
        '      "reference": "hi what\'s the maximum amount of money I can withdraw from",\n'
        '      "raw_sha256": "2ae64adb9e58df5193f1524ed7edfcb495a018a5cca388863c6f9519de238c12",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 53248,\n'
        '      "seconds": 6.656,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "fcc6de86805f9e9319b334a8d9fb399d57789501fbb0b9e81edc5290e6037264",\n'
        '      "waveform_sha256": "a3798e441a5e7122cae6c251c3c93be7e8a58a812fa2dfc0c7e3631d4ba83225",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 107,\n'
        '      "id": "en-US~PAY_BILL/602ba74a05f96973d679446d.wav",\n'
        '      "reference": "hello yes I\'m calling in regards to my credit card bill at a little bit early if I could that would be great thank you",\n'
        '      "raw_sha256": "9dd93257d39c3063b777bea7ff1c0ce418c474b9a4137b1095cd9d14ebfb647a",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 86699,\n'
        '      "seconds": 10.837375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "b1c54745c05ba518eef570ff820a0bc9ac68aa98c6d98ba14edf44e56de6f5db",\n'
        '      "waveform_sha256": "24fc44a9c19738c4bcc117e99d6967763810ba755013acc2250889cd1b55e48b",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 113,\n'
        '      "id": "en-US~PAY_BILL/602ba736bb1e6d0fbce920c3.wav",\n'
        '      "reference": "epic to make a payment on the bill",\n'
        '      "raw_sha256": "772af361a3c4b9c626a0f8e87b6ba59b540bdc1cf2d6c0e931cc1e05ed0419d6",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 28672,\n'
        '      "seconds": 3.584,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "ac864d56bfb3e019cf9cb57924f72491749bbd16e043aea454928713dcdf2f07",\n'
        '      "waveform_sha256": "f6b57294f7fa3371bb86a09d4202de667f296c569ff60188c4c34d39aba5bf00",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 471,\n'
        '      "id": "en-US~CASH_DEPOSIT/602b9d7cbb1e6d0fbce91fac.wav",\n'
        '      "reference": "how do I deposit money into my account",\n'
        '      "raw_sha256": "169298fe23b20a6e8fde0085fde2c04ca35965173e687aec5ccd8775688faf2a",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 38638,\n'
        '      "seconds": 4.82975,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "b25ba108f302c14ab6bb1f24349980878b5ac64723ea7463761a7e980563eeeb",\n'
        '      "waveform_sha256": "1c73d3f66ab92b53fecd3c8f7c82531c81b0dc82d380e2e913e22935f2464526",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 287,\n'
        '      "id": "en-US~ABROAD/602ba878963e11ccd901cd1d.wav",\n'
        '      "reference": "I\'m leaving for Amsterdam this weekend and I want to know if I can use my Wells Fargo credit card charges are there",\n'
        '      "raw_sha256": "5b867af87c3d04572238c7219cbf72aa8b0ff0a04928facc990f18dece3b0031",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 92160,\n'
        '      "seconds": 11.52,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "e09490402dc02e4053e4a4d3926268da28831fb4763043abe86e145b627d5255",\n'
        '      "waveform_sha256": "0415f5d73f69f66bcbd0f0789fcece324bbb6e0a73d948faca9109331cf1b0ee",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 231,\n'
        '      "id": "en-US~ADDRESS/602badbd05f96973d67944e6.wav",\n'
        '      "reference": "I need to change my address can I do that here thank you",\n'
        '      "raw_sha256": "ccfa2ac9c4670737d58f845ada19e6d6586eda0fe30a719e9ea4f2e814ff45d3",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 52565,\n'
        '      "seconds": 6.570625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "481e302b178270f1d10f30a92fd8bd28f2d1bf9905e96d6cb0887632191d45a6",\n'
        '      "waveform_sha256": "85e1c7bae710beba24fd9866b4404ed957a1a6a44163bb60e8fa95adce3cbf08",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 174,\n'
        '      "id": "en-US~APP_ERROR/602ba668bb1e6d0fbce9209d.wav",\n'
        '      "reference": "I\'m usually asleep use the app for my big services and today it\'s not working any help me access my account",\n'
        '      "raw_sha256": "030ef21c8ed2730f379243eed691a1a259ef00530786d2da0adeebcf6313ff92",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 96939,\n'
        '      "seconds": 12.117375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "81143ead8ead4dad10686456aafa9ef6011182b212d7f57d7ac327db380956f2",\n'
        '      "waveform_sha256": "3fcdb3a411ef72116205ced49b91d145f7dbe61e3a48eea5c1dfda3e7f312716",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 439,\n'
        '      "id": "en-US~CASH_DEPOSIT/602ba6fb5f67b421554f6420.wav",\n'
        '      "reference": "how can I make a deposit",\n'
        '      "raw_sha256": "168e2c680803801051cabb678d3538c56cf4165db9dd714fd02efd4a3a8f98d0",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 24576,\n'
        '      "seconds": 3.072,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "28806c9533e8f34f346b5777e5529da854f469e4100d11c285cb27c16cfe02b3",\n'
        '      "waveform_sha256": "b047c633c2878732edd968aa3c174a27aaa6396a1089cb398b37de6ed8d301e8",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "evaluation"\n'
        '    },\n'
        '    {\n'
        '      "row": 476,\n'
        '      "id": "en-US~CASH_DEPOSIT/602ba2c05f67b421554f63da.wav",\n'
        '      "reference": "hello I need to deposit money into my account",\n'
        '      "raw_sha256": "81a03b941bafd8e03b7e3b02d8267171a3423e0cade1127d72135cfa41f69ec3",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 63488,\n'
        '      "seconds": 7.936,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "4f2628807d239cd16ac62902dffc1407f87d071fbc1114b4dad078c6ecfef727",\n'
        '      "waveform_sha256": "97f0fa0b4ff8259523fc2450f0ae2e7d3519d3d4eab9297da25e7857a74c8bfb",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 323,\n'
        '      "id": "en-US~DIRECT_DEBIT/602ba4d0963e11ccd901cc9f.wav",\n'
        '      "reference": "I see this thing called direct debits I don\'t know exactly what those are can you tell me about them",\n'
        '      "raw_sha256": "5761f5532c203258bd1284bd324488b1b8577fd305a5ea392995b775d3bf9a09",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 48469,\n'
        '      "seconds": 6.058625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "3d7a2410b10708a981058f7551e4e61c2b8293695000f73a4050a4f014384de9",\n'
        '      "waveform_sha256": "7891dfa45bbd1932ee126d1cbef45d42c3ac769bbf5c9b5b9244b876527287b7",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 370,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602baeea963e11ccd901ce14.wav",\n'
        '      "reference": "show my latest transactions",\n'
        '      "raw_sha256": "fac4418ca9f86452a16bf571a7dca14711b23976060afa81f985953154d631b6",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 59392,\n'
        '      "seconds": 7.424,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "e39807020534ddcd8f90b5ec58e86dc4a2495655b6065a17c62b3ec7879bb79e",\n'
        '      "waveform_sha256": "ec1df37c15d49f58fc65f9ad6c7acef0d76d934d1fc83ab412e0608346a73916",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 180,\n'
        '      "id": "en-US~APP_ERROR/602baddbbb1e6d0fbce92215.wav",\n'
        '      "reference": "hi good morning to you don\'t like to make a complaint I\'ve been trying to look up my house I don\'t know why thank you",\n'
        '      "raw_sha256": "93d4110a3ac5bbaba63066edce60ecabf52630e737e31ca8edc85227f71f7f4b",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 79872,\n'
        '      "seconds": 9.984,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "23727303e47e871753b9a09e6e175bea0c97bda919c1d4d2fcbb7eb8093421c4",\n'
        '      "waveform_sha256": "a7c39ed9c60f387161d60d7f552b12be21715058fccfba5572b58e7f9ac9a789",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 534,\n'
        '      "id": "en-US~BUSINESS_LOAN/602b9026963e11ccd901cb62.wav",\n'
        '      "reference": "hi I\'m calling about a business loan",\n'
        '      "raw_sha256": "e1cf47724f28b574b6cb8e0309d288cd46fa4a729757e96c0e72eef057bd251c",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 43839,\n'
        '      "seconds": 5.479875,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "dc500620e9e7ff26e77796de9b5380462d445eb060bf01a7edd2e13aa4a7f105",\n'
        '      "waveform_sha256": "ae1ffc47a0fffe8bc7df8b368f86750e52046acf3f8017ccd030a887be92bc2e",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 48,\n'
        '      "id": "en-US~BALANCE/602baec5963e11ccd901ce0b.wav",\n'
        '      "reference": "account balance",\n'
        '      "raw_sha256": "b67f2b0e14ed061a8e4b20393b69c6555755fe5c7b702ad57ce663a4b016c4a1",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 27989,\n'
        '      "seconds": 3.498625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "94755b14cc5ef4adaf16a6df096004229f381bb97e2f252f142c7b357725d5f1",\n'
        '      "waveform_sha256": "99bc05dd696403d5e2af5825937e63a22661a3bcbc6ebdb86a19852157f763d4",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 351,\n'
        '      "id": "en-US~DIRECT_DEBIT/602baf1a05f96973d6794511.wav",\n'
        '      "reference": "I\'d like to see my direct debit",\n'
        '      "raw_sha256": "16c9c16b365e505e7c5232a91f6700355f5a0263bac809e167f49d00bf88dc2c",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 27989,\n'
        '      "seconds": 3.498625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "29c6f2fb98e55c7213aec9405185b6c60ef7a00db554fdf3535421c74654ce33",\n'
        '      "waveform_sha256": "5f9ae9ff384555cd2f867f7ce5f9453502ae93cdcb017e3980a4e8d6ab6e9cda",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 24,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602bad57bb1e6d0fbce921f2.wav",\n'
        '      "reference": "I\'d like to set up a joint account",\n'
        '      "raw_sha256": "e7d43fe867a05f614e512273d92768a714d1916e05fc514b3bd20dd85b3f8ed2",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 30720,\n'
        '      "seconds": 3.84,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "2636e7ada9a3d0bb222b8aade76af8e3dfc32d49983c9de244413f859b708d86",\n'
        '      "waveform_sha256": "c7c76b1171091dbf9da2d31a9a59649ccc394594c29ffff3a57df7bdf43460de",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 373,\n'
        '      "id": "en-US~LATEST_TRANSACTIONS/602ba44005f96973d6794427.wav",\n'
        '      "reference": "please tell me my most recent transaction",\n'
        '      "raw_sha256": "29c69158a0f1b79f30754e655664f631ef70510e76ac48096114910ffae46e06",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 47104,\n'
        '      "seconds": 5.888,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "edd119a197fff41bbf187c091c7f519574c6d5f711b0156a1ef024f492ebf0b9",\n'
        '      "waveform_sha256": "2ca59470af1c211800ae600a66048089ccdd323bb2c52a7bfee139c8e92fd8fc",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 297,\n'
        '      "id": "en-US~ABROAD/602ba3f25f67b421554f63ea.wav",\n'
        '      "reference": "hi I am traveling in Italy right now and I\'m just trying to figure out if my",\n'
        '      "raw_sha256": "441078229575a89e4490d9040601e84002e0dd7791fbc9f876f7e458ad46ec89",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 73561,\n'
        '      "seconds": 9.195125,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "d50c5a59902059ed5f151a8a8995524df063c17775eb99f3a47ed940d9f7ca30",\n'
        '      "waveform_sha256": "a29c3005e9a72191b9dbf0a51f99643618c66ab2cbeb2dfa781afe61ab0f145e",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 444,\n'
        '      "id": "en-US~CASH_DEPOSIT/602b9c95963e11ccd901cc00.wav",\n'
        '      "reference": "I\'d like to deposit money into my account",\n'
        '      "raw_sha256": "585adece8948f1356e1161ca5617a89f11cd56accc695628738f65963bb94eb5",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 31403,\n'
        '      "seconds": 3.925375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "ffa8b679125bda9add25892f88a5934333ef65e310a12d80d7e9ae3bd57110fb",\n'
        '      "waveform_sha256": "2f22c05306d91791b8f203ec25ffbdc390af483decd992cb6f95e380b62ba166",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 38,\n'
        '      "id": "en-US~JOINT_ACCOUNT/602ba6b8963e11ccd901ccda.wav",\n'
        '      "reference": "add someone to my account please",\n'
        '      "raw_sha256": "a209d0b5d165f0e179d9e2b4cc7dba134ed5959779104693e1a397c9508dab80",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 38229,\n'
        '      "seconds": 4.778625,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "8e388e85d504e829b40357bd75c5c03007c034e8a0ec3bd531ffcd0ca01da4e8",\n'
        '      "waveform_sha256": "99f2959c34ff9075d64e332a606b04ecfd35bf200be073a6cae555d5aa1cc31a",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 213,\n'
        '      "id": "en-US~ADDRESS/602ba79f05f96973d6794473.wav",\n'
        '      "reference": "hi I just moved and I need to change my address on file",\n'
        '      "raw_sha256": "ca134d5af3e529e057e52370ca4d40f06c279dae9a515a6445f2aef3e6d3153c",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 71680,\n'
        '      "seconds": 8.96,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "b811b015413ed1d4357fbbc4a7021725c04a1455f455222a72f4b37728178374",\n'
        '      "waveform_sha256": "37d0342e529b2abe850962516fdd5b7dd5cfc82ecafa65d92d57f03e694f56cd",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 329,\n'
        '      "id": "en-US~DIRECT_DEBIT/602badbd963e11ccd901cde8.wav",\n'
        '      "reference": "return to direct deposit",\n'
        '      "raw_sha256": "4b3f07098548ed56413eeb87ae7844c5e806cc0a71522fed5f976555febf878d",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 69632,\n'
        '      "seconds": 8.704,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "ea604e44f1ede2f79a1a63bb1b151362b69b7095a012d10bdec81bd1e482783e",\n'
        '      "waveform_sha256": "148167c9f4d82cb41c5653b1c1b74703d1803cd6735e28bdb0b8c82c0e277338",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 391,\n'
        '      "id": "en-US~HIGH_VALUE_PAYMENT/602b99f6963e11ccd901cbc2.wav",\n'
        '      "reference": "hello I just made a large payment online and it told me that I lose you receive an SMS message but I haven\'t received it yet you know what the problem is",\n'
        '      "raw_sha256": "bfd8e4ca8bb4540095f5584afc7dcc6b39186c1d37e12f32636a2fbe39fe7485",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 74411,\n'
        '      "seconds": 9.301375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "e919271613a4732c6322921e62e709edfd959e189197c7416fad9d61ac64c4b6",\n'
        '      "waveform_sha256": "a48b6e78831b1abaa862ab5896485c367c6bf1801913ddadea057d2c1117b498",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    },\n'
        '    {\n'
        '      "row": 76,\n'
        '      "id": "en-US~BALANCE/602badf5bb1e6d0fbce92224.wav",\n'
        '      "reference": "what\'s my account balance",\n'
        '      "raw_sha256": "0b42d3cea298041eccebb6f0080801632f19dda1c99149aae1e8c894a6b2c450",\n'
        '      "input_rate": 8000,\n'
        '      "input_channels": 1,\n'
        '      "input_frames": 35499,\n'
        '      "seconds": 4.437375,\n'
        '      "sampling_rate": 16000,\n'
        '      "decoded_audio_sha256": "3565b15c5e2f8dd602b721c3563278e9f88f02dcda78ddbe72ddc187c50f7af8",\n'
        '      "waveform_sha256": "e612f16da370fffb01baa23d26442445cf837b45922563e2e91024aa1aa7d1bf",\n'
        '      "preprocessing": "float32-channel-mean-scipy-resample_poly-v1",\n'
        '      "role": "activity"\n'
        '    }\n'
        '  ]\n'
        '}\n'
    ),
    'requirements.txt': (
        '# This file was autogenerated by uv via the following command:\n'
        '#    uv pip compile tools/workshop-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --output-file tools/workshop-requirements.lock\n'
        'accelerate==1.3.0 \\\n'
        '    --hash=sha256:518631c0adb80bd3d42fb29e7e2dc2256bcd7c786b0ba9119bbaa08611b36d9c \\\n'
        '    --hash=sha256:5788d9e6a7a9f80fed665cf09681c4dddd9dc056bea656db4140ffc285ce423e\n'
        '    # via -r tools/workshop-requirements.in\n'
        'certifi==2026.7.22 \\\n'
        '    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n'
        '    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n'
        '    # via requests\n'
        'cffi==2.1.1 \\\n'
        '    --hash=sha256:046bfc24911b37851ee1b51aab8bffe713d89c68c6a057b09484ce9fd5f69b4e \\\n'
        '    --hash=sha256:06c72bb76605a4b0cd0aad6930b69d4baf7dd5d806cfc409b824191099700e66 \\\n'
        '    --hash=sha256:0beceaabe56af686895136a2de78db54ecd8e4046b236b8fd6d6cb61389e9bf2 \\\n'
        '    --hash=sha256:154852545011f779917b11c78db2358d095da62a9a172b78ad0a583ee5adc0d0 \\\n'
        '    --hash=sha256:194cffa889098ced9976c3fc6340305e43f6303657d298da55366907c05c22d6 \\\n'
        '    --hash=sha256:19ee6127ee34de7d83ce3d371ebc5ed91addbdcc39f9ab15ce4eb35a4e534971 \\\n'
        '    --hash=sha256:1a18a57b58cfb21fc28d72e876acf10eaed67a1ed96226f92af4df681d571c4c \\\n'
        '    --hash=sha256:1aa5645c30469b09530c4ebca77ebf8f17618293c58f8549cb1a543a50236e7d \\\n'
        '    --hash=sha256:1dea0e4d7d4f11f619fe8c1d76caf49e24405b4b5743c0e3be16a500ecd930c9 \\\n'
        '    --hash=sha256:208f941bb9d18e768138677f0a6d2ce01f590df56043dda1df1535ac57c88517 \\\n'
        '    --hash=sha256:210019b6c7cf07f081b4c54635c8cf744377001350e29cc0f81c4377b4797735 \\\n'
        '    --hash=sha256:246fa40ce8645a614ff682e0b70f37134e460eaf93a775e0cbe3cca585a67a80 \\\n'
        '    --hash=sha256:25792eac27877609e7bb06d42ff88278a6624fff2ba9bbb523c09616b117e80f \\\n'
        '    --hash=sha256:27350daa11d4f10c540e6e89dada4c54feb7256ad03e9a4dc075ebad7ba360d1 \\\n'
        '    --hash=sha256:28907ab9bfb6aa13184cfc17c6b8e1023c5ab6fd7076d8c20a35e59fe04f8f29 \\\n'
        '    --hash=sha256:2ae64be792b8966f2c69538199728b290e34726562896df1e5dc8ffd8d8188e8 \\\n'
        '    --hash=sha256:31348097ff5bbe827ccc41795d4dd099d9f0625e7def00ee653c137a490c2a6c \\\n'
        '    --hash=sha256:3143d81e29e1e20a9ce10901ec369012947876596f75a222235965f2b7ae832e \\\n'
        '    --hash=sha256:3222ba5d678f80a030e6afbcc33dc1ae5cb45facabb61cee2c7016b8432fde48 \\\n'
        '    --hash=sha256:3311ed60d36f83378794e1009ac6258bafbf81f7888b4caa7b35a521e3f95813 \\\n'
        '    --hash=sha256:334644fbac4eff73d985a17a91226df55d0f394160c4cfb880e084c8f7161cac \\\n'
        '    --hash=sha256:34e261f78cb6ceaaa36f42f2613f4380d94d9c759a9c73c769ee6e0247364632 \\\n'
        '    --hash=sha256:363e05fa78e15116c3c32c210ee36884fd6b9afa6d440e47112c3bd511d64cb6 \\\n'
        '    --hash=sha256:398aff33cee2767e3e781d2554c54bd0dff386bb437581e0d8011fde1a942ec1 \\\n'
        '    --hash=sha256:3d22a20b1fb1632cc72c22f95f7b0d2961c3e1c235f245ba4c606c4771035659 \\\n'
        '    --hash=sha256:42a494cee34437f05546455144f2b5d9ac09b1face62bcfce597d2e521066688 \\\n'
        '    --hash=sha256:42e2f76b9455f5a9a844f770bf3e200ed3da0e15f5df3db9c31fe80b04b3d004 \\\n'
        '    --hash=sha256:42f6930c31dc7f50732c9ae793c2786c7b6b044195967bbdde40bb9be81c4cc0 \\\n'
        '    --hash=sha256:456a61fa52d579ebf9df2e9552ead5129855dbaff6c1e5a9b1bc408809bdc062 \\\n'
        '    --hash=sha256:471cee653ae88de62096552e6d24ccb4a5adb8c8c9f10b5054d0122c15bf2779 \\\n'
        '    --hash=sha256:49cbc70e6542d4ccccb936558d1064a8012541e78f821f955cff24e357776c94 \\\n'
        '    --hash=sha256:4a7c934f7360e8cd64fe9efadcbd10c7c6364f531e432b9a4bf5ccbc9e0e8b50 \\\n'
        '    --hash=sha256:4be96343e422f2dfcd12ab5c9f5aebe03f82f737c6bffeca6830b3875cb44aab \\\n'
        '    --hash=sha256:4f42141fc14250de6dde5ee7ea4432be017252d91f19c5ad043c084cea629cac \\\n'
        '    --hash=sha256:507a24c282e0f42f8ed737cf048572cbf580468da5555764a8331735e9c736b6 \\\n'
        '    --hash=sha256:51b31d1c98274844cfd7838ce00bfc27c7423a4dc00fc0772fc3331c2cc90676 \\\n'
        '    --hash=sha256:58acb8ab8e295e6c5ea12f888cbb13cf21511ef2a3303a23f4325c29d17fe5c1 \\\n'
        '    --hash=sha256:5a59cc1c4442bc3d5c703bf720b51138d0bfc173618807c9ee2490a7541dd3d9 \\\n'
        '    --hash=sha256:5bb4e7ea95dcd6a014a6fef62e62467d67d8e582326443f3d68e71d6320a9fcf \\\n'
        '    --hash=sha256:5c58fe613dc5e5336357eff555824a314d8e43282600435c8d1cb6a7a2fedd13 \\\n'
        '    --hash=sha256:5e7cecbaadb83884793e05828cee59b210b24583b9c7425d0ba6a754fe22eb4e \\\n'
        '    --hash=sha256:616f097f2fe415bc92a247f02e11f634e1f9e9a83d327e3c915c15089c87869e \\\n'
        '    --hash=sha256:63bbfd5ded17c4840ac07cd8f1c21ba9d9708141f840b324f422f41b207e3973 \\\n'
        '    --hash=sha256:64faea20f4e2613363a1a9b9c7dd73058f3ecd00133a511e72ad7c511658f527 \\\n'
        '    --hash=sha256:661c298b4821edebead0c91edd2b00374d67ad7c5a1f7a91d4442633b79d6a72 \\\n'
        '    --hash=sha256:68e62fe11f30d5ca8289242866f0a5291402d8529ca2178ab8afc5c9694ae890 \\\n'
        '    --hash=sha256:6a8dddef476fab96d066d578fc88526767b836ab5ab21754e1d5bf3879c31c7c \\\n'
        '    --hash=sha256:6e192623c49c94421616a5778fba35cf0d5a8d000650c1967ef4448ee5cdd990 \\\n'
        '    --hash=sha256:7225e4514edb64eb6740324353e0da0711954fd8d7da4576755b1c6e09b697cd \\\n'
        '    --hash=sha256:75f80557d1389eddbd0de2681f6a390a0c5338c31ddaa821381c203fc3fd50d9 \\\n'
        '    --hash=sha256:770de9db11e84213beec501cfcaa013b019820ca881e03344dea5844f7876d94 \\\n'
        '    --hash=sha256:7750c6449dff7864bb9bb27ddfb0267756189201a3afc911d82b3caacd70dfc3 \\\n'
        '    --hash=sha256:7bde5e4cc5c10140859842b9d383af292b22639a4dffb725314baf45968cef80 \\\n'
        '    --hash=sha256:7ce713ace7c0e4520535b42b77eaa742c16dab813978064913e5a3cf82973b41 \\\n'
        '    --hash=sha256:7da0c5eff80f0197f3b3d1232ec5a682a9325f4ae9016a78f5f5ca35f9ced1f5 \\\n'
        '    --hash=sha256:7dbb61fe3a7699468030f71bbe5f8a0e326a151daa91beb11a6fc1f980c55e1c \\\n'
        '    --hash=sha256:811bd1e21d32de12efca32393a0ab3f5133b54fce9bd44b8bd77ab07da14bf6a \\\n'
        '    --hash=sha256:8ef53b2de9bcb9197d31854256575d59dbac0cba72ac627bb291ef5eceb74be4 \\\n'
        '    --hash=sha256:937c0052c05a31ca1daf18de3158eed4dbfcb9cc107adbea227728d647be701e \\\n'
        '    --hash=sha256:9d2055050ea716bd38b7f7f1579c275386646b4894c155a3e2f3cd62ed41b7c6 \\\n'
        '    --hash=sha256:9f8d177621de5cb38ee3e731eda45d421db093ec0739f46a5594babda7987a98 \\\n'
        '    --hash=sha256:a2d7755bef5a12ed488f4ef1f1b69ee9191d7396083b755a5d2295f6edb4768b \\\n'
        '    --hash=sha256:a48d62ab9d6f4f98c983223a547af44be6ca3691074c31cecced6facd3ba2dc1 \\\n'
        '    --hash=sha256:a4f00aa42f75d6e4595e8866e748cc1705adc0cddfeb2ca86d0d03993d63ba03 \\\n'
        '    --hash=sha256:a6e721d4b0e45d5b65e87534470e67b18dcd092c83f68fba09f152b9cbc061af \\\n'
        '    --hash=sha256:a730a083190634c65cca36ba5f489531576ebd79bcd5c8e172130f6453127231 \\\n'
        '    --hash=sha256:a931079504ecc49efed7744c476a5c343a92fabf66dec2db95edb1b2fdc770e2 \\\n'
        '    --hash=sha256:aa9511c62d14da7aacc9b4bf51f3f697a621e83b2d6919008243c3aad168eea3 \\\n'
        '    --hash=sha256:ab36d55f9ed2d067327667c2fea18dda018eb628dd6347aa01dda6cf1f5d3836 \\\n'
        '    --hash=sha256:ad2c86c495b899d862ea0f4b42891b8713a3bd45dd4105c7fd51c2a72f39f3a5 \\\n'
        '    --hash=sha256:aeae0e330c9f6acd681f647d46cefd30c29f93e3392882e792e82080c9691399 \\\n'
        '    --hash=sha256:b0431303acaea1089ad4b3e9ce4e6518193def1118d4073ca848635ee4ea2e96 \\\n'
        '    --hash=sha256:b5bdfd1c873d4e093aabc0ca84c4ca6dbc4f752afb5c86f146d9742580c9da2e \\\n'
        '    --hash=sha256:baed1e86cc735622097354b9d1281406caf42ff42a886d29faa8e8d1630333be \\\n'
        '    --hash=sha256:c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf \\\n'
        '    --hash=sha256:c26608d2222fb1e94487e4a387d85f13eb55d5ed725cb25a0c589ac4ee60e7bc \\\n'
        '    --hash=sha256:c7659f22557c5a0bc4855cd635f55edec690cc008a40768527762cb9fb263455 \\\n'
        '    --hash=sha256:c8c69575568085ba0b1b10c0249d779a214aea6f6522e949a0fc9fb0fcb449d0 \\\n'
        '    --hash=sha256:c8d2c9fd1f2d16f780d15127abb050d13d1a76c03a4bd87d7e4980e45e511e12 \\\n'
        '    --hash=sha256:ca82be1a1d406ecfe1d25dc16cb33488e5a16bf4438c9fb590484ea29d92478b \\\n'
        '    --hash=sha256:cc572dace3f60ef98d7b12ff411d20f5362feb31a0439eab0085bbfd349982d7 \\\n'
        '    --hash=sha256:d18e5ac0f2f03f4f518d3e23db0f0cad7faa1da8620e9c09461d443bbf6e6692 \\\n'
        '    --hash=sha256:d28630f5854ab07ab1fd4aba756de52326c82e6be15d414b12793f1975048b54 \\\n'
        '    --hash=sha256:d9c275eaacd24aa73f94ffd6de08fc3f932424d8b6c376f4bed7cde376fe7bc3 \\\n'
        '    --hash=sha256:da0e573f9f97159390c89d9f1a9e41908b66d408cc5b58d08cf3847d844c531b \\\n'
        '    --hash=sha256:dd31f52ea1086513bb9df30f8fcee9b8918323ae067a3d5b78bc826a000712be \\\n'
        '    --hash=sha256:dddad92b554513a31f272570678ba307fb9f618f05e3d4a5eacafff9eae03e1d \\\n'
        '    --hash=sha256:df423d40ee8654634421812bc3b196da3f9bd7d32929da813f8394c4348a5358 \\\n'
        '    --hash=sha256:df913725b79db7bcf03448f36b7bf8815363417d5b58deecf9305e3e30f0f21a \\\n'
        '    --hash=sha256:e0bcb7e0f677f543555d2adff3bf19c05f66cdb4796e5ff602442ab2fe3c4ef7 \\\n'
        '    --hash=sha256:e2d65b31f36619cda3999b78b2aa9632e76b78448e7a56fc4240824200e7c4fc \\\n'
        '    --hash=sha256:e6e8cff14d6fb0be70a09c0bdc58096f501952d04624ebf867e0e56da2df8960 \\\n'
        '    --hash=sha256:f16c709686a78c727bbbf059f92b0bf41c6fc60deec706d2dc19f529175a6125 \\\n'
        '    --hash=sha256:f24fb43132a4c6b4cb4eb029492919b2db645be6808d738f244fd146c03c32cb \\\n'
        '    --hash=sha256:f53e442b08449d42821fa4a4fba000095af9f62742a500f978a9f557ec44339a \\\n'
        '    --hash=sha256:f5cfbc5fe74540d335175b656c725d74d90e3730c626d92575eea35029d9afaa \\\n'
        '    --hash=sha256:f81b3b8f3d4e343550fa4baa0e479bba9f2d29ce9c2e9b51d1ce1718d7442fcf \\\n'
        '    --hash=sha256:f8ec5e643a9a937f64e1999eb9f75d072263751912dc5cd06d3c85f8f44be7c3 \\\n'
        '    --hash=sha256:fb92203a88b3d3053034db775110081c49d28be6551923805e039924093761e4 \\\n'
        '    --hash=sha256:fcd22650c908d7b7da162bbfaab594a1227a15d1643a98c68b122ac642fa2264\n'
        '    # via soundfile\n'
        'charset-normalizer==3.5.1 \\\n'
        '    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \\\n'
        '    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \\\n'
        '    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \\\n'
        '    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \\\n'
        '    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \\\n'
        '    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \\\n'
        '    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \\\n'
        '    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \\\n'
        '    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \\\n'
        '    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \\\n'
        '    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \\\n'
        '    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \\\n'
        '    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \\\n'
        '    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \\\n'
        '    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \\\n'
        '    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \\\n'
        '    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \\\n'
        '    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \\\n'
        '    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \\\n'
        '    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \\\n'
        '    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \\\n'
        '    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \\\n'
        '    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \\\n'
        '    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \\\n'
        '    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \\\n'
        '    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \\\n'
        '    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \\\n'
        '    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \\\n'
        '    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \\\n'
        '    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \\\n'
        '    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \\\n'
        '    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \\\n'
        '    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \\\n'
        '    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \\\n'
        '    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \\\n'
        '    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \\\n'
        '    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \\\n'
        '    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \\\n'
        '    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \\\n'
        '    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \\\n'
        '    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \\\n'
        '    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \\\n'
        '    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \\\n'
        '    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \\\n'
        '    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \\\n'
        '    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \\\n'
        '    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \\\n'
        '    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \\\n'
        '    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \\\n'
        '    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \\\n'
        '    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \\\n'
        '    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \\\n'
        '    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \\\n'
        '    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \\\n'
        '    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \\\n'
        '    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \\\n'
        '    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \\\n'
        '    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \\\n'
        '    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \\\n'
        '    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \\\n'
        '    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \\\n'
        '    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \\\n'
        '    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \\\n'
        '    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \\\n'
        '    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \\\n'
        '    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \\\n'
        '    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \\\n'
        '    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \\\n'
        '    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \\\n'
        '    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \\\n'
        '    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \\\n'
        '    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \\\n'
        '    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \\\n'
        '    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \\\n'
        '    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \\\n'
        '    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \\\n'
        '    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \\\n'
        '    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \\\n'
        '    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \\\n'
        '    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \\\n'
        '    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \\\n'
        '    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \\\n'
        '    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \\\n'
        '    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \\\n'
        '    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \\\n'
        '    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \\\n'
        '    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \\\n'
        '    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \\\n'
        '    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \\\n'
        '    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \\\n'
        '    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \\\n'
        '    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \\\n'
        '    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \\\n'
        '    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \\\n'
        '    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \\\n'
        '    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \\\n'
        '    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \\\n'
        '    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \\\n'
        '    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \\\n'
        '    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \\\n'
        '    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \\\n'
        '    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \\\n'
        '    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \\\n'
        '    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \\\n'
        '    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \\\n'
        '    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \\\n'
        '    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \\\n'
        '    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \\\n'
        '    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \\\n'
        '    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \\\n'
        '    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \\\n'
        '    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \\\n'
        '    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \\\n'
        '    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \\\n'
        '    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \\\n'
        '    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \\\n'
        '    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \\\n'
        '    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \\\n'
        '    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \\\n'
        '    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \\\n'
        '    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \\\n'
        '    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \\\n'
        '    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \\\n'
        '    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \\\n'
        '    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \\\n'
        '    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \\\n'
        '    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \\\n'
        '    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \\\n'
        '    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \\\n'
        '    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \\\n'
        '    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \\\n'
        '    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \\\n'
        '    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \\\n'
        '    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \\\n'
        '    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \\\n'
        '    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \\\n'
        '    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \\\n'
        '    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \\\n'
        '    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \\\n'
        '    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \\\n'
        '    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \\\n'
        '    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \\\n'
        '    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \\\n'
        '    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \\\n'
        '    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \\\n'
        '    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \\\n'
        '    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \\\n'
        '    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \\\n'
        '    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \\\n'
        '    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \\\n'
        '    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \\\n'
        '    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \\\n'
        '    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \\\n'
        '    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \\\n'
        '    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \\\n'
        '    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \\\n'
        '    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \\\n'
        '    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \\\n'
        '    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \\\n'
        '    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \\\n'
        '    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \\\n'
        '    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \\\n'
        '    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \\\n'
        '    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \\\n'
        '    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \\\n'
        '    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \\\n'
        '    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \\\n'
        '    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \\\n'
        '    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \\\n'
        '    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \\\n'
        '    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \\\n'
        '    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f\n'
        '    # via requests\n'
        'filelock==4.0.4 \\\n'
        '    --hash=sha256:0df72be195ca7892216d16f2edce8d9b93a571f02402972020a8cff84c594c7b \\\n'
        '    --hash=sha256:90999ed63a26ccf86b93b959ab10cf1017f422d816be454ed54cbed263e71ab5\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        '    #   transformers\n'
        'fsspec==2026.9.0 \\\n'
        '    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n'
        '    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        'hf-xet==1.6.0 \\\n'
        '    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n'
        '    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n'
        '    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n'
        '    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n'
        '    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n'
        '    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n'
        '    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n'
        '    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n'
        '    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n'
        '    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n'
        '    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n'
        '    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n'
        '    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n'
        '    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n'
        '    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n'
        '    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n'
        '    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n'
        '    # via huggingface-hub\n'
        'huggingface-hub==0.36.2 \\\n'
        '    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n'
        '    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n'
        '    # via\n'
        '    #   -r tools/workshop-requirements.in\n'
        '    #   accelerate\n'
        '    #   tokenizers\n'
        '    #   transformers\n'
        'idna==3.20 \\\n'
        '    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n'
        '    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n'
        '    # via requests\n'
        'jinja2==3.1.6 \\\n'
        '    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n'
        '    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n'
        '    # via torch\n'
        'markupsafe==3.0.3 \\\n'
        '    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n'
        '    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n'
        '    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n'
        '    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n'
        '    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n'
        '    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n'
        '    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n'
        '    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n'
        '    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n'
        '    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n'
        '    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n'
        '    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n'
        '    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n'
        '    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n'
        '    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n'
        '    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n'
        '    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n'
        '    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n'
        '    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n'
        '    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n'
        '    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n'
        '    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n'
        '    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n'
        '    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n'
        '    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n'
        '    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n'
        '    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n'
        '    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n'
        '    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n'
        '    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n'
        '    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n'
        '    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n'
        '    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n'
        '    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n'
        '    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n'
        '    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n'
        '    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n'
        '    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n'
        '    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n'
        '    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n'
        '    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n'
        '    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n'
        '    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n'
        '    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n'
        '    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n'
        '    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n'
        '    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n'
        '    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n'
        '    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n'
        '    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n'
        '    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n'
        '    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n'
        '    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n'
        '    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n'
        '    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n'
        '    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n'
        '    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n'
        '    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n'
        '    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n'
        '    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n'
        '    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n'
        '    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n'
        '    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n'
        '    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n'
        '    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n'
        '    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n'
        '    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n'
        '    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n'
        '    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n'
        '    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n'
        '    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n'
        '    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n'
        '    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n'
        '    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n'
        '    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n'
        '    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n'
        '    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n'
        '    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n'
        '    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n'
        '    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n'
        '    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n'
        '    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n'
        '    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n'
        '    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n'
        '    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n'
        '    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n'
        '    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n'
        '    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n'
        '    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n'
        '    # via jinja2\n'
        'mpmath==1.3.0 \\\n'
        '    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n'
        '    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n'
        '    # via sympy\n'
        'networkx==3.7 \\\n'
        '    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n'
        '    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n'
        '    # via torch\n'
        'numpy==1.26.4 \\\n'
        '    --hash=sha256:03a8c78d01d9781b28a6989f6fa1bb2c4f2d51201cf99d3dd875df6fbd96b23b \\\n'
        '    --hash=sha256:08beddf13648eb95f8d867350f6a018a4be2e5ad54c8d8caed89ebca558b2818 \\\n'
        '    --hash=sha256:1af303d6b2210eb850fcf03064d364652b7120803a0b872f5211f5234b399f20 \\\n'
        '    --hash=sha256:1dda2e7b4ec9dd512f84935c5f126c8bd8b9f2fc001e9f54af255e8c5f16b0e0 \\\n'
        '    --hash=sha256:2a02aba9ed12e4ac4eb3ea9421c420301a0c6460d9830d74a9df87efa4912010 \\\n'
        '    --hash=sha256:2e4ee3380d6de9c9ec04745830fd9e2eccb3e6cf790d39d7b98ffd19b0dd754a \\\n'
        '    --hash=sha256:3373d5d70a5fe74a2c1bb6d2cfd9609ecf686d47a2d7b1d37a8f3b6bf6003aea \\\n'
        '    --hash=sha256:47711010ad8555514b434df65f7d7b076bb8261df1ca9bb78f53d3b2db02e95c \\\n'
        '    --hash=sha256:4c66707fabe114439db9068ee468c26bbdf909cac0fb58686a42a24de1760c71 \\\n'
        '    --hash=sha256:50193e430acfc1346175fcbdaa28ffec49947a06918b7b92130744e81e640110 \\\n'
        '    --hash=sha256:52b8b60467cd7dd1e9ed082188b4e6bb35aa5cdd01777621a1658910745b90be \\\n'
        '    --hash=sha256:60dedbb91afcbfdc9bc0b1f3f402804070deed7392c23eb7a7f07fa857868e8a \\\n'
        '    --hash=sha256:62b8e4b1e28009ef2846b4c7852046736bab361f7aeadeb6a5b89ebec3c7055a \\\n'
        '    --hash=sha256:666dbfb6ec68962c033a450943ded891bed2d54e6755e35e5835d63f4f6931d5 \\\n'
        '    --hash=sha256:675d61ffbfa78604709862923189bad94014bef562cc35cf61d3a07bba02a7ed \\\n'
        '    --hash=sha256:679b0076f67ecc0138fd2ede3a8fd196dddc2ad3254069bcb9faf9a79b1cebcd \\\n'
        '    --hash=sha256:7349ab0fa0c429c82442a27a9673fc802ffdb7c7775fad780226cb234965e53c \\\n'
        '    --hash=sha256:7ab55401287bfec946ced39700c053796e7cc0e3acbef09993a9ad2adba6ca6e \\\n'
        '    --hash=sha256:7e50d0a0cc3189f9cb0aeb3a6a6af18c16f59f004b866cd2be1c14b36134a4a0 \\\n'
        '    --hash=sha256:95a7476c59002f2f6c590b9b7b998306fba6a5aa646b1e22ddfeaf8f78c3a29c \\\n'
        '    --hash=sha256:96ff0b2ad353d8f990b63294c8986f1ec3cb19d749234014f4e7eb0112ceba5a \\\n'
        '    --hash=sha256:9fad7dcb1aac3c7f0584a5a8133e3a43eeb2fe127f47e3632d43d677c66c102b \\\n'
        '    --hash=sha256:9ff0f4f29c51e2803569d7a51c2304de5554655a60c5d776e35b4a41413830d0 \\\n'
        '    --hash=sha256:a354325ee03388678242a4d7ebcd08b5c727033fcff3b2f536aea978e15ee9e6 \\\n'
        '    --hash=sha256:a4abb4f9001ad2858e7ac189089c42178fcce737e4169dc61321660f1a96c7d2 \\\n'
        '    --hash=sha256:ab47dbe5cc8210f55aa58e4805fe224dac469cde56b9f731a4c098b91917159a \\\n'
        '    --hash=sha256:afedb719a9dcfc7eaf2287b839d8198e06dcd4cb5d276a3df279231138e83d30 \\\n'
        '    --hash=sha256:b3ce300f3644fb06443ee2222c2201dd3a89ea6040541412b8fa189341847218 \\\n'
        '    --hash=sha256:b97fe8060236edf3662adfc2c633f56a08ae30560c56310562cb4f95500022d5 \\\n'
        '    --hash=sha256:bfe25acf8b437eb2a8b2d49d443800a5f18508cd811fea3181723922a8a82b07 \\\n'
        '    --hash=sha256:cd25bcecc4974d09257ffcd1f098ee778f7834c3ad767fe5db785be9a4aa9cb2 \\\n'
        '    --hash=sha256:d209d8969599b27ad20994c8e41936ee0964e6da07478d6c35016bc386b66ad4 \\\n'
        '    --hash=sha256:d5241e0a80d808d70546c697135da2c613f30e28251ff8307eb72ba696945764 \\\n'
        '    --hash=sha256:edd8b5fe47dab091176d21bb6de568acdd906d1887a4584a15a9a96a1dca06ef \\\n'
        '    --hash=sha256:f870204a840a60da0b12273ef34f7051e98c3b5961b61b0c2c1be6dfd64fbcd3 \\\n'
        '    --hash=sha256:ffa75af20b44f8dba823498024771d5ac50620e6915abac414251bd971b4529f\n'
        '    # via\n'
        '    #   -r tools/workshop-requirements.in\n'
        '    #   accelerate\n'
        '    #   scipy\n'
        '    #   soundfile\n'
        '    #   transformers\n'
        'nvidia-cublas-cu12==12.4.5.8 \\\n'
        '    --hash=sha256:0f8aa1706812e00b9f19dfe0cdb3999b092ccb8ca168c0db5b8ea712456fd9b3 \\\n'
        '    --hash=sha256:2fc8da60df463fdefa81e323eef2e36489e1c94335b5358bcb38360adf75ac9b \\\n'
        '    --hash=sha256:5a796786da89203a0657eda402bcdcec6180254a8ac22d72213abc42069522dc\n'
        '    # via\n'
        '    #   nvidia-cudnn-cu12\n'
        '    #   nvidia-cusolver-cu12\n'
        '    #   torch\n'
        'nvidia-cuda-cupti-cu12==12.4.127 \\\n'
        '    --hash=sha256:5688d203301ab051449a2b1cb6690fbe90d2b372f411521c86018b950f3d7922 \\\n'
        '    --hash=sha256:79279b35cf6f91da114182a5ce1864997fd52294a87a16179ce275773799458a \\\n'
        '    --hash=sha256:9dec60f5ac126f7bb551c055072b69d85392b13311fcc1bcda2202d172df30fb\n'
        '    # via torch\n'
        'nvidia-cuda-nvrtc-cu12==12.4.127 \\\n'
        '    --hash=sha256:0eedf14185e04b76aa05b1fea04133e59f465b6f960c0cbf4e37c3cb6b0ea198 \\\n'
        '    --hash=sha256:a178759ebb095827bd30ef56598ec182b85547f1508941a3d560eb7ea1fbf338 \\\n'
        '    --hash=sha256:a961b2f1d5f17b14867c619ceb99ef6fcec12e46612711bcec78eb05068a60ec\n'
        '    # via torch\n'
        'nvidia-cuda-runtime-cu12==12.4.127 \\\n'
        '    --hash=sha256:09c2e35f48359752dfa822c09918211844a3d93c100a715d79b59591130c5e1e \\\n'
        '    --hash=sha256:64403288fa2136ee8e467cdc9c9427e0434110899d07c779f25b5c068934faa5 \\\n'
        '    --hash=sha256:961fe0e2e716a2a1d967aab7caee97512f71767f852f67432d572e36cb3a11f3\n'
        '    # via torch\n'
        'nvidia-cudnn-cu12==9.1.0.70 \\\n'
        '    --hash=sha256:165764f44ef8c61fcdfdfdbe769d687e06374059fbb388b6c89ecb0e28793a6f \\\n'
        '    --hash=sha256:6278562929433d68365a07a4a1546c237ba2849852c0d4b2262a486e805b977a\n'
        '    # via torch\n'
        'nvidia-cufft-cu12==11.2.1.3 \\\n'
        '    --hash=sha256:5dad8008fc7f92f5ddfa2101430917ce2ffacd86824914c82e28990ad7f00399 \\\n'
        '    --hash=sha256:d802f4954291101186078ccbe22fc285a902136f974d369540fd4a5333d1440b \\\n'
        '    --hash=sha256:f083fc24912aa410be21fa16d157fed2055dab1cc4b6934a0e03cba69eb242b9\n'
        '    # via torch\n'
        'nvidia-curand-cu12==10.3.5.147 \\\n'
        '    --hash=sha256:1f173f09e3e3c76ab084aba0de819c49e56614feae5c12f69883f4ae9bb5fad9 \\\n'
        '    --hash=sha256:a88f583d4e0bb643c49743469964103aa59f7f708d862c3ddb0fc07f851e3b8b \\\n'
        '    --hash=sha256:f307cc191f96efe9e8f05a87096abc20d08845a841889ef78cb06924437f6771\n'
        '    # via torch\n'
        'nvidia-cusolver-cu12==11.6.1.9 \\\n'
        '    --hash=sha256:19e33fa442bcfd085b3086c4ebf7e8debc07cfe01e11513cc6d332fd918ac260 \\\n'
        '    --hash=sha256:d338f155f174f90724bbde3758b7ac375a70ce8e706d70b018dd3375545fc84e \\\n'
        '    --hash=sha256:e77314c9d7b694fcebc84f58989f3aa4fb4cb442f12ca1a9bde50f5e8f6d1b9c\n'
        '    # via torch\n'
        'nvidia-cusparse-cu12==12.3.1.170 \\\n'
        '    --hash=sha256:9bc90fb087bc7b4c15641521f31c0371e9a612fc2ba12c338d3ae032e6b6797f \\\n'
        '    --hash=sha256:9d32f62896231ebe0480efd8a7f702e143c98cfaa0e8a76df3386c1ba2b54df3 \\\n'
        '    --hash=sha256:ea4f11a2904e2a8dc4b1833cc1b5181cde564edd0d5cd33e3c168eff2d1863f1\n'
        '    # via\n'
        '    #   nvidia-cusolver-cu12\n'
        '    #   torch\n'
        'nvidia-cusparselt-cu12==0.6.2 \\\n'
        '    --hash=sha256:0057c91d230703924c0422feabe4ce768841f9b4b44d28586b6f6d2eb86fbe70 \\\n'
        '    --hash=sha256:067a7f6d03ea0d4841c85f0c6f1991c5dda98211f6302cb83a4ab234ee95bef8 \\\n'
        '    --hash=sha256:df2c24502fd76ebafe7457dbc4716b2fec071aabaed4fb7691a201cde03704d9\n'
        '    # via torch\n'
        'nvidia-nccl-cu12==2.21.5 \\\n'
        '    --hash=sha256:8579076d30a8c24988834445f8d633c697d42397e92ffc3f63fa26766d25e0a0\n'
        '    # via torch\n'
        'nvidia-nvjitlink-cu12==12.4.127 \\\n'
        '    --hash=sha256:06b3b9b25bf3f8af351d664978ca26a16d2c5127dbd53c0497e28d1fb9611d57 \\\n'
        '    --hash=sha256:4abe7fef64914ccfa909bc2ba39739670ecc9e820c83ccc7a6ed414122599b83 \\\n'
        '    --hash=sha256:fd9020c501d27d135f983c6d3e244b197a7ccad769e34df53a42e276b0e25fa1\n'
        '    # via\n'
        '    #   nvidia-cusolver-cu12\n'
        '    #   nvidia-cusparse-cu12\n'
        '    #   torch\n'
        'nvidia-nvtx-cu12==12.4.127 \\\n'
        '    --hash=sha256:641dccaaa1139f3ffb0d3164b4b84f9d253397e38246a4f2f36728b48566d485 \\\n'
        '    --hash=sha256:781e950d9b9f60d8241ccea575b32f5105a5baf4c2351cab5256a24869f12a1a \\\n'
        '    --hash=sha256:7959ad635db13edf4fc65c06a6e9f9e55fc2f92596db928d169c0bb031e88ef3\n'
        '    # via torch\n'
        'packaging==26.3 \\\n'
        '    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n'
        '    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n'
        '    # via\n'
        '    #   accelerate\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'psutil==7.2.2 \\\n'
        '    --hash=sha256:0746f5f8d406af344fd547f1c8daa5f5c33dbc293bb8d6a16d80b4bb88f59372 \\\n'
        '    --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9 \\\n'
        '    --hash=sha256:11fe5a4f613759764e79c65cf11ebdf26e33d6dd34336f8a337aa2996d71c841 \\\n'
        '    --hash=sha256:1a571f2330c966c62aeda00dd24620425d4b0cc86881c89861fbc04549e5dc63 \\\n'
        '    --hash=sha256:1a7b04c10f32cc88ab39cbf606e117fd74721c831c98a27dc04578deb0c16979 \\\n'
        '    --hash=sha256:1fa4ecf83bcdf6e6c8f4449aff98eefb5d0604bf88cb883d7da3d8d2d909546a \\\n'
        '    --hash=sha256:2edccc433cbfa046b980b0df0171cd25bcaeb3a68fe9022db0979e7aa74a826b \\\n'
        '    --hash=sha256:7b6d09433a10592ce39b13d7be5a54fbac1d1228ed29abc880fb23df7cb694c9 \\\n'
        '    --hash=sha256:8c233660f575a5a89e6d4cb65d9f938126312bca76d8fe087b947b3a1aaac9ee \\\n'
        '    --hash=sha256:917e891983ca3c1887b4ef36447b1e0873e70c933afc831c6b6da078ba474312 \\\n'
        '    --hash=sha256:ab486563df44c17f5173621c7b198955bd6b613fb87c71c161f827d3fb149a9b \\\n'
        '    --hash=sha256:ae0aefdd8796a7737eccea863f80f81e468a1e4cf14d926bd9b6f5f2d5f90ca9 \\\n'
        '    --hash=sha256:b0726cecd84f9474419d67252add4ac0cd9811b04d61123054b9fb6f57df6e9e \\\n'
        '    --hash=sha256:b58fabe35e80b264a4e3bb23e6b96f9e45a3df7fb7eed419ac0e5947c61e47cc \\\n'
        '    --hash=sha256:c7663d4e37f13e884d13994247449e9f8f574bc4655d509c3b95e9ec9e2b9dc1 \\\n'
        '    --hash=sha256:e452c464a02e7dc7822a05d25db4cde564444a67e58539a00f929c51eddda0cf \\\n'
        '    --hash=sha256:e78c8603dcd9a04c7364f1a3e670cea95d51ee865e4efb3556a3a63adef958ea \\\n'
        '    --hash=sha256:eb7e81434c8d223ec4a219b5fc1c47d0417b12be7ea866e24fb5ad6e84b3d988 \\\n'
        '    --hash=sha256:ed0cace939114f62738d808fdcecd4c869222507e266e574799e9c0faa17d486 \\\n'
        '    --hash=sha256:eed63d3b4d62449571547b60578c5b2c4bcccc5387148db46e0c2313dad0ee00 \\\n'
        '    --hash=sha256:fd04ef36b4a6d599bbdb225dd1d3f51e00105f6d48a28f006da7f9822f2606d8\n'
        '    # via accelerate\n'
        'pyarrow==19.0.1 \\\n'
        '    --hash=sha256:008a4009efdb4ea3d2e18f05cd31f9d43c388aad29c636112c2966605ba33466 \\\n'
        '    --hash=sha256:0148bb4fc158bfbc3d6dfe5001d93ebeed253793fff4435167f6ce1dc4bddeae \\\n'
        '    --hash=sha256:1b93ef2c93e77c442c979b0d596af45e4665d8b96da598db145b0fec014b9136 \\\n'
        '    --hash=sha256:1c7556165bd38cf0cd992df2636f8bcdd2d4b26916c6b7e646101aff3c16f76f \\\n'
        '    --hash=sha256:335d170e050bcc7da867a1ed8ffb8b44c57aaa6e0843b156a501298657b1e972 \\\n'
        '    --hash=sha256:3bf266b485df66a400f282ac0b6d1b500b9d2ae73314a153dbe97d6d5cc8a99e \\\n'
        '    --hash=sha256:41f9706fbe505e0abc10e84bf3a906a1338905cbbcf1177b71486b03e6ea6608 \\\n'
        '    --hash=sha256:4982f8e2b7afd6dae8608d70ba5bd91699077323f812a0448d8b7abdff6cb5d3 \\\n'
        '    --hash=sha256:49a3aecb62c1be1d822f8bf629226d4a96418228a42f5b40835c1f10d42e4db6 \\\n'
        '    --hash=sha256:4d5d1ec7ec5324b98887bdc006f4d2ce534e10e60f7ad995e7875ffa0ff9cb14 \\\n'
        '    --hash=sha256:58d9397b2e273ef76264b45531e9d552d8ec8a6688b7390b5be44c02a37aade8 \\\n'
        '    --hash=sha256:5a9137cf7e1640dce4c190551ee69d478f7121b5c6f323553b319cac936395f6 \\\n'
        '    --hash=sha256:5bd1618ae5e5476b7654c7b55a6364ae87686d4724538c24185bbb2952679960 \\\n'
        '    --hash=sha256:65cf9feebab489b19cdfcfe4aa82f62147218558d8d3f0fc1e9dea0ab8e7905a \\\n'
        '    --hash=sha256:699799f9c80bebcf1da0983ba86d7f289c5a2a5c04b945e2f2bcf7e874a91911 \\\n'
        '    --hash=sha256:6c5941c1aac89a6c2f2b16cd64fe76bcdb94b2b1e99ca6459de4e6f07638d755 \\\n'
        '    --hash=sha256:6ebfb5171bb5f4a52319344ebbbecc731af3f021e49318c74f33d520d31ae0c4 \\\n'
        '    --hash=sha256:7a544ec12de66769612b2d6988c36adc96fb9767ecc8ee0a4d270b10b1c51e00 \\\n'
        '    --hash=sha256:7c1bca1897c28013db5e4c83944a2ab53231f541b9e0c3f4791206d0c0de389a \\\n'
        '    --hash=sha256:80b2ad2b193e7d19e81008a96e313fbd53157945c7be9ac65f44f8937a55427b \\\n'
        '    --hash=sha256:8464c9fbe6d94a7fe1599e7e8965f350fd233532868232ab2596a71586c5a429 \\\n'
        '    --hash=sha256:8f04d49a6b64cf24719c080b3c2029a3a5b16417fd5fd7c4041f94233af732f3 \\\n'
        '    --hash=sha256:96606c3ba57944d128e8a8399da4812f56c7f61de8c647e3470b417f795d0ef9 \\\n'
        '    --hash=sha256:99bc1bec6d234359743b01e70d4310d0ab240c3d6b0da7e2a93663b0158616f6 \\\n'
        '    --hash=sha256:ad76aef7f5f7e4a757fddcdcf010a8290958f09e3470ea458c80d26f4316ae89 \\\n'
        '    --hash=sha256:b4c4156a625f1e35d6c0b2132635a237708944eb41df5fbe7d50f20d20c17832 \\\n'
        '    --hash=sha256:b9766a47a9cb56fefe95cb27f535038b5a195707a08bf61b180e642324963b46 \\\n'
        '    --hash=sha256:c0fe3dbbf054a00d1f162fda94ce236a899ca01123a798c561ba307ca38af5f0 \\\n'
        '    --hash=sha256:c6cb2335a411b713fdf1e82a752162f72d4a7b5dbc588e32aa18383318b05866 \\\n'
        '    --hash=sha256:cc55d71898ea30dc95900297d191377caba257612f384207fe9f8293b5850f90 \\\n'
        '    --hash=sha256:d03c9d6f2a3dffbd62671ca070f13fc527bb1867b4ec2b98c7eeed381d4f389a \\\n'
        '    --hash=sha256:d383591f3dcbe545f6cc62daaef9c7cdfe0dff0fb9e1c8121101cabe9098cfa6 \\\n'
        '    --hash=sha256:d9d46e06846a41ba906ab25302cf0fd522f81aa2a85a71021826f34639ad31ef \\\n'
        '    --hash=sha256:d9dedeaf19097a143ed6da37f04f4051aba353c95ef507764d344229b2b740ae \\\n'
        '    --hash=sha256:e45274b20e524ae5c39d7fc1ca2aa923aab494776d2d4b316b49ec7572ca324c \\\n'
        '    --hash=sha256:ee8dec072569f43835932a3b10c55973593abc00936c202707a4ad06af7cb294 \\\n'
        '    --hash=sha256:f24faab6ed18f216a37870d8c5623f9c044566d75ec586ef884e13a02a9d62c5 \\\n'
        '    --hash=sha256:f2a21d39fbdb948857f67eacb5bbaaf36802de044ec36fbef7a1c8f0dd3a4ab2 \\\n'
        '    --hash=sha256:f3ad4c0eb4e2a9aeb990af6c09e6fa0b195c8c0e7b272ecc8d4d2b6574809d34 \\\n'
        '    --hash=sha256:fc28912a2dc924dddc2087679cc8b7263accc71b9ff025a1362b004711661a69 \\\n'
        '    --hash=sha256:fca15aabbe9b8355800d923cc2e82c8ef514af321e18b437c3d782aa884eaeec \\\n'
        '    --hash=sha256:fd44d66093a239358d07c42a91eebf5015aa54fccba959db899f932218ac9cc8\n'
        '    # via -r tools/workshop-requirements.in\n'
        'pycparser==3.0 \\\n'
        '    --hash=sha256:600f49d217304a5902ac3c37e1281c9fe94e4d0489de643a9504c5cdfdfc6b29 \\\n'
        '    --hash=sha256:b727414169a36b7d524c1c3e31839a521725078d7b2ff038656844266160a992\n'
        '    # via cffi\n'
        'pyyaml==6.0.3 \\\n'
        '    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n'
        '    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n'
        '    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n'
        '    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n'
        '    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n'
        '    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n'
        '    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n'
        '    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n'
        '    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n'
        '    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n'
        '    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n'
        '    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n'
        '    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n'
        '    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n'
        '    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n'
        '    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n'
        '    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n'
        '    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n'
        '    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n'
        '    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n'
        '    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n'
        '    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n'
        '    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n'
        '    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n'
        '    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n'
        '    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n'
        '    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n'
        '    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n'
        '    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n'
        '    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n'
        '    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n'
        '    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n'
        '    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n'
        '    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n'
        '    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n'
        '    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n'
        '    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n'
        '    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n'
        '    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n'
        '    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n'
        '    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n'
        '    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n'
        '    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n'
        '    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n'
        '    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n'
        '    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n'
        '    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n'
        '    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n'
        '    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n'
        '    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n'
        '    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n'
        '    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n'
        '    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n'
        '    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n'
        '    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n'
        '    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n'
        '    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n'
        '    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n'
        '    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n'
        '    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n'
        '    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n'
        '    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n'
        '    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n'
        '    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n'
        '    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n'
        '    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n'
        '    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n'
        '    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n'
        '    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n'
        '    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n'
        '    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n'
        '    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n'
        '    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n'
        '    # via\n'
        '    #   accelerate\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'regex==2026.9.10 \\\n'
        '    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \\\n'
        '    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \\\n'
        '    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \\\n'
        '    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \\\n'
        '    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \\\n'
        '    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \\\n'
        '    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \\\n'
        '    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \\\n'
        '    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \\\n'
        '    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \\\n'
        '    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \\\n'
        '    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \\\n'
        '    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \\\n'
        '    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \\\n'
        '    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \\\n'
        '    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \\\n'
        '    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \\\n'
        '    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \\\n'
        '    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \\\n'
        '    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \\\n'
        '    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \\\n'
        '    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \\\n'
        '    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \\\n'
        '    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \\\n'
        '    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \\\n'
        '    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \\\n'
        '    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \\\n'
        '    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \\\n'
        '    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \\\n'
        '    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \\\n'
        '    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \\\n'
        '    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \\\n'
        '    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \\\n'
        '    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \\\n'
        '    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \\\n'
        '    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \\\n'
        '    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \\\n'
        '    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \\\n'
        '    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \\\n'
        '    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \\\n'
        '    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \\\n'
        '    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \\\n'
        '    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \\\n'
        '    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \\\n'
        '    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \\\n'
        '    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \\\n'
        '    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \\\n'
        '    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \\\n'
        '    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \\\n'
        '    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \\\n'
        '    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \\\n'
        '    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \\\n'
        '    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \\\n'
        '    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \\\n'
        '    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \\\n'
        '    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \\\n'
        '    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \\\n'
        '    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \\\n'
        '    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \\\n'
        '    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \\\n'
        '    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \\\n'
        '    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \\\n'
        '    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \\\n'
        '    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \\\n'
        '    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \\\n'
        '    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \\\n'
        '    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \\\n'
        '    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \\\n'
        '    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \\\n'
        '    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \\\n'
        '    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \\\n'
        '    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \\\n'
        '    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \\\n'
        '    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \\\n'
        '    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \\\n'
        '    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \\\n'
        '    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \\\n'
        '    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \\\n'
        '    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \\\n'
        '    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \\\n'
        '    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \\\n'
        '    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \\\n'
        '    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \\\n'
        '    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \\\n'
        '    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \\\n'
        '    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \\\n'
        '    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \\\n'
        '    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \\\n'
        '    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \\\n'
        '    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \\\n'
        '    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \\\n'
        '    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \\\n'
        '    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \\\n'
        '    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \\\n'
        '    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \\\n'
        '    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \\\n'
        '    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \\\n'
        '    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \\\n'
        '    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \\\n'
        '    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \\\n'
        '    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \\\n'
        '    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \\\n'
        '    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \\\n'
        '    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \\\n'
        '    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \\\n'
        '    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \\\n'
        '    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \\\n'
        '    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \\\n'
        '    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \\\n'
        '    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \\\n'
        '    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \\\n'
        '    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \\\n'
        '    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \\\n'
        '    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \\\n'
        '    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \\\n'
        '    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \\\n'
        '    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \\\n'
        '    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \\\n'
        '    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \\\n'
        '    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \\\n'
        '    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \\\n'
        '    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \\\n'
        '    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \\\n'
        '    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \\\n'
        '    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \\\n'
        '    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \\\n'
        '    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \\\n'
        '    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \\\n'
        '    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \\\n'
        '    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07\n'
        '    # via transformers\n'
        'requests==2.34.2 \\\n'
        '    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n'
        '    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'safetensors==0.8.0 \\\n'
        '    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n'
        '    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n'
        '    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n'
        '    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n'
        '    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n'
        '    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n'
        '    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n'
        '    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n'
        '    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n'
        '    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n'
        '    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n'
        '    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n'
        '    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n'
        '    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n'
        '    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n'
        '    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n'
        '    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n'
        '    # via\n'
        '    #   accelerate\n'
        '    #   transformers\n'
        'scipy==1.15.2 \\\n'
        '    --hash=sha256:01edfac9f0798ad6b46d9c4c9ca0e0ad23dbf0b1eb70e96adb9fa7f525eff0bf \\\n'
        '    --hash=sha256:03205d57a28e18dfd39f0377d5002725bf1f19a46f444108c29bdb246b6c8a11 \\\n'
        '    --hash=sha256:08b57a9336b8e79b305a143c3655cc5bdbe6d5ece3378578888d2afbb51c4e37 \\\n'
        '    --hash=sha256:11e7ad32cf184b74380f43d3c0a706f49358b904fa7d5345f16ddf993609184d \\\n'
        '    --hash=sha256:28a0d2c2075946346e4408b211240764759e0fabaeb08d871639b5f3b1aca8a0 \\\n'
        '    --hash=sha256:2b871df1fe1a3ba85d90e22742b93584f8d2b8e6124f8372ab15c71b73e428b8 \\\n'
        '    --hash=sha256:302093e7dfb120e55515936cb55618ee0b895f8bcaf18ff81eca086c17bd80af \\\n'
        '    --hash=sha256:42dabaaa798e987c425ed76062794e93a243be8f0f20fff6e7a89f4d61cb3d40 \\\n'
        '    --hash=sha256:447ce30cee6a9d5d1379087c9e474628dab3db4a67484be1b7dc3196bfb2fac9 \\\n'
        '    --hash=sha256:4c6676490ad76d1c2894d77f976144b41bd1a4052107902238047fb6a473e971 \\\n'
        '    --hash=sha256:54c462098484e7466362a9f1672d20888f724911a74c22ae35b61f9c5919183d \\\n'
        '    --hash=sha256:597a0c7008b21c035831c39927406c6181bcf8f60a73f36219b69d010aa04737 \\\n'
        '    --hash=sha256:5a6fd6eac1ce74a9f77a7fc724080d507c5812d61e72bd5e4c489b042455865e \\\n'
        '    --hash=sha256:5ea7ed46d437fc52350b028b1d44e002646e28f3e8ddc714011aaf87330f2f32 \\\n'
        '    --hash=sha256:601881dfb761311045b03114c5fe718a12634e5608c3b403737ae463c9885d53 \\\n'
        '    --hash=sha256:62ca1ff3eb513e09ed17a5736929429189adf16d2d740f44e53270cc800ecff1 \\\n'
        '    --hash=sha256:69ea6e56d00977f355c0f84eba69877b6df084516c602d93a33812aa04d90a3d \\\n'
        '    --hash=sha256:6a8e34cf4c188b6dd004654f88586d78f95639e48a25dfae9c5e34a6dc34547e \\\n'
        '    --hash=sha256:6d0194c37037707b2afa7a2f2a924cf7bac3dc292d51b6a925e5fcb89bc5c776 \\\n'
        '    --hash=sha256:6f223753c6ea76983af380787611ae1291e3ceb23917393079dcc746ba60cfb5 \\\n'
        '    --hash=sha256:6f5e296ec63c5da6ba6fa0343ea73fd51b8b3e1a300b0a8cae3ed4b1122c7462 \\\n'
        '    --hash=sha256:7cd5b77413e1855351cdde594eca99c1f4a588c2d63711388b6a1f1c01f62274 \\\n'
        '    --hash=sha256:869269b767d5ee7ea6991ed7e22b3ca1f22de73ab9a49c44bad338b725603301 \\\n'
        '    --hash=sha256:87994da02e73549dfecaed9e09a4f9d58a045a053865679aeb8d6d43747d4df3 \\\n'
        '    --hash=sha256:888307125ea0c4466287191e5606a2c910963405ce9671448ff9c81c53f85f58 \\\n'
        '    --hash=sha256:92233b2df6938147be6fa8824b8136f29a18f016ecde986666be5f4d686a91a4 \\\n'
        '    --hash=sha256:9412f5e408b397ff5641080ed1e798623dbe1ec0d78e72c9eca8992976fa65aa \\\n'
        '    --hash=sha256:9b18aa747da280664642997e65aab1dd19d0c3d17068a04b3fe34e2559196cb9 \\\n'
        '    --hash=sha256:9de9d1416b3d9e7df9923ab23cd2fe714244af10b763975bea9e4f2e81cebd27 \\\n'
        '    --hash=sha256:a2ec871edaa863e8213ea5df811cd600734f6400b4af272e1c011e69401218e9 \\\n'
        '    --hash=sha256:a5080a79dfb9b78b768cebf3c9dcbc7b665c5875793569f48bf0e2b1d7f68f6f \\\n'
        '    --hash=sha256:a8bf5cb4a25046ac61d38f8d3c3426ec11ebc350246a4642f2f315fe95bda655 \\\n'
        '    --hash=sha256:b09ae80010f52efddb15551025f9016c910296cf70adbf03ce2a8704f3a5ad20 \\\n'
        '    --hash=sha256:b5e025e903b4f166ea03b109bb241355b9c42c279ea694d8864d033727205e65 \\\n'
        '    --hash=sha256:bad78d580270a4d32470563ea86c6590b465cb98f83d760ff5b0990cb5518a93 \\\n'
        '    --hash=sha256:bae43364d600fdc3ac327db99659dcb79e6e7ecd279a75fe1266669d9a652828 \\\n'
        '    --hash=sha256:c4697a10da8f8765bb7c83e24a470da5797e37041edfd77fd95ba3811a47c4fd \\\n'
        '    --hash=sha256:c90ebe8aaa4397eaefa8455a8182b164a6cc1d59ad53f79943f266d99f68687f \\\n'
        '    --hash=sha256:cd58a314d92838f7e6f755c8a2167ead4f27e1fd5c1251fd54289569ef3495ec \\\n'
        '    --hash=sha256:cf72ff559a53a6a6d77bd8eefd12a17995ffa44ad86c77a5df96f533d4e6c6bb \\\n'
        '    --hash=sha256:def751dd08243934c884a3221156d63e15234a3155cf25978b0a668409d45eb6 \\\n'
        '    --hash=sha256:e7c68b6a43259ba0aab737237876e5c2c549a031ddb7abc28c7b47f22e202ded \\\n'
        '    --hash=sha256:ecf797d2d798cf7c838c6d98321061eb3e72a74710e6c40540f0e8087e3b499e \\\n'
        '    --hash=sha256:f031846580d9acccd0044efd1a90e6f4df3a6e12b4b6bd694a7bc03a89892b28 \\\n'
        '    --hash=sha256:fb530e4794fc8ea76a4a21ccb67dea33e5e0e60f07fc38a49e821e1eae3b71a0 \\\n'
        '    --hash=sha256:fe8a9eb875d430d81755472c5ba75e84acc980e4a8f6204d402849234d3017db\n'
        '    # via -r tools/workshop-requirements.in\n'
        'setuptools==84.0.0 \\\n'
        '    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n'
        '    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n'
        '    # via torch\n'
        'soundfile==0.13.1 \\\n'
        '    --hash=sha256:03267c4e493315294834a0870f31dbb3b28a95561b80b134f0bd3cf2d5f0e618 \\\n'
        '    --hash=sha256:1e70a05a0626524a69e9f0f4dd2ec174b4e9567f4d8b6c11d38b5c289be36ee9 \\\n'
        '    --hash=sha256:743f12c12c4054921e15736c6be09ac26b3b3d603aef6fd69f9dde68748f2593 \\\n'
        '    --hash=sha256:82dc664d19831933fe59adad199bf3945ad06d84bc111a5b4c0d3089a5b9ec33 \\\n'
        '    --hash=sha256:9c9e855f5a4d06ce4213f31918653ab7de0c5a8d8107cd2427e44b42df547deb \\\n'
        '    --hash=sha256:a23c717560da2cf4c7b5ae1142514e0fd82d6bbd9dfc93a50423447142f2c445 \\\n'
        '    --hash=sha256:b2c68dab1e30297317080a5b43df57e302584c49e2942defdde0acccc53f0e5b \\\n'
        '    --hash=sha256:c734564fab7c5ddf8e9be5bf70bab68042cd17e9c214c06e365e20d64f9a69d5\n'
        '    # via -r tools/workshop-requirements.in\n'
        'sympy==1.13.1 \\\n'
        '    --hash=sha256:9cebf7e04ff162015ce31c9c6c9144daa34a93bd082f54fd8f12deca4f47515f \\\n'
        '    --hash=sha256:db36cdc64bf61b9b24578b6f7bab1ecdd2452cf008f34faa33776680c26d66f8\n'
        '    # via torch\n'
        'tokenizers==0.21.4 \\\n'
        '    --hash=sha256:1340ff877ceedfa937544b7d79f5b7becf33a4cfb58f89b3b49927004ef66f78 \\\n'
        '    --hash=sha256:2107ad649e2cda4488d41dfd031469e9da3fcbfd6183e74e4958fa729ffbf9c6 \\\n'
        '    --hash=sha256:2ccc10a7c3bcefe0f242867dc914fc1226ee44321eb618cfe3019b5df3400133 \\\n'
        '    --hash=sha256:39b376f5a1aee67b4d29032ee85511bbd1b99007ec735f7f35c8a2eb104eade5 \\\n'
        '    --hash=sha256:3c1f4317576e465ac9ef0d165b247825a2a4078bcd01cba6b54b867bdf9fdd8b \\\n'
        '    --hash=sha256:3c73012da95afafdf235ba80047699df4384fdc481527448a078ffd00e45a7d9 \\\n'
        '    --hash=sha256:475d807a5c3eb72c59ad9b5fcdb254f6e17f53dfcbb9903233b0dfa9c943b597 \\\n'
        '    --hash=sha256:51b7eabb104f46c1c50b486520555715457ae833d5aee9ff6ae853d1130506ff \\\n'
        '    --hash=sha256:5e2f601a8e0cd5be5cc7506b20a79112370b9b3e9cb5f13f68ab11acd6ca7d60 \\\n'
        '    --hash=sha256:6c42a930bc5f4c47f4ea775c91de47d27910881902b0f20e4990ebe045a415d0 \\\n'
        '    --hash=sha256:714b05b2e1af1288bd1bc56ce496c4cebb64a20d158ee802887757791191e6e2 \\\n'
        '    --hash=sha256:c212aa4e45ec0bb5274b16b6f31dd3f1c41944025c2358faaa5782c754e84c24 \\\n'
        '    --hash=sha256:cc88bb34e23a54cc42713d6d98af5f1bf79c07653d24fe984d2d695ba2c922a2 \\\n'
        '    --hash=sha256:f23186c40395fc390d27f519679a58023f368a0aad234af145e0f39ad1212732 \\\n'
        '    --hash=sha256:fa23f85fbc9a02ec5c6978da172cdcbac23498c3ca9f3645c5c68740ac007880\n'
        '    # via transformers\n'
        'torch==2.6.0 \\\n'
        '    --hash=sha256:09e06f9949e1a0518c5b09fe95295bc9661f219d9ecb6f9893e5123e10696628 \\\n'
        '    --hash=sha256:265f70de5fd45b864d924b64be1797f86e76c8e48a02c2a3a6fc7ec247d2226c \\\n'
        '    --hash=sha256:2bb8987f3bb1ef2675897034402373ddfc8f5ef0e156e2d8cfc47cacafdda4a9 \\\n'
        '    --hash=sha256:46763dcb051180ce1ed23d1891d9b1598e07d051ce4c9d14307029809c4d64f7 \\\n'
        '    --hash=sha256:4874a73507a300a5d089ceaff616a569e7bb7c613c56f37f63ec3ffac65259cf \\\n'
        '    --hash=sha256:510c73251bee9ba02ae1cb6c9d4ee0907b3ce6020e62784e2d7598e0cfa4d6cc \\\n'
        '    --hash=sha256:56eeaf2ecac90da5d9e35f7f35eb286da82673ec3c582e310a8d1631a1c02341 \\\n'
        '    --hash=sha256:683410f97984103148e31b38a8631acf31c3034c020c0f4d26171e7626d8317a \\\n'
        '    --hash=sha256:6860df13d9911ac158f4c44031609700e1eba07916fff62e21e6ffa0a9e01961 \\\n'
        '    --hash=sha256:7979834102cd5b7a43cc64e87f2f3b14bd0e1458f06e9f88ffa386d07c7446e1 \\\n'
        '    --hash=sha256:7e1448426d0ba3620408218b50aa6ada88aeae34f7a239ba5431f6c8774b1239 \\\n'
        '    --hash=sha256:94fc63b3b4bedd327af588696559f68c264440e2503cc9e6954019473d74ae21 \\\n'
        '    --hash=sha256:9a610afe216a85a8b9bc9f8365ed561535c93e804c2a317ef7fabcc5deda0989 \\\n'
        '    --hash=sha256:9ea955317cfcd3852b1402b62af258ce735c2edeee42ca9419b6bc889e5ae053 \\\n'
        '    --hash=sha256:a0d5e1b9874c1a6c25556840ab8920569a7a4137afa8a63a32cee0bc7d89bd4b \\\n'
        '    --hash=sha256:b789069020c5588c70d5c2158ac0aa23fd24a028f34a8b4fcb8fcb4d7efcf5fb \\\n'
        '    --hash=sha256:bb2c6c3e65049f081940f5ab15c9136c7de40d3f01192541c920a07c7c585b7e \\\n'
        '    --hash=sha256:c4f103a49830ce4c7561ef4434cc7926e5a5fe4e5eb100c19ab36ea1e2b634ab \\\n'
        '    --hash=sha256:ccbd0320411fe1a3b3fec7b4d3185aa7d0c52adac94480ab024b5c8f74a0bf1d \\\n'
        '    --hash=sha256:ff96f4038f8af9f7ec4231710ed4549da1bdebad95923953a25045dcf6fd87e2\n'
        '    # via\n'
        '    #   -r tools/workshop-requirements.in\n'
        '    #   accelerate\n'
        '    #   torchaudio\n'
        'torchaudio==2.6.0 \\\n'
        '    --hash=sha256:04803a969710bdb77a4ddfdb85a32fa9b9e0310dc91f7eb7e54d6083dd69bfab \\\n'
        '    --hash=sha256:0eda1cd876f44fc014dc04aa680db2fa355a83df5d834398db6dd5f5cd911f4c \\\n'
        '    --hash=sha256:0f0db5c997d031c34066d8be1c0ce7d2a1f2b6c016a92885b20b00bfeb17b753 \\\n'
        '    --hash=sha256:22798d5d8e37869bd5875d37f42270efbeb8ae94bda97fed40c1c5e0e1c62fa3 \\\n'
        '    --hash=sha256:377b177a3d683a9163e4cab5a06f0346dac9ff96fa527477338fd90fc6a2a4b6 \\\n'
        '    --hash=sha256:393fa74ec40d167f0170728ea21c9b5e0f830648fd02df7db2bf7e62f64245ec \\\n'
        '    --hash=sha256:52182f6de4e7b342d139e54b703185d428de9cce3c4cf914a9b2ab2359d192a3 \\\n'
        '    --hash=sha256:52f15185349c370fc1faa84e8b8b2782c007472db9d586a16bba314130b322f2 \\\n'
        '    --hash=sha256:6291d9507dc1d6b4ffe8843fbfb201e6c8270dd8c42ad70bb76226c0ebdcad56 \\\n'
        '    --hash=sha256:66f2e0bd5ab56fd81419d2f5afb74a9a70141688594646441756c8c24f424a73 \\\n'
        '    --hash=sha256:715aa21f6bdbd085454c313ae3a2c7cc07bf2e8cf05752f819afb5b4c57f4e6f \\\n'
        '    --hash=sha256:72e77055d8e742475c6dfacf59fab09b1fc94d4423e14897e188b67cad3851c6 \\\n'
        '    --hash=sha256:7d0e4b08c42325bf4b887de9a25c44ed882997001740e1bd7d901f65581cf1ab \\\n'
        '    --hash=sha256:86d6239792bf94741a41acd6fe3d549faaf0d50e7275d17d076a190bd007e2f9 \\\n'
        '    --hash=sha256:8c1a4d08e35a9ceaadadbff6e60bcb3442482f800369be350103dfd08b4ddf52 \\\n'
        '    --hash=sha256:9d8e07789452efdb8132d62afe21f2293a72805f26c2891c6c53e4e4df38ddf6 \\\n'
        '    --hash=sha256:b521ea9618fb4c29a6f8071628170c222291f46a48a3bf424cfeb488f54af714 \\\n'
        '    --hash=sha256:c12fc41241b8dfce3ccc1917f1c81a0f92f532d9917706600046f1eb21d2d765 \\\n'
        '    --hash=sha256:c6386bfa478afae2137715bb60f35520e3b05f5fc6d3bcc6969cf9cdfb11c09c \\\n'
        '    --hash=sha256:d855da878a28c2e5e6fb3d76fcddd544f4d957a320b29602cea5af2fe0ad1f3a\n'
        '    # via -r tools/workshop-requirements.in\n'
        'tqdm==4.70.1 \\\n'
        '    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n'
        '    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'transformers==4.52.1 \\\n'
        '    --hash=sha256:604b2bb357c480dc5883b7944e8562c967f6b06f63dfb6a1c4665d13d067148f \\\n'
        '    --hash=sha256:c380d583ed9c7ebe3e30ca5e55ec1249db39eb9ee277f8e74dab1abc6a03c938\n'
        '    # via -r tools/workshop-requirements.in\n'
        'triton==3.2.0 \\\n'
        '    --hash=sha256:30ceed0eff2c4a73b14eb63e052992f44bbdf175f3fad21e1ac8097a772de7ee \\\n'
        '    --hash=sha256:8009a1fb093ee8546495e96731336a33fb8856a38e45bb4ab6affd6dbc3ba220 \\\n'
        '    --hash=sha256:8d9b215efc1c26fa7eefb9a157915c92d52e000d2bf83e5f69704047e63f125c \\\n'
        '    --hash=sha256:b3e54983cd51875855da7c68ec05c05cf8bb08df361b1d5b69e05e40b0c9bd62 \\\n'
        '    --hash=sha256:e5dfa23ba84541d7c0a531dfce76d8bcd19159d50a4a8b14ad01e91734a5c1b0\n'
        '    # via torch\n'
        'typing-extensions==4.16.0 \\\n'
        '    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n'
        '    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        'urllib3==2.8.0 \\\n'
        '    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n'
        '    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n'
        '    # via requests\n'
    ),
    'weights/dimer-base-manifest.json': (
        '{\n'
        '  "format": "dimer_hf_snapshot",\n'
        '  "formatVersion": 1,\n'
        '  "modelKey": "whisper-large-v3-turbo",\n'
        '  "modelId": "openai/whisper-large-v3-turbo",\n'
        '  "revision": "41f01f3fe87f28c78e2fbf8b568835947dd65ed9",\n'
        '  "files": [\n'
        '    {\n'
        '      "path": "README.md",\n'
        '      "bytes": 21196,\n'
        '      "sha256": "aaef74a740faca90fa1899c4233ebe17f2093b9846d0acf16d9131bf650e9585"\n'
        '    },\n'
        '    {\n'
        '      "path": "added_tokens.json",\n'
        '      "bytes": 34648,\n'
        '      "sha256": "3c51f66c4c21f9e126970078f11ae77a78c74aee8df606ee9daba86e467108e0"\n'
        '    },\n'
        '    {\n'
        '      "path": "config.json",\n'
        '      "bytes": 1256,\n'
        '      "sha256": "c5b526b3e3cd64cd8940dabb45e8ba726629e22d8ed389c29b552f9140daf04a"\n'
        '    },\n'
        '    {\n'
        '      "path": "generation_config.json",\n'
        '      "bytes": 3772,\n'
        '      "sha256": "cce11bfe3aaa6ae9e072ea2637caaec8795e68d9b67e655a5af16ee509681a4c"\n'
        '    },\n'
        '    {\n'
        '      "path": "merges.txt",\n'
        '      "bytes": 493869,\n'
        '      "sha256": "2df2990a395e35e8dfbc7511e08c12d56018d8d04691e0133e5d63b21e154dc6"\n'
        '    },\n'
        '    {\n'
        '      "path": "model.safetensors",\n'
        '      "bytes": 1617824864,\n'
        '      "sha256": "542566a422ae4f3fd23f1ba11add198fca01bbf82e66e6a2857b3f608b1eb9d1"\n'
        '    },\n'
        '    {\n'
        '      "path": "normalizer.json",\n'
        '      "bytes": 52666,\n'
        '      "sha256": "bf1c507dc8724ca9cf9903640dacfb69dae2f00edee4f21ceba106a7392f26dd"\n'
        '    },\n'
        '    {\n'
        '      "path": "preprocessor_config.json",\n'
        '      "bytes": 340,\n'
        '      "sha256": "7ccc62c6f2765af1f3b46c00c9b5894426835a05021c8b9c01eecb6dfb542711"\n'
        '    },\n'
        '    {\n'
        '      "path": "special_tokens_map.json",\n'
        '      "bytes": 2186,\n'
        '      "sha256": "baea4ea09372eb4fca86b4e4346139fd73cb807d5087e9de0948e971739c3e74"\n'
        '    },\n'
        '    {\n'
        '      "path": "tokenizer.json",\n'
        '      "bytes": 2710337,\n'
        '      "sha256": "297b13372ac43916285644fb9687add3cc62ee2a1adb60da3dc25cc94c1871fd"\n'
        '    },\n'
        '    {\n'
        '      "path": "tokenizer_config.json",\n'
        '      "bytes": 282843,\n'
        '      "sha256": "844b642c73a91359722f47b35705f7174686df33d252695d8572cf9ac03a6389"\n'
        '    },\n'
        '    {\n'
        '      "path": "vocab.json",\n'
        '      "bytes": 1036558,\n'
        '      "sha256": "e2aa043ef015641d363d8288e7c241c85e36a5c761fb303598e0710233344387"\n'
        '    }\n'
        '  ],\n'
        '  "totalBytes": 1622464535\n'
        '}\n'
    ),
    'source.json': (
        '{\n'
        '  "repository": "kurtvalcorza/whisper-asr-pipeline",\n'
        '  "base_revision": "1eb99622614d547170675b24c051510168a694da",\n'
        '  "generator": "build_whisper_speech_workshop.py/1",\n'
        '  "sources": {\n'
        '    "whisper_reference.py": "1febc8f7f7d0bbffa1fc078e3ab956f2fab102c4357f6587473399085c0b3694",\n'
        '    "workshop.py": "0272e121370e0842e52d117c574b5de568c1cbe79ba269bc563cd244df5ef148",\n'
        '    "sample.json": "d9c71725057b5724319237a1facccd4fae05e516992e4b12a20dfecfc9544c2f",\n'
        '    "requirements.txt": "0d3aed7e4fb747db272327809068b1a6370b8064dcb35ad3f0275306abf6eef1",\n'
        '    "weights/dimer-base-manifest.json": "72b8646d80a2b821599e75eb99cedfddf438382158a3539a47e01335bd2aafa2"\n'
        '  }\n'
        '}'
    ),
}
CARRIED_HASHES = {'whisper_reference.py': '1febc8f7f7d0bbffa1fc078e3ab956f2fab102c4357f6587473399085c0b3694', 'workshop.py': '0272e121370e0842e52d117c574b5de568c1cbe79ba269bc563cd244df5ef148', 'sample.json': 'd9c71725057b5724319237a1facccd4fae05e516992e4b12a20dfecfc9544c2f', 'requirements.txt': '0d3aed7e4fb747db272327809068b1a6370b8064dcb35ad3f0275306abf6eef1', 'weights/dimer-base-manifest.json': '72b8646d80a2b821599e75eb99cedfddf438382158a3539a47e01335bd2aafa2', 'source.json': '433ebc23c46b4daca4452e86dac3f6b4808eaafa227aca6b75690f4bf12e691a'}
for name, source in CARRIED_FILES.items():
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
    if hashlib.sha256(target.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried source integrity mismatch: ' + name)
print('Carried code, sample manifest, model manifest and lock verified.')

In [ ]:
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
wheel = ROOT / 'uv.whl'
with urllib.request.urlopen(UV_URL, timeout=90) as response:
    data = response.read(20081405)
if len(data) != 20081404 or hashlib.sha256(data).hexdigest() != UV_SHA256:
    raise RuntimeError('uv bootstrap integrity failure')
wheel.write_bytes(data)
with zipfile.ZipFile(wheel) as archive:
    member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
    UV = ROOT / 'uv'
    UV.write_bytes(archive.read(member))
UV.chmod(0o700)
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1')
ENV.pop('HF_TOKEN', None)
ENV.pop('HUGGING_FACE_HUB_TOKEN', None)
subprocess.run([str(UV), 'venv', '--python', '3.12.12', '--managed-python', str(ROOT / 'env')], env=ENV, check=True)
PYTHON = ROOT / 'env/bin/python'
subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes',
                '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple',
                '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
subprocess.run([str(PYTHON), '-c', 'import sys,torch; print(sys.version); print(torch.__version__); assert torch.cuda.is_available(), "CUDA unavailable in isolated environment"'], env=ENV, check=True)
BOOTSTRAP_SECONDS = time.perf_counter() - SESSION_START

def run_stage(stage, *options):
    print(f'Running {stage}; progress and diagnostics are saved to its log.', flush=True)
    command = [str(PYTHON), str(ROOT / 'workshop.py'), '--root', str(ROOT), '--stage', stage, *options]
    log = ROOT / (stage + '.log')
    with log.open('w', encoding='utf-8') as handle:
        result = subprocess.run(command, env=ENV, stdout=handle, stderr=subprocess.STDOUT)
    tail = log.read_text(encoding='utf-8')[-7000:]
    print(tail)
    if result.returncode:
        raise RuntimeError(f'{stage} failed (exit {result.returncode}); see {log}. Last diagnostics above.')

def result_dir(stage):
    return Path(json.loads((ROOT / (stage + '_output.json')).read_text())['path'])

def show_results(stage):
    from IPython.display import Audio, Markdown, display
    result = result_dir(stage)
    rows = json.loads((result / 'transcripts.json').read_text())
    metrics = json.loads((result / 'metrics.json').read_text())
    def rate(value):
        return 'not measurable' if value is None else f'{value:.4f}'
    table = ['| Condition | Clips | WER | CER | S | D | I |', '|---|---:|---:|---:|---:|---:|---:|']
    for condition, values in metrics.items():
        if 'wer' in values:
            table.append(f"| {condition} | {values['evaluated']} | {rate(values['wer'])} | {rate(values['cer'])} | {values['S']} | {values['D']} | {values['I']} |")
    display(Markdown('\n'.join(table)))
    print('Coverage and timing:', json.dumps(metrics, indent=2))
    # Fixed first example and deterministic best/worst labelled examples.
    prepared_file = ROOT / 'prepared.json'
    audio_by_id = {r['id']: r['audio_path'] for r in json.loads(prepared_file.read_text())} if stage != 'byod' and prepared_file.exists() else {}
    labelled = [r for r in rows if r['wer'] is not None]
    examples = [('Fixed first example', rows[0])] + ([('Lowest WER', min(labelled, key=lambda r: (r['wer'], r['id']))), ('Highest WER', max(labelled, key=lambda r: (r['wer'], r['id'])))] if labelled else [])
    for label, row in examples:
        print(label + ':', row['id'], '/', row['condition'], *(['/ file', row['file']] if row.get('file') else []))
        print('Reference:', row['reference'])
        print('Prediction:', row['hypothesis'])
        print('Normalized reference:', row['normalized_reference'])
        print('Normalized prediction:', row['normalized_hypothesis'])
        print('WER:', row['wer'], 'CER:', row['cer'], 'Ceiling flag:', row['truncated'])
        print('Alignment:', row['alignment'])
        if label != 'Fixed first example' and row['id'] in audio_by_id:
            source = '' if row['condition'] == 'clean' else ' (clean source; the scored version also had the synthetic noise)'
            question = 'Who is right here, Whisper or the reference?' if label == 'Highest WER' else 'Do both transcripts match what you hear?'
            display(Markdown(f'**{label}: listen{source}.** {question}'))
            display(Audio(filename=audio_by_id[row['id']], normalize=False))
    for condition in ('clean', 'noisy'):
        audio = result / ('listen_' + condition + '.wav')
        if audio.exists():
            display(Markdown('**First recording: ' + condition + '** (scored gain, no playback normalization)'))
            display(Audio(filename=str(audio), normalize=False))

## 2. Listen before predicting

**Input:** a pinned MINDS-14 English banking-intent corpus. **System:** integrity checks, bounded decoding, mono conversion and 16 kHz resampling. **Output:** 80 frozen recordings: 64 evaluation + 16 activity. All recordings are 1–20 seconds; none is silently cropped.

The build inspected 563 source rows, found 533 eligible unique recordings, and froze 80 IDs with seed 42. The selected audio totals about 597 seconds. The manifest records every exclusion and the exact raw/decoded hashes. These roles come from the source's train split; they are not an official ASR benchmark. **The references are not human transcripts:** the dataset authors produced them by running a commercial speech recognizer (Google ASR) and keeping its single best transcript (Gerz et al., 2021, §4). Every score below therefore measures *agreement between two automatic transcripts*. Speaker/session IDs are unavailable, so the roles are recording-disjoint, **not speaker-disjoint**. Whisper's pretraining overlap is unknown.

**Predict:** Which words would be easy to miss in a short customer request: names, numbers, or common phrases? Listen to the example, then read its supplied reference. The reference itself may contain errors; when Whisper and the reference disagree, either one can be wrong.

In [ ]:
run_stage('prepare')
from IPython.display import Audio, display
prepared = json.loads((ROOT / 'prepared.json').read_text())
example = next(row for row in prepared if row['role'] == 'activity')
display(Audio(filename=example['audio_path'], normalize=False))
print('Reference:', example['reference'])
print({k: example[k] for k in ('id', 'role', 'seconds', 'input_rate', 'input_channels', 'sampling_rate')})

**What to notice:** changing sample rate changes the representation, not the words. Mono conversion averages channels. The display tells you which input changes occurred.

<details><summary>Worked interpretation</summary>A correctly decoded waveform and a plausible reference establish a usable example, not model accuracy. Keep noisy or difficult eligible recordings: removing them after seeing predictions would bias the result.</details>

## 3. Understand word and character errors

Whisper generates text tokens from audio features. It does not return a trustworthy probability that the whole transcript is correct.

**WER = (substitutions + deletions + insertions) / reference words.** CER uses character edits over reference characters. This notebook case-folds Unicode NFC text, normalizes curly apostrophes, replaces punctuation with spaces while retaining internal apostrophes/hyphens, and collapses whitespace. CER removes spaces. Numbers and abbreviations are not rewritten. Raw strings are always retained.

**Predict:** reference “pay my bill”, prediction “pay the bills now”. How many substitutions and insertions do you count? Why can WER exceed 1?

In [ ]:
# Metric demo executes the carried source in the isolated environment, without loading Whisper.
demo = subprocess.run([str(PYTHON), '-c', "from workshop import score; import json; print(json.dumps(score('pay my bill', 'pay the bills now'), indent=2))"], cwd=ROOT, env=ENV, capture_output=True, text=True, check=True)
print(demo.stdout)

<details><summary>Worked interpretation</summary>Three edits are needed and no alignment uses fewer, so WER = 3/3 = 1. More than one minimal alignment usually exists. A natural reading is two substitutions (my→the, bill→bills) and one insertion (now). The code breaks ties deterministically, working back from the end of the sentence, so it prints a different alignment with the same counts: insertion of **the**, then substitutions **my→bills** and **bill→now** (`op` values `I`, `S`, `S`; `=` marks a match). Both give S = 2, D = 0, I = 1. WER depends on the counts, not on which words are paired. More insertions could push it above 1. The empty-transcript baseline deletes every reference word, giving WER/CER 1 on nonempty references. It is a sanity reference, not a competitive recognizer. Corpus rates sum edits and reference lengths; they do not average each clip's percentage.</details>

## 4. Evaluate the frozen model

**Input:** the 64 evaluation recordings. **Model:** the fixed Whisper large-v3-turbo checkpoint, greedy English transcription, batch size 1. **Output:** transcripts, alignments, corpus WER/CER, timing and coverage.

**Predict:** Will Whisper beat the empty-transcript baseline? Write a reason before running. Model and data manifests are verified before loading. The configuration is saved before evaluation. No labels are used to tune settings. A flagged token ceiling stays in the table; a model failure stops the stage and preserves diagnostics.

In [ ]:
MODEL_START = time.perf_counter()
run_stage('model')
MODEL_ACQUISITION_SECONDS = time.perf_counter() - MODEL_START
run_stage('evaluate')
show_results('evaluate')

**What to notice:** compare model WER/CER with the empty baseline, inspect substitutions/deletions/insertions, and check all 64 rows are represented. Listen to the first scored example, then listen to the deterministically selected lowest- and highest-WER examples and decide by ear whether Whisper or the reference transcript is wrong. Real-time factor is inference seconds divided by audio seconds; it excludes installation, downloads and model loading and is not streaming latency.

<details><summary>Worked interpretation</summary>A lower WER shows closer agreement with these references, which are another recognizer's output: an edit can be Whisper's error, the reference's error, or both, and listening is the only way to tell. A large insertion count can mean the reference omitted words that were spoken. A wrong amount or name can matter even when WER is low. The worst example helps explain the aggregate. WER is neither calibrated confidence nor “1 minus accuracy”; short clips and imperfect references affect its interpretation.</details>

## 5. Change one thing: noise power

**Input:** 16 separate activity recordings. **System:** one deterministic Gaussian-noise vector per recording, scaled to **10 dB SNR**. **Model and decoding:** unchanged. **Output:** paired clean/noisy results.

SNR compares the recording's power to the added noise power. Both are measured over the whole recording, pauses included, so during the spoken parts of a clip with long pauses the ratio is somewhat higher than the target. Lower dB means stronger noise. Both members use the same gain when headroom is needed; there is no clipping. The gain is held across all offered SNR settings. Actual SNR, gain, random seed and waveform hashes are exported.

**Predict:** Will every noisy transcript be worse? What would an improved example mean? Gaussian noise is a controlled teaching intervention, not a simulation of every real environment.

In [ ]:
run_stage('activity')
show_results('activity')
print((result_dir('activity') / 'noise_experiment.csv').read_text())

**What to notice:** compare paired corpus rates and improved/unchanged/worsened counts, not unrelated recordings. The clean and noisy playback use the scored gain. Noise can occasionally improve a prediction; keep that result.

<details><summary>Worked interpretation</summary>A higher noisy corpus WER supports a narrow conclusion about these 16 recordings under this synthetic perturbation. It does not establish performance in traffic, reverberant rooms, other languages or unseen speakers. A changed transcript may also interact with reference errors.</details>

## 6. Inspect and retain your evidence

Each stage exports raw and normalized text, alignment operations, full metrics, configuration and provenance. JSON and CSV are reloaded and checked, and every report member is hashed. The ZIP allowlist excludes audio and credentials. This is an inference report, not a trained model artifact.

In [ ]:
from IPython.display import FileLink
for stage in ('evaluate', 'activity'):
    result = result_dir(stage)
    print(stage, json.loads((result / 'report_identity.json').read_text()))
    display(FileLink(str(result / 'report.zip')))
session = {'bootstrap_seconds': BOOTSTRAP_SECONDS, 'model_acquisition_seconds': MODEL_ACQUISITION_SECONDS,
           'elapsed_seconds': time.perf_counter() - SESSION_START,
           'stage_outputs': {s: str(result_dir(s)) for s in ('evaluate', 'activity')},
           'status': 'executed locally in this hosted runtime; maintainer qualification pending'}
(ROOT / 'session_summary.json').write_text(json.dumps(session, indent=2))
print(session)

## 7. Conclude with evidence

Complete this optional personal note:

> On [sample/scope], Whisper produced [metric] compared with [baseline]. Changing only [noise setting] produced [paired observation]. A notable error was [example]. This does not establish [generalization claim]. Before using it for [application], I would test [specific requirement].

**Self-check:** Can you explain the difference between successful execution, reference agreement and suitability for an application? Can you identify what stayed fixed? What new evidence would you need for different accents or longer recordings?

<details><summary>Worked conclusion structure</summary>Use the values printed in your own run. Name the 64-clip evaluation and 16-clip activity scopes separately. Cite one concrete error and avoid claiming reliability from fluent text. Noise robustness, speaker generalization and critical-word accuracy require additional representative tests.</details>

## 8. Optional exploration

After the default run, enable the control below to compare 5 or 20 dB on the same activity set. This writes a new output directory and cannot overwrite the canonical 10 dB results. Predict the direction first, then compare and explain a counterexample. This is exploratory evidence; do not retune using the evaluation set.

In [ ]:
RUN_EXPLORATION = False  # @param {type:"boolean"}
EXPLORATION_SNR = 5  # @param [5, 20] {type:"raw"}
if RUN_EXPLORATION:
    run_stage('explore', '--snr', str(EXPLORATION_SNR))
    show_results('explore')
else:
    print('Optional exploration disabled; canonical results complete.')

## 9. Optional: bring your own audio

Use recordings you have consent and permission to process in Google Colab, a third-party hosted environment. Audio stays inside this runtime; it is not sent to DIMER or an external inference API. Report exports omit the audio itself, but include your transcript text, so review them before sharing.

Set `BYOD_PATH` to one WAV/FLAC file or a ZIP, then enable `RUN_BYOD`. A ZIP can contain 1–20 WAV/FLAC clips and an optional UTF-8 `transcripts.csv` with columns **id,file,reference_text**, one unique ID and relative filename per audio file. Every file must be declared exactly once. Omit the CSV for wholly unlabelled audio. Every output row keeps the clip's file name in a `file` column; without a CSV, IDs are assigned `0`, `1`, … in sorted file-name order. Do not mix labelled and missing references. An all-empty reference CSV is an explicitly labelled silence set; its per-clip WER/CER remain undefined and insertions show false transcriptions.

Limits: 1–30 seconds per clip, 300 seconds total, mono/stereo, 8–48 kHz, 100 MiB archive, 200 MiB expanded, 100 members, 20 MiB per member. Traversal, symlinks, collisions, duplicate recordings and malformed inputs are rejected before inference; a refusal names the clip or archive member that failed. No audio is cropped. Without a reference, quality is **not measurable**; fluency is not accuracy.

Use Colab's Files sidebar to upload first, or leave the path empty to request the optional upload widget. The default Run all never opens it. BYOD runs validation → transcription → evaluation when labelled → report export in a separate directory.

In [ ]:
RUN_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
if RUN_BYOD:
    selected = BYOD_PATH.strip()
    if not selected:
        from google.colab import files
        upload_dir = ROOT / ('upload-' + uuid.uuid4().hex[:8])
        upload_dir.mkdir()
        previous = Path.cwd()
        try:
            os.chdir(upload_dir)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one WAV, FLAC, or ZIP.')
        selected = str(upload_dir / next(iter(uploaded)))
    run_stage('byod', '--byod', str(Path(selected).resolve()))
    show_results('byod')
    display(FileLink(str(result_dir('byod') / 'report.zip')))
else:
    print('BYOD disabled. No upload or reference is required to finish this unit.')

## Troubleshooting and limitations

| Observation | Action |
|---|---|
| GPU missing / CUDA unavailable | Select a fresh T4 runtime before rerunning from the top. CPU smoke results are not this canonical configuration. |
| Package/bootstrap failure | Read the setup log; use a fresh supported Linux Colab runtime. Do not replace pins or patch the host NumPy. |
| Download timeout | Retry the failed acquisition stage; partial files are not trusted. Persistent integrity failures require maintainer investigation. |
| Out of memory | Start fresh and close other GPU work. Do not silently reduce the sample and call it the same run. |
| Empty or strange transcript | Listen, inspect raw text/reference and ceiling flag. Silence or non-speech can elicit fabricated text. |
| Large WER | Read the edit alignment and reference; rates can exceed 1. Do not clip errors or drop the row. |
| BYOD refused | The message names the clip or archive member and the failed rule (duration, format, schema, path or duplicate). Fix or remove it and rerun the BYOD cell; the completed canonical reports are kept. |

No claim is made about unseen speakers, multilingual recognition, translation, critical decisions or deployment reliability. MINDS-14 covers a narrow banking domain, and its references are a commercial recognizer's automatic transcripts rather than human transcriptions, so every rate here is agreement with that system, not accuracy against verified text. A small synthetic-noise comparison is not a broad robustness benchmark.

**Adaptation continuation:** the repository also contains an older LoRA tutorial. Its guided continuation upgrades have not been qualified for this learning unit. It is not needed to complete this notebook.

## Glossary

- **Reference transcript:** the text a prediction is scored against. In the default sample it is the dataset's Google ASR transcript, not a human one; in BYOD it is your `reference_text`.
- **Prediction (hypothesis):** Whisper's transcript of the same audio.
- **Normalization:** the fixed text clean-up applied to both sides before scoring (Unicode NFC, case folding, apostrophes, punctuation to spaces, whitespace collapsed). Raw text is kept.
- **Alignment:** the matches (`=`), substitutions (`S`), deletions (`D`) and insertions (`I`) that turn the reference words into the predicted words with the fewest edits. The count is unique; the pairing may not be.
- **WER / CER:** (S + D + I) divided by the number of reference words; CER does the same over characters with spaces removed. Corpus rates sum edits over all clips. They can exceed 1 and are not confidence scores.
- **Empty-transcript baseline:** a "recognizer" that outputs nothing; it scores WER/CER 1 on nonempty references and is a sanity reference.
- **Sample rate, resampling, mono:** samples per second (Whisper expects 16 kHz); converting between rates without changing the words; averaging stereo channels into one.
- **SNR (dB):** recording power over added-noise power on a logarithmic scale, measured over the whole recording; lower means noisier.
- **Real-time factor:** inference seconds divided by audio seconds, excluding installation, downloads and model loading.
- **Token ceiling:** the maximum number of generated tokens (256); a transcript that hits it is flagged, not dropped.
- **Greedy decoding:** always taking the most likely next token, with no sampling; the same audio gives the same transcript in the same runtime environment.
- **BYOD:** bring your own data; here, your own WAV/FLAC audio.

## AI Assistance Disclosure

Generative AI assisted with this notebook's code and instructional content under maintainer direction. The maintainer remains responsible for review, validation, and release decisions. AI-generated material may contain errors; validation claims are limited to documented runs and configurations. AI use does not imply independent verification, provider endorsement, or release approval.

## References and attribution

- [OpenAI Whisper](https://github.com/openai/whisper): model/task guidance; Turbo is not trained for translation. Model weights: MIT, pinned `openai/whisper-large-v3-turbo` revision `41f01f3fe87f28c78e2fbf8b568835947dd65ed9`.
- [MINDS-14 / PolyAI](https://huggingface.co/datasets/PolyAI/minds14): Gerz et al., *Multilingual and Cross-Lingual Intent Detection from Spoken Data* (2021), CC-BY-4.0; pinned English-US parquet at revision `40ce77cb32a384e4d50a568e1ec39ac804019d33`. Its `transcription` field is the 1-best output of Google's speech-to-text service, not a human transcription ([arXiv:2104.08524](https://arxiv.org/abs/2104.08524), §4). Tutorial roles, mono conversion, resampling and synthetic noise are our modifications.
- [DIMER notebook specification](https://github.com/kurtvalcorza/ml-worker/blob/main/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md), version 2.2.
- [Pipeline repository](https://github.com/kurtvalcorza/whisper-asr-pipeline). Source identities are carried in this notebook; no repository access is needed during execution.